# Capas y activaciones experimentales

Capas con memoria, activaciones adaptativas, neuronas resonantes y polinómicas, memoria híbrida, redes desde cero con NumPy.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

## Neuronas de ingeniería

#CapaConMemoriaErrores

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import deque

# Dataset artificial
torch.manual_seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.4 * torch.randn_like(X)

# -----------------------------------------
# 🧠 Capa con Memoria de Fallas (errores grandes)
# -----------------------------------------
class CapaConMemoriaErrores(nn.Module):
    def __init__(self, in_features, out_features, k=10, lr_extra=0.01):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.pesos = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.k = k
        self.lr_extra = lr_extra  # tasa de aprendizaje extra para fallos
        self.buffer_fallas = deque(maxlen=k)  # [(x, error), ...]

    def forward(self, x):
        self.entrada_actual = x.detach()
        z = F.linear(x, self.pesos, self.bias)
        self.activacion = torch.tanh(z)
        return self.activacion

    def actualizar_memoria(self, x, error_batch):
        """
        Guarda las K entradas con errores mayores.
        """
        magnitudes_error = error_batch.abs().mean(dim=1)  # error por muestra
        for i in range(x.size(0)):
            entrada = x[i].detach()
            error = magnitudes_error[i].item()
            self.buffer_fallas.append((entrada, error))

    def aplicar_correccion_memoria(self):
        """
        Recorre la memoria de errores y realiza pequeñas correcciones extra.
        """
        if not self.buffer_fallas:
            return

        # Agrupa las entradas con mayor error
        entradas = torch.stack([x for (x, _) in self.buffer_fallas])  # shape: [k, in_features]
        entradas = entradas.to(self.pesos.device)

        with torch.no_grad():
            z = F.linear(entradas, self.pesos, self.bias)
            activacion = torch.tanh(z)
            # Para el ejemplo, corregimos hacia activación = 0 (neutro)
            error_local = activacion  # queremos minimizarla

            # Corrección tipo Hebbiana (simplificada)
            grad_w = torch.matmul(error_local.T, entradas) / entradas.size(0)
            grad_b = error_local.mean(dim=0)

            self.pesos.data -= self.lr_extra * grad_w
            self.bias.data -= self.lr_extra * grad_b

# -----------------------------------------
# Red con memoria de errores
# -----------------------------------------
class RedMemoriaErrores(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = CapaConMemoriaErrores(1, 30, k=15, lr_extra=0.005)
        self.salida = nn.Linear(30, 1)

    def forward(self, x):
        h = self.capa1(x)
        return self.salida(h)

# -----------------------------------------
# Entrenamiento
# -----------------------------------------
modelo = RedMemoriaErrores()
opt = torch.optim.Adam(list(modelo.parameters()), lr=0.01)
loss_fn = nn.MSELoss()

historial_loss = []

for epoch in range(300):
    y_pred = modelo(X)
    loss = loss_fn(y_pred, y)
    historial_loss.append(loss.item())

    opt.zero_grad()
    loss.backward()
    opt.step()

    # Almacenar errores recientes
    error_batch = y_pred - y
    modelo.capa1.actualizar_memoria(X, error_batch)

    # Corrección adicional con memoria
    modelo.capa1.aplicar_correccion_memoria()

    if epoch % 50 == 0:
        print(f"Época {epoch} - Loss: {loss.item():.4f}")

# -----------------------------------------
# Visualización
# -----------------------------------------
with torch.no_grad():
    pred = modelo(X)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.title("Predicción vs Real")
plt.plot(X.numpy(), y.numpy(), label="Real")
plt.plot(X.numpy(), pred.numpy(), label="Predicho")
plt.legend()

plt.subplot(1, 2, 2)
plt.title("Pérdida durante entrenamiento")
plt.plot(historial_loss)
plt.grid(True)

plt.tight_layout()
plt.show()

#CapaConMemoriaErrores

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import deque
import numpy as np

# Dataset artificial
torch.manual_seed(0)
np.random.seed(0) # Asegurar reproducibilidad del ruido
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.4 * torch.randn_like(X)

# -----------------------------------------
# 🧠 Capa con Memoria de Fallas (errores grandes)
# -----------------------------------------
class CapaConMemoriaErrores(nn.Module):
    def __init__(self, in_features, out_features, k=10, lr_extra=0.01):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.pesos = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.k = k
        self.lr_extra = lr_extra  # tasa de aprendizaje extra para fallos
        # buffer_fallas almacena tuplas de (entrada, error_magnitud)
        self.buffer_fallas = deque(maxlen=k)

    def forward(self, x):
        self.entrada_actual = x.detach()
        z = F.linear(x, self.pesos, self.bias)
        self.activacion = torch.tanh(z)
        return self.activacion

    def actualizar_memoria(self, x_batch, error_batch_global):
        """
        Almacena las 'k' muestras con los mayores errores en el batch actual.
        """
        # Calcular el error absoluto promedio por muestra
        # Usamos abs().mean(dim=1) para obtener una magnitud de error por cada fila (muestra)
        errores_por_muestra = error_batch_global.abs().mean(dim=1)

        # Emparejar error con la entrada y mantenerlas en una lista para ordenar
        muestras_con_error = []
        for i in range(x_batch.size(0)):
            muestras_con_error.append((x_batch[i].detach(), errores_por_muestra[i].item()))

        # Ordenar por error de mayor a menor y añadir al buffer (los más grandes se quedan)
        # Añadimos solo si el error es significativo (opcional, para filtrar ruido)
        muestras_con_error.sort(key=lambda item: item[1], reverse=True)

        # Añadir al buffer, manteniendo solo los 'k' más grandes si el buffer no está lleno,
        # o los más grandes que superen a los actuales en el buffer (lógica más compleja)
        # Por simplicidad, aquí solo añadimos todos los del batch y el deque se encarga del maxlen.
        # Si quieres solo los 'k' mayores del batch, sería algo como:
        for i in range(min(self.k, len(muestras_con_error))):
             self.buffer_fallas.append(muestras_con_error[i])


    def aplicar_correccion_memoria(self):
        """
        Recorre la memoria de fallas y realiza correcciones adicionales en los pesos
        para mejorar el rendimiento en esos puntos de "falla" pasados.
        """
        if not self.buffer_fallas:
            return

        # Extraer entradas del buffer
        entradas_fallas = torch.stack([item[0] for item in self.buffer_fallas]).to(self.pesos.device)

        # Calcular las activaciones de estas entradas con los pesos actuales
        # No queremos que esto afecte el gradiente principal de la red
        with torch.no_grad():
            z_fallas = F.linear(entradas_fallas, self.pesos, self.bias)
            activacion_fallas = torch.tanh(z_fallas)

        # Aquí definimos la "dirección deseada" para estas fallas.
        # Por ejemplo, si queremos que la neurona sea "menos activa" para ciertos inputs
        # o que su activación se acerque a un valor deseado para esa falla.
        # En este ejemplo, el error_local es simplemente la activación, sugiriendo que queremos reducirla.
        # Esto es una simplificación de una regla de aprendizaje más compleja.
        # Un enfoque más general sería:
        # error_deseado = 0.0 # O algún otro valor objetivo
        # error_local = activacion_fallas - error_deseado

        # Para el ejemplo, seguimos tu lógica de corregir hacia una activación más neutra.
        # Si la activación es positiva, queremos reducir pesos. Si es negativa, queremos aumentarlos.
        error_local = activacion_fallas # Queremos que estas activaciones se muevan hacia cero

        # Calcular un pseudo-gradiente para estos puntos de falla
        # Esto es una forma de regla de aprendizaje local.
        # El gradiente de la activación con respecto a los pesos/bias
        # multiplicado por el error local.
        grad_w_extra = torch.matmul(error_local.T, entradas_fallas) / entradas_fallas.size(0)
        grad_b_extra = error_local.mean(dim=0)

        # Aplicar la corrección adicional
        with torch.no_grad():
            self.pesos.data -= self.lr_extra * grad_w_extra
            self.bias.data -= self.lr_extra * grad_b_extra

# -----------------------------------------
# Red principal que usa la CapaConMemoriaErrores
# -----------------------------------------
class RedMemoriaErrores(nn.Module):
    def __init__(self):
        super().__init__()
        # La primera capa usa la capa con memoria
        self.capa1 = CapaConMemoriaErrores(1, 30, k=15, lr_extra=0.005) # k=15 fallas, lr_extra=0.005
        # Capa de salida estándar
        self.salida = nn.Linear(30, 1)

    def forward(self, x):
        h = self.capa1(x)
        return self.salida(h)

# -----------------------------------------
# Red Estándar para comparación justa
# -----------------------------------------
class RedEstandar(nn.Module):
    def __init__(self):
        super().__init__()
        # La primera capa es una nn.Linear estándar con la misma configuración
        self.capa1 = nn.Linear(1, 30)
        # Capa de salida estándar
        self.salida = nn.Linear(30, 1)

    def forward(self, x):
        # Aplicar tanh explícitamente para que la activación sea la misma que CapaConMemoriaErrores
        h = torch.tanh(self.capa1(x))
        return self.salida(h)

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01 # Tasa de aprendizaje para Adam
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con Memoria de Errores
# -----------------------------------------
print("--- Entrenando Red con Memoria de Errores ---")
modelo_memoria = RedMemoriaErrores()
# Optimizamos TODOS los parámetros del modelo, incluyendo los de CapaConMemoriaErrores
opt_memoria = torch.optim.Adam(modelo_memoria.parameters(), lr=LEARNING_RATE)
historial_loss_memoria = []

for epoch in range(EPOCHS):
    y_pred_memoria = modelo_memoria(X)
    # Usar el nombre de variable correcto LOSS_FN
    loss_memoria = LOSS_FN(y_pred_memoria, y)
    historial_loss_memoria.append(loss_memoria.item())

    opt_memoria.zero_grad()
    loss_memoria.backward() # Calcula gradientes para Adam

    opt_memoria.step() # Aplica el paso de Adam

    # 🚨 Lógica de memoria personalizada: Actualizar y aplicar corrección
    # Pasa el batch original X y el error global del modelo
    error_batch_global = y_pred_memoria - y
    modelo_memoria.capa1.actualizar_memoria(X, error_batch_global)
    modelo_memoria.capa1.aplicar_correccion_memoria()

    if epoch % 50 == 0:
        print(f"Memoria - Época {epoch} - Pérdida: {loss_memoria.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar()
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    # Usar el nombre de variable correcto LOSS_FN
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step() # Aplica el paso de Adam

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_memoria = modelo_memoria(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_memoria, label="Predicción con Memoria", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_memoria, label="Pérdida con Memoria", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización de las diferencias en valores pequeños

plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Memoria): {historial_loss_memoria[-1]:.6f}")
print(f"Pérdida final (Red Estándar):    {historial_loss_estandar[-1]:.6f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Definición de la Neurona con Memoria de Gradiente ---
class NeuronaMemoriaGradiente(nn.Module):
    def __init__(self, in_features, out_features, memoria_tamano=10, alfa=0.01):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.w = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.b = nn.Parameter(torch.zeros(out_features))

        self.historial_grad_w = deque(maxlen=memoria_tamano)
        self.historial_grad_b = deque(maxlen=memoria_tamano)
        self.historial_error = deque(maxlen=memoria_tamano) # Se guarda, aunque no se usa en este método

        self.alfa = alfa

    def forward(self, x):
        self.entrada_actual = x.detach()
        z = F.linear(x, self.w, self.b)
        self.activacion = torch.tanh(z) # Usamos tanh para que sea comparable con nn.Linear + tanh
        return self.activacion

    def guardar_memoria(self, grad_w, grad_b, error_batch):
        # Asegúrate de que los gradientes no sean None antes de guardar
        if grad_w is not None and grad_b is not None:
            self.historial_grad_w.append(grad_w.detach().clone())
            self.historial_grad_b.append(grad_b.detach().clone())
            self.historial_error.append(error_batch.detach().clone())

    def aplicar_correccion_memoria(self):
        if not self.historial_grad_w:
            return

        prom_grad_w = torch.stack(list(self.historial_grad_w)).mean(dim=0)
        prom_grad_b = torch.stack(list(self.historial_grad_b)).mean(dim=0)

        with torch.no_grad():
            self.w.data -= self.alfa * prom_grad_w
            self.b.data -= self.alfa * prom_grad_b

# --- 2. Red que utiliza la neurona con memoria (RedConMemoria) ---
class RedConMemoria(nn.Module):
    def __init__(self):
        super().__init__()
        # La primera capa usa tu NeuronaMemoriaGradiente
        self.neurona_memoria = NeuronaMemoriaGradiente(1, 20)
        # La capa de salida es estándar
        self.salida = nn.Linear(20, 1)

    def forward(self, x):
        h = self.neurona_memoria(x)
        return self.salida(h)

# --- 3. Red Estándar (para una comparación justa) ---
class RedEstandar(nn.Module):
    def __init__(self):
        super().__init__()
        # La primera capa es una nn.Linear estándar, con la misma configuración
        self.capa_oculta = nn.Linear(1, 20)
        # La capa de salida es estándar
        self.salida = nn.Linear(20, 1)

    def forward(self, x):
        # Aplicamos torch.tanh explícitamente para que la activación sea la misma
        h = torch.tanh(self.capa_oculta(x))
        return self.salida(h)

# -----------------------------------------
# Dataset y Preparación de Datos
# -----------------------------------------
torch.manual_seed(0) # Para reproducibilidad de los resultados
np.random.seed(0)

N_PUNTOS = 300
X = torch.linspace(-3, 3, N_PUNTOS).unsqueeze(1)
y = X.pow(3) - 0.5 * X + 0.3 * torch.randn_like(X) # Función cúbica con ruido

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con Memoria
# -----------------------------------------
print("--- Entrenando Red con Memoria de Gradiente ---")
modelo_memoria = RedConMemoria()
opt_memoria = torch.optim.Adam(modelo_memoria.parameters(), lr=LEARNING_RATE)
historial_loss_memoria = []

for epoch in range(EPOCHS):
    y_pred_memoria = modelo_memoria(X)
    loss_memoria = LOSS_FN(y_pred_memoria, y)
    historial_loss_memoria.append(loss_memoria.item())

    opt_memoria.zero_grad()
    loss_memoria.backward()

    # Guardar gradientes de la neurona de memoria antes del paso de optimización
    grad_w_memoria = modelo_memoria.neurona_memoria.w.grad
    grad_b_memoria = modelo_memoria.neurona_memoria.b.grad
    error_batch_memoria = y_pred_memoria - y

    opt_memoria.step() # Aplica el paso de Adam

    # Guardar historial y aplicar corrección secundaria de la neurona de memoria
    modelo_memoria.neurona_memoria.guardar_memoria(grad_w_memoria, grad_b_memoria, error_batch_memoria)
    modelo_memoria.neurona_memoria.aplicar_correccion_memoria()

    if epoch % 50 == 0:
        print(f"Memoria - Época {epoch} - Pérdida: {loss_memoria.item():.4f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar()
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step() # Aplica el paso de Adam

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.4f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_memoria = modelo_memoria(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_memoria, label="Predicción con Memoria", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_memoria, label="Pérdida con Memoria", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización de las diferencias en valores pequeños

plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Memoria): {historial_loss_memoria[-1]:.4f}")
print(f"Pérdida final (Red Estándar):    {historial_loss_estandar[-1]:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import matplotlib.pyplot as plt

class NeuronaMemoriaGradiente(nn.Module):
    def __init__(self, in_features, out_features, memoria_tamano=10, alfa=0.01):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.w = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.b = nn.Parameter(torch.zeros(out_features))

        # Memoria de gradientes y pesos
        self.historial_grad_w = deque(maxlen=memoria_tamano)
        self.historial_grad_b = deque(maxlen=memoria_tamano)
        self.historial_error = deque(maxlen=memoria_tamano)

        self.alfa = alfa  # factor de ajuste con la memoria

    def forward(self, x):
        self.entrada_actual = x.detach()
        z = F.linear(x, self.w, self.b)
        self.activacion = torch.tanh(z)
        return self.activacion

    def guardar_memoria(self, grad_w, grad_b, error_batch):
        self.historial_grad_w.append(grad_w.detach().clone())
        self.historial_grad_b.append(grad_b.detach().clone())
        self.historial_error.append(error_batch.detach().clone())

    def aplicar_correccion_memoria(self):
        if not self.historial_grad_w:
            return

        # Media de gradientes anteriores
        prom_grad_w = torch.stack(list(self.historial_grad_w)).mean(dim=0)
        prom_grad_b = torch.stack(list(self.historial_grad_b)).mean(dim=0)

        # Ajuste leve adicional en dirección promedio
        with torch.no_grad():
            self.w.data -= self.alfa * prom_grad_w
            self.b.data -= self.alfa * prom_grad_b

# -----------------------------------------
# Red simple usando la neurona con memoria
# -----------------------------------------
class RedConMemoria(nn.Module):
    def __init__(self):
        super().__init__()
        self.neurona = NeuronaMemoriaGradiente(1, 20)
        self.salida = nn.Linear(20, 1)

    def forward(self, x):
        h = self.neurona(x)
        return self.salida(h)

# -----------------------------------------
# Dataset y entrenamiento
# -----------------------------------------
torch.manual_seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - 0.5 * X + 0.3 * torch.randn_like(X)

modelo = RedConMemoria()
opt = torch.optim.Adam(modelo.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

historial_loss = []

for epoch in range(300):
    y_pred = modelo(X)
    loss = loss_fn(y_pred, y)
    historial_loss.append(loss.item())

    opt.zero_grad()
    loss.backward()

    # Guardar gradientes antes del paso de optimización
    grad_w = modelo.neurona.w.grad.clone()
    grad_b = modelo.neurona.b.grad.clone()
    error_batch = y_pred - y

    opt.step()

    # Guardar historial y aplicar corrección secundaria
    modelo.neurona.guardar_memoria(grad_w, grad_b, error_batch)
    modelo.neurona.aplicar_correccion_memoria()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} - Loss: {loss.item():.4f}")

# -----------------------------------------
# Visualización
# -----------------------------------------
with torch.no_grad():
    y_hat = modelo(X)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.title("Predicción vs Real")
plt.plot(X.numpy(), y.numpy(), label="Real")
plt.plot(X.numpy(), y_hat.numpy(), label="Predicho")
plt.legend()

plt.subplot(1, 2, 2)
plt.title("Evolución del Loss")
plt.plot(historial_loss)
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Definición de la Neurona con Memoria de Gradiente (Igual que tu código) ---
class NeuronaMemoriaGradiente(nn.Module):
    def __init__(self, in_features, out_features, memoria_tamano=10, alfa=0.01):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.w = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.b = nn.Parameter(torch.zeros(out_features))

        self.historial_grad_w = deque(maxlen=memoria_tamano)
        self.historial_grad_b = deque(maxlen=memoria_tamano)
        self.historial_error = deque(maxlen=memoria_tamano) # Aunque no se usa directamente en aplicar_correccion

        self.alfa = alfa

    def forward(self, x):
        self.entrada_actual = x.detach()
        z = F.linear(x, self.w, self.b)
        self.activacion = torch.tanh(z)
        return self.activacion

    def guardar_memoria(self, grad_w, grad_b, error_batch):
        # Asegúrate de que los gradientes no sean None antes de guardar
        if grad_w is not None and grad_b is not None:
            self.historial_grad_w.append(grad_w.detach().clone())
            self.historial_grad_b.append(grad_b.detach().clone())
            self.historial_error.append(error_batch.detach().clone())

    def aplicar_correccion_memoria(self):
        if not self.historial_grad_w:
            return

        prom_grad_w = torch.stack(list(self.historial_grad_w)).mean(dim=0)
        prom_grad_b = torch.stack(list(self.historial_grad_b)).mean(dim=0)

        with torch.no_grad():
            self.w.data -= self.alfa * prom_grad_w
            self.b.data -= self.alfa * prom_grad_b

# --- 2. Definición de la Red con Memoria (Igual que tu código) ---
class RedConMemoria(nn.Module):
    def __init__(self):
        super().__init__()
        # Usamos 1 input, 20 outputs en la primera capa
        self.neurona_memoria = NeuronaMemoriaGradiente(1, 20)
        self.salida = nn.Linear(20, 1) # 20 inputs, 1 output en la capa final

    def forward(self, x):
        h = self.neurona_memoria(x)
        return self.salida(h)

# --- 3. Definición de la Red Estándar (Para Comparación Justa) ---
class RedEstandar(nn.Module):
    def __init__(self):
        super().__init__()
        # La primera capa es un nn.Linear estándar con la misma configuración
        # (1 entrada, 20 salidas) y la misma función de activación.
        self.capa_oculta = nn.Linear(1, 20)
        self.salida = nn.Linear(20, 1)

    def forward(self, x):
        h = torch.tanh(self.capa_oculta(x)) # Aplicar tanh como en NeuronaMemoriaGradiente
        return self.salida(h)

# -----------------------------------------
# Dataset y Preparación
# -----------------------------------------
torch.manual_seed(0) # Semilla para reproducibilidad
np.random.seed(0)

N_PUNTOS = 300
X = torch.linspace(-3, 3, N_PUNTOS).unsqueeze(1)
y = X.pow(3) - 0.5 * X + 0.3 * torch.randn_like(X) # Función cúbica con ruido

# -----------------------------------------
# Configuración del Entrenamiento
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con Memoria
# -----------------------------------------
modelo_memoria = RedConMemoria()
opt_memoria = torch.optim.Adam(modelo_memoria.parameters(), lr=LEARNING_RATE)
historial_loss_memoria = []

print("--- Entrenando Red con Memoria de Gradiente ---")
for epoch in range(EPOCHS):
    y_pred_memoria = modelo_memoria(X)
    loss_memoria = LOSS_FN(y_pred_memoria, y)
    historial_loss_memoria.append(loss_memoria.item())

    opt_memoria.zero_grad()
    loss_memoria.backward()

    # Guardar gradientes de la neurona de memoria antes del paso de optimización
    # Solo si la neurona de memoria tiene gradientes (después de loss.backward())
    grad_w_memoria = modelo_memoria.neurona_memoria.w.grad
    grad_b_memoria = modelo_memoria.neurona_memoria.b.grad
    error_batch_memoria = y_pred_memoria - y

    opt_memoria.step()

    # Guardar historial y aplicar corrección secundaria de la neurona de memoria
    modelo_memoria.neurona_memoria.guardar_memoria(grad_w_memoria, grad_b_memoria, error_batch_memoria)
    modelo_memoria.neurona_memoria.aplicar_correccion_memoria()

    if epoch % 50 == 0:
        print(f"Memoria - Epoch {epoch} - Loss: {loss_memoria.item():.4f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
modelo_estandar = RedEstandar()
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

print("\n--- Entrenando Red Estándar ---")
for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step()

    if epoch % 50 == 0:
        print(f"Estándar - Epoch {epoch} - Loss: {loss_estandar.item():.4f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_memoria = modelo_memoria(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Predicciones: Memoria vs. Estándar")
plt.plot(X_np, y_np, label="Real", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_memoria, label="Predicción con Memoria", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución del Loss
plt.subplot(1, 2, 2)
plt.title("Evolución del Loss (MSE)")
plt.plot(historial_loss_memoria, label="Loss con Memoria", color='red')
plt.plot(historial_loss_estandar, label="Loss Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para apreciar mejor las diferencias en valores pequeños

plt.tight_layout()
plt.show()

# Resumen final de Loss
print(f"\n--- Resumen Final ---")
print(f"Loss final (Red con Memoria): {historial_loss_memoria[-1]:.4f}")
print(f"Loss final (Red Estándar):    {historial_loss_estandar[-1]:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Capa con umbrales dinámicos ---
class CapaUmbralDinamico(nn.Module):
    def __init__(self, in_features, out_features, alpha=0.05, beta=0.01):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        # nn.Parameter makes these tensors trainable by the optimizer
        self.pesos = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.alpha = alpha  # tasa de actualización de activación media
        self.beta = beta    # tasa de ajuste del umbral (bias)

        # Historial de activaciones promedio (media móvil por neurona)
        # register_buffer registers a tensor that should not be considered a model parameter.
        self.register_buffer('media_activacion', torch.zeros(out_features))
        self.register_buffer('media_error', torch.zeros(out_features))

    def forward(self, x):
        # Linear transformation: y = x * W^T + b
        z = F.linear(x, self.pesos, self.bias)
        self.activaciones = torch.tanh(z)  # activación no lineal
        self.ultima_entrada = x.detach() # Detach to avoid tracking gradients for visualization/debugging if needed later
        return self.activaciones

    def ajustar_umbral(self, error_batch):
        """
        Ajusta el bias en función de activaciones recientes y errores.
        error_batch: error de predicción respecto a la salida (assumed shape is [batch_size, out_features])
        """
        # Media de activación por neurona a través del batch
        act_mean = self.activaciones.mean(dim=0).detach()  # [out_features]
        self.media_activacion = (1 - self.alpha) * self.media_activacion + self.alpha * act_mean

        # Error medio atribuido a cada neurona. Assuming error_batch has shape [batch_size, out_features].
        # If error_batch is [batch_size, 1], you might need to adjust how error is attributed.
        # For this simple network, the final layer output is 1, so error_batch would be [batch_size, 1].
        # We need to think about how error from the output layer [batch_size, 1] affects the bias of capa1 [out_features].
        # A simple approach for visualization is to just use the overall batch error, but that's not a true attribution.
        # For a proper backpropagation-based attribution, you'd need gradients w.r.t. activations, which is more complex.
        # Let's use the batch mean absolute error for now, as intended in the original logic, assuming it's broadcasting or handled appropriately.
        # Original code assumes error_batch.mean(dim=0) works to get error per neuron bias, which is only strictly true if error_batch is [batch_size, out_features].
        # Given the network structure (20 -> 1), the error is [batch_size, 1].
        # Let's stick to the original logic's apparent intent, acknowledging potential simplification.
        error_per_output_neuron = error_batch.mean(dim=0).detach() # Shape [1] in this case if error_batch is [batch_size, 1]

        # To adjust the bias of capa1 (out_features=20), we need an error signal of shape [20].
        # The original code error_batch.mean(dim=0) when error_batch is [batch_size, 1] results in shape [1].
        # This won't work for updating media_error which has shape [out_features].
        # Let's simplify the error signal for bias adjustment to use the overall MSE loss value per batch.
        # This is a simplification but aligns better with available information.
        # Or, let's use the gradient of the loss w.r.t. the bias, which is a more standard way.
        # However, the original code doesn't show calculating gradients w.r.t. bias for this adjustment.
        # Let's revert to the most likely intended logic based on the original code, assuming error_batch is somehow made compatible.
        # If error_batch is the gradient w.r.t. the activations of this layer, then mean(dim=0) works.
        # But error_batch is defined as y_pred - y, which is [batch_size, 1].

        # Let's correct the error attribution based on network structure.
        # The error signal for adjusting bias of `capa1` should be related to the gradient flowing *into* `capa1`.
        # A simplified approach, closer to the original intent of using batch stats:
        # Use the mean absolute value of the *overall* batch error as a scalar signal.
        # Then, this scalar signal influences all biases of `capa1` uniformly or scaled by activation magnitude.
        # The original `error_batch.mean(dim=0).detach()` on a [batch_size, 1] tensor yields a single value mean.
        # This single value mean is then used to update `self.media_error` [out_features] via broadcasting.
        # Let's keep this logic for now, as it seems the most likely interpretation of the original code's dimension mismatch.

        # Broadcast the mean error value to match the shape of media_error [out_features]
        mean_batch_error_abs = error_batch.abs().mean().detach() # Scalar
        self.media_error = (1 - self.alpha) * self.media_error + self.alpha * mean_batch_error_abs


        # Ajustar bias dinámicamente hacia estabilización
        # Original logic: ajuste = self.beta * (self.media_activacion + self.media_error)
        # This implies media_error is also [out_features]. Let's stick to the broadcasting interpretation from above.
        # The mean_batch_error_abs scalar will be added to each element of self.media_activacion [out_features].
        ajuste = self.beta * (self.media_activacion + mean_batch_error_abs) # scalar + [out_features] -> [out_features]
        with torch.no_grad():
            self.bias -= ajuste # [out_features] -= [out_features]


# --- 2. Red usando umbrales adaptativos ---
class RedConUmbralDinamico(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = CapaUmbralDinamico(1, 20)
        self.capa2 = nn.Linear(20, 1)

    def forward(self, x):
        self.h = self.capa1(x)
        return self.capa2(self.h)

# --- 3. Entrenamiento ---
modelo = RedConUmbralDinamico()

# Corrected: Pass the parameters directly
# model.parameters() is an iterator over ALL parameters in the model.
# You can collect specific parameter groups if needed, but model.parameters() is usually sufficient.
# If you only wanted to optimize capa1.pesos and all of capa2:
# opt = torch.optim.Adam([{'params': modelo.capa1.pesos}, {'params': modelo.capa2.parameters()}], lr=0.01)
# If you want to optimize all parameters in the model:
opt = torch.optim.Adam(modelo.parameters(), lr=0.01) # This is the standard and likely intended approach

loss_fn = nn.MSELoss()
hist = []

for epoch in range(300):
    y_pred = modelo(X)
    loss = loss_fn(y_pred, y)
    hist.append(loss.item())

    # Backward y optimización
    opt.zero_grad()
    loss.backward()
    opt.step()

    # Ajuste dinámico del bias basado en activaciones y errores
    error_batch = y_pred - y # error_batch will be [batch_size, 1]
    modelo.capa1.ajustar_umbral(error_batch) # Pass the error to the layer's adjustment method

    if epoch % 50 == 0:
        print(f"Época {epoch} - Loss: {loss.item():.4f}")

# --- 4. Visualización ---
with torch.no_grad():
    pred = modelo(X)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(X.numpy(), y.numpy(), label='Real')
plt.plot(X.numpy(), pred.numpy(), label='Predicho')
plt.title("Regresión con umbral dinámico")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist)
plt.title("Pérdida durante entrenamiento")
plt.xlabel("Época")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
# Ajustamos la semilla de numpy también si random.randn() es usado en alguna parte no controlada por torch
# Aunque en este código solo se usa torch.randn_like, es buena práctica.
# import numpy as np # Descomentar si usas numpy.random en otros lugares
# np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Capa con umbrales dinámicos ---
class CapaUmbralDinamico(nn.Module):
    def __init__(self, in_features, out_features, alpha=0.05, beta=0.01):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.pesos = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features)) # Este es el umbral

        self.alpha = alpha  # tasa de actualización de la media de activación
        self.beta = beta    # tasa de ajuste del bias

        # Historial de activaciones promedio y error promedio (media móvil por neurona)
        self.register_buffer('media_activacion', torch.zeros(out_features))
        self.register_buffer('media_error', torch.zeros(out_features)) # Inicializado a ceros

    def forward(self, x):
        self.entrada_actual = x.detach() # Para depuración o uso futuro, no directamente en el forward pass
        z = F.linear(x, self.pesos, self.bias)
        self.activaciones = torch.tanh(z)
        return self.activaciones

    def ajustar_umbral(self, error_batch_global):
        """
        Ajusta el bias de la capa en función de las activaciones recientes
        y el error global del batch.
        error_batch_global: Tensor de error de la salida de la red [batch_size, 1].
        """
        # Calcular la media de las activaciones de esta capa en el batch actual
        act_mean_current_batch = self.activaciones.mean(dim=0).detach() # [out_features]
        # Actualizar la media móvil de las activaciones
        self.media_activacion.mul_((1 - self.alpha)).add_(act_mean_current_batch, alpha=self.alpha)

        # Usar la magnitud del error global del batch como señal para el ajuste del bias.
        # Esto es una simplificación, ya que el error global [batch_size, 1]
        # no se atribuye directamente a cada una de las 'out_features' neuronas de esta capa.
        # Sin embargo, sigue la lógica implícita de usar el error del batch.
        mean_abs_global_error = error_batch_global.abs().mean().detach() # Escalar

        # Actualizar la media móvil del error.
        # El escalar `mean_abs_global_error` se propaga a todas las `out_features`
        # de `self.media_error` a través del broadcasting.
        self.media_error.mul_((1 - self.alpha)).add_(mean_abs_global_error, alpha=self.alpha)

        # Ajuste dinámico del bias.
        # La lógica de ajuste aquí es: ajuste = beta * (media_activacion + media_error).
        # Esto implica que si las activaciones son altas o el error global es alto,
        # el bias se ajusta para mover las activaciones hacia cero (debido al tanh).
        # Si la media_activacion es positiva, el bias disminuye (mueve z hacia la izquierda).
        # Si la media_activacion es negativa, el bias aumenta (mueve z hacia la derecha).
        # Con media_error positiva, el bias siempre disminuye, haciendo las neuronas más difíciles de activar.
        ajuste = self.beta * (self.media_activacion + self.media_error) # [out_features] + [out_features] -> [out_features]
        with torch.no_grad():
            self.bias.sub_(ajuste) # Resta el ajuste al bias


# --- 2. Red que utiliza la CapaUmbralDinamico ---
class RedConUmbralDinamico(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = CapaUmbralDinamico(1, 20) # 1 entrada, 20 salidas
        self.capa2 = nn.Linear(20, 1) # 20 entradas, 1 salida

    def forward(self, x):
        self.h = self.capa1(x)
        return self.capa2(self.h)

# --- 3. Red Estándar para una comparación justa ---
class RedEstandar(nn.Module):
    def __init__(self):
        super().__init__()
        # La primera capa es un nn.Linear estándar con la misma configuración
        self.capa1 = nn.Linear(1, 20)
        # La capa de salida es estándar
        self.capa2 = nn.Linear(20, 1)

    def forward(self, x):
        # Aplicar torch.tanh explícitamente para que la activación sea la misma que CapaUmbralDinamico
        h = torch.tanh(self.capa1(x))
        return self.capa2(h)

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con Umbral Dinámico
# -----------------------------------------
print("--- Entrenando Red con Umbral Dinámico ---")
modelo_dinamico = RedConUmbralDinamico()
opt_dinamico = torch.optim.Adam(modelo_dinamico.parameters(), lr=LEARNING_RATE)
historial_loss_dinamico = []

for epoch in range(EPOCHS):
    y_pred_dinamico = modelo_dinamico(X)
    loss_dinamico = LOSS_FN(y_pred_dinamico, y)
    historial_loss_dinamico.append(loss_dinamico.item())

    opt_dinamico.zero_grad()
    loss_dinamico.backward()
    opt_dinamico.step() # Aplica el paso de Adam

    # 🚨 Lógica de ajuste de umbral personalizada
    error_batch_global = y_pred_dinamico - y # Error global del batch
    modelo_dinamico.capa1.ajustar_umbral(error_batch_global)

    if epoch % 50 == 0:
        print(f"Dinámico - Época {epoch} - Pérdida: {loss_dinamico.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar()
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step() # Aplica el paso de Adam

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_dinamico = modelo_dinamico(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_dinamico, label="Predicción Dinámica", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_dinamico, label="Pérdida Dinámica", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización de las diferencias en valores pequeños

plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Umbral Dinámico): {historial_loss_dinamico[-1]:.6f}")
print(f"Pérdida final (Red Estándar):           {historial_loss_estandar[-1]:.6f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np # Importar numpy para np.random.seed si es necesario

# Dataset sintético simple (regresión)
torch.manual_seed(0)
np.random.seed(0) # Para reproducibilidad si se usa alguna función de numpy.random
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Capa con LR por neurona ---
# NOTA: Esta clase asume que los pesos serán actualizados externamente,
# o manejará su propia actualización fuera del optimizador global.
# Para una comparación justa, la "LR por neurona" se aplicará *después*
# del paso del optimizador principal (Adam) a los parámetros de esta capa.
class CapaLRporNeurona(nn.Module):
    def __init__(self, in_features, out_features, base_lr_ajuste=0.01, eps=1e-8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.pesos = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.base_lr_ajuste = base_lr_ajuste # Tasa de aprendizaje para el ajuste de esta capa
        self.eps = eps

        # Historial de acumulación de gradientes (RMSprop-like)
        self.register_buffer('G_pesos', torch.zeros_like(self.pesos))
        self.register_buffer('G_bias', torch.zeros_like(self.bias)) # También para bias

    def forward(self, x):
        # Guardamos la entrada para calcular gradientes de pesos
        self.ultima_entrada = x.detach()
        return F.linear(x, self.pesos, self.bias)

    def aplicar_ajuste_personalizado(self):
        """
        Calcula y aplica un ajuste adicional a los pesos/bias de esta capa
        basado en la lógica de LR por neurona, *después* del paso del optimizador principal.
        """
        # Asegurarse de que los gradientes de esta capa existan después del loss.backward()
        if self.pesos.grad is None or self.bias.grad is None:
            return

        with torch.no_grad(): # No queremos que estas operaciones sean rastreadas por el grafo de PyTorch
            # Gradientes después del backward pass (ya calculados por el optimizador principal)
            grad_pesos_main = self.pesos.grad
            grad_bias_main = self.bias.grad

            # Acumular cuadrados de gradientes para pesos y bias
            self.G_pesos.mul_(0.9).addcmul_(grad_pesos_main, grad_pesos_main, value=0.1) # RMSprop-like update
            self.G_bias.mul_(0.9).addcmul_(grad_bias_main, grad_bias_main, value=0.1) # RMSprop-like update

            # Calcular LR individual por peso/bias
            lrs_individuales_pesos = self.base_lr_ajuste / (torch.sqrt(self.G_pesos) + self.eps)
            lrs_individuales_bias = self.base_lr_ajuste / (torch.sqrt(self.G_bias) + self.eps)

            # Aplicar el ajuste adicional
            self.pesos.sub_(lrs_individuales_pesos * grad_pesos_main)
            self.bias.sub_(lrs_individuales_bias * grad_bias_main)

            # Opcional: Reiniciar gradientes después del ajuste manual para evitar doble contabilización
            # self.pesos.grad.zero_()
            # self.bias.grad.zero_()


# --- 2. Red neuronal usando la CapaLRporNeurona ---
class RedConLRNeuronas(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = CapaLRporNeurona(1, 20, base_lr_ajuste=0.01) # base_lr para el ajuste personalizado
        self.act = nn.Tanh()
        self.capa2 = nn.Linear(20, 1)

    def forward(self, x):
        h = self.capa1(x)
        self.activacion_capa1 = self.act(h) # Guardamos la activación para poder acceder a sus gradientes si fuera necesario
        return self.capa2(self.activacion_capa1)

# --- 3. Red Estándar para una comparación justa ---
class RedEstandar(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(1, 20)
        self.act = nn.Tanh()
        self.capa2 = nn.Linear(20, 1)

    def forward(self, x):
        h = self.capa1(x)
        return self.capa2(self.act(h))

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE_ADAM = 0.01 # Tasa de aprendizaje para el optimizador Adam principal
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con LR por Neuronas
# -----------------------------------------
print("--- Entrenando Red con LR por Neuronas ---")
modelo_lr_neuronas = RedConLRNeuronas()
# Optimizamos TODOS los parámetros del modelo con Adam
opt_lr_neuronas = torch.optim.Adam(modelo_lr_neuronas.parameters(), lr=LEARNING_RATE_ADAM)
historial_loss_lr_neuronas = []

for epoch in range(EPOCHS):
    y_pred_lr = modelo_lr_neuronas(X)
    loss_lr = LOSS_FN(y_pred_lr, y)
    historial_loss_lr_neuronas.append(loss_lr.item())

    opt_lr_neuronas.zero_grad()
    loss_lr.backward() # Calcula los gradientes para todas las capas

    opt_lr_neuronas.step() # Adam aplica sus actualizaciones

    # 🚨 Lógica de ajuste personalizada de la CapaLRporNeurona
    # Se aplica *después* del paso de Adam, usando los gradientes ya calculados.
    modelo_lr_neuronas.capa1.aplicar_ajuste_personalizado()

    if epoch % 50 == 0:
        print(f"LR por Neuronas - Época {epoch} - Pérdida: {loss_lr.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar()
# Optimizamos TODOS los parámetros del modelo estándar con Adam
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE_ADAM)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step() # Adam aplica sus actualizaciones

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    pred_lr_neuronas = modelo_lr_neuronas(X).numpy()
    pred_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, pred_lr_neuronas, label="Predicción con LR por Neurona", color='red', linestyle='--')
plt.plot(X_np, pred_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_lr_neuronas, label="Pérdida con LR por Neurona", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización de las diferencias en valores pequeños

plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con LR por Neurona): {historial_loss_lr_neuronas[-1]:.6f}")
print(f"Pérdida final (Red Estándar):           {historial_loss_estandar[-1]:.6f}")

## Neurona con memoria y atención

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import matplotlib.pyplot as plt

class NeuronaMemoriaAtencion(nn.Module):
    def __init__(self, in_features, out_features, memoria_tamano=10, alfa=0.05):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.w = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.b = nn.Parameter(torch.zeros(out_features))

        self.memoria_inputs = deque(maxlen=memoria_tamano)
        self.memoria_grad_w = deque(maxlen=memoria_tamano)

        self.alfa = alfa

    def forward(self, x):
        self.entrada_actual = x.detach()
        z = F.linear(x, self.w, self.b)
        self.activacion = torch.tanh(z)
        return self.activacion

    def guardar_memoria(self, entrada, grad_w):
        self.memoria_inputs.append(entrada.detach().clone())
        self.memoria_grad_w.append(grad_w.detach().clone())

    def atencion_similitud(self, x_actual):
        """
        Calcula pesos de atención basados en similitud de coseno entre el input actual y los inputs de la memoria.
        """
        if not self.memoria_inputs:
            return None  # no hay memoria aún

        entradas = torch.stack(list(self.memoria_inputs))  # [M, in_features]
        gradientes = torch.stack(list(self.memoria_grad_w))  # [M, out, in]

        # Similitud de coseno entre x_actual y cada input almacenado
        x_norm = F.normalize(x_actual, dim=1)  # [B, in]
        entradas_norm = F.normalize(entradas, dim=1)  # [M, in]

        similitudes = torch.matmul(x_norm, entradas_norm.T)  # [B, M]

        pesos = F.softmax(similitudes, dim=1)  # [B, M]

        # Mezcla ponderada de gradientes usando atención
        mezcla_grad = torch.zeros_like(self.w)
        for i in range(pesos.size(1)):  # sobre memoria
            mezcla_grad += pesos[:, i].mean() * gradientes[i]

        return mezcla_grad

    def aplicar_correccion_memoria(self):
        with torch.no_grad():
            correccion = self.atencion_similitud(self.entrada_actual)
            if correccion is not None:
                self.w.data -= self.alfa * correccion  # ajuste extra inteligente

# Red mínima que usa esta neurona
class RedAtencion(nn.Module):
    def __init__(self):
        super().__init__()
        self.neurona = NeuronaMemoriaAtencion(1, 20)
        self.out = nn.Linear(20, 1)

    def forward(self, x):
        h = self.neurona(x)
        return self.out(h)

# ---------------------------------------
# Datos y entrenamiento
# ---------------------------------------
torch.manual_seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - 0.5 * X + 0.3 * torch.randn_like(X)

modelo = RedAtencion()
opt = torch.optim.Adam(modelo.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

losses = []

for epoch in range(300):
    y_pred = modelo(X)
    loss = loss_fn(y_pred, y)
    losses.append(loss.item())

    opt.zero_grad()
    loss.backward()

    # guardar gradientes + input
    grad_w = modelo.neurona.w.grad.clone()
    modelo.neurona.guardar_memoria(X.mean(dim=0), grad_w)

    opt.step()

    # aplicar corrección por atención
    modelo.neurona.aplicar_correccion_memoria()

    if epoch % 50 == 0:
        print(f"Epoch {epoch} - Loss: {loss.item():.4f}")

# ---------------------------------------
# Visualización
# ---------------------------------------
with torch.no_grad():
    y_hat = modelo(X)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.title("Predicción vs Real")
plt.plot(X.numpy(), y.numpy(), label="Real")
plt.plot(X.numpy(), y_hat.numpy(), label="Predicho")
plt.legend()

plt.subplot(1, 2, 2)
plt.title("Evolución del Loss")
plt.plot(losses)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import matplotlib.pyplot as plt
import numpy as np # Para asegurar la reproducibilidad del ruido si se usa

# Dataset sintético
torch.manual_seed(0)
np.random.seed(0) # Para la reproducibilidad del ruido
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - 0.5 * X + 0.3 * torch.randn_like(X)

# --- 1. Capa con Memoria y Atención ---
class NeuronaMemoriaAtencion(nn.Module):
    def __init__(self, in_features, out_features, memoria_tamano=10, alfa=0.05):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.w = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.b = nn.Parameter(torch.zeros(out_features))

        # Almacenaremos el promedio del batch de entrada y los gradientes de pesos
        self.memoria_inputs_avg = deque(maxlen=memoria_tamano)
        self.memoria_grad_w = deque(maxlen=memoria_tamano)

        self.alfa = alfa # Factor de ajuste para la corrección por atención

    def forward(self, x):
        self.entrada_actual = x.detach() # Guardar el input actual (batch completo)
        z = F.linear(x, self.w, self.b)
        self.activacion = torch.tanh(z) # Usamos tanh para que sea comparable con nn.Linear + tanh
        return self.activacion

    def guardar_memoria(self, entrada_batch, grad_w_de_capa):
        """
        Calcula y guarda el promedio del batch de entrada y los gradientes de pesos de esta capa.
        """
        if grad_w_de_capa is not None: # Asegurarse de que el gradiente exista
            # Calcular el promedio del batch de entrada y guardarlo
            entrada_promedio = entrada_batch.mean(dim=0, keepdim=True) # [1, in_features]
            self.memoria_inputs_avg.append(entrada_promedio.detach().clone())
            self.memoria_grad_w.append(grad_w_de_capa.detach().clone())

    def atencion_similitud(self, x_actual_batch):
        """
        Calcula pesos de atención basados en similitud de coseno entre el input actual
        (su promedio) y los inputs promedio almacenados en memoria.
        Luego, genera una mezcla ponderada de los gradientes memorizados.
        """
        if not self.memoria_inputs_avg:
            return None  # No hay memoria aún

        # Promedio del input actual del batch
        x_actual_promedio = x_actual_batch.mean(dim=0, keepdim=True) # [1, in_features]

        # Apilar los promedios de inputs memorizados y los gradientes memorizados
        entradas_memorizadas_avg = torch.stack(list(self.memoria_inputs_avg)).squeeze(1) # [M, in_features]
        gradientes_memorizados = torch.stack(list(self.memoria_grad_w))  # [M, out_features, in_features]

        # Asegurarse de que las normas sean correctas para la similitud de coseno
        # Normalizar a lo largo de la dimensión de features (dim=1 para los tensores [..., in_features])
        x_actual_norm = F.normalize(x_actual_promedio, dim=1, p=2, eps=1e-12)  # [1, in_features]
        entradas_memorizadas_norm = F.normalize(entradas_memorizadas_avg, dim=1, p=2, eps=1e-12)  # [M, in_features]

        # Calcular similitudes de coseno
        # Matmul: [1, in_features] @ [in_features, M] -> [1, M]
        similitudes = torch.matmul(x_actual_norm, entradas_memorizadas_norm.T)

        # Aplicar Softmax para obtener pesos de atención
        pesos_atencion = F.softmax(similitudes, dim=1)  # [1, M]

        # Crear una mezcla ponderada de los gradientes memorizados
        # gradientes_memorizados tiene forma [M, out, in]
        # pesos_atencion tiene forma [1, M]
        # Queremos sum_m(pesos_atencion[0, m] * gradientes_memorizados[m, out, in]) -> [out, in]

        # Reshape pesos_atencion to [M, 1, 1] for broadcasting with [M, out, in]
        weights_for_mul = pesos_atencion.T.unsqueeze(-1) # Shape: [len(memoria), 1, 1]

        # Perform weighted sum
        mezcla_grad = torch.sum(weights_for_mul * gradientes_memorizados, dim=0)
        # Resulting shape: [out_features, in_features] - This is the desired shape.

        return mezcla_grad


    def aplicar_correccion_memoria(self):
        """
        Aplica un ajuste adicional a los pesos basado en la memoria atendida.
        """
        with torch.no_grad(): # Esto debe hacerse fuera del grafo de cómputo de gradientes
            # self.entrada_actual es el batch completo [B, in_features]
            correccion = self.atencion_similitud(self.entrada_actual)
            if correccion is not None:
                # Asegurarse de que 'correccion' tiene la misma forma que 'self.w.data'
                if correccion.shape == self.w.data.shape:
                    self.w.data -= self.alfa * correccion # Ajuste extra inteligente
                    # No ajustamos el bias con la memoria de grad_w en esta implementación,
                    # si quieres ajustar el bias, necesitarías guardar grad_b también.
                else:
                     print(f"Advertencia: La corrección calculada ({correccion.shape}) no coincide con la forma de los pesos ({self.w.data.shape}). Saltando ajuste.")


# --- 2. Red que usa la neurona con memoria y atención (RedAtencion) ---
class RedAtencion(nn.Module):
    def __init__(self):
        super().__init__()
        self.neurona = NeuronaMemoriaAtencion(1, 20) # 1 input, 20 outputs para la capa oculta
        self.out = nn.Linear(20, 1) # 20 inputs, 1 output para la capa final

    def forward(self, x):
        h = self.neurona(x)
        return self.out(h)

# --- 3. Red Estándar para una comparación justa ---
class RedEstandar(nn.Module):
    def __init__(self):
        super().__init__()
        # La primera capa es un nn.Linear estándar con la misma configuración
        self.capa1 = nn.Linear(1, 20)
        # La capa de salida es estándar
        self.out = nn.Linear(20, 1)

    def forward(self, x):
        # Aplicamos torch.tanh explícitamente para que la activación sea la misma
        h = torch.tanh(self.capa1(x))
        return self.out(h)

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con Memoria y Atención
# -----------------------------------------
print("--- Entrenando Red con Memoria y Atención ---")
modelo_atencion = RedAtencion()
# El optimizador principal solo optimiza los parámetros normales (w, b)
# La corrección por memoria se aplica manualmente fuera del optimizador
opt_atencion = torch.optim.Adam(modelo_atencion.parameters(), lr=LEARNING_RATE)
historial_loss_atencion = []

for epoch in range(EPOCHS):
    y_pred_atencion = modelo_atencion(X)
    loss_atencion = LOSS_FN(y_pred_atencion, y)
    historial_loss_atencion.append(loss_atencion.item())

    opt_atencion.zero_grad()
    loss_atencion.backward() # Calcula gradientes para todas las capas

    # Guardar gradientes e input ANTES del paso de optimización principal (Adam)
    # Esto captura los gradientes calculados por el backward pass normal.
    grad_w_actual = modelo_atencion.neurona.w.grad.clone() if modelo_atencion.neurona.w.grad is not None else None
    # Pasamos el batch completo X para que guardar_memoria calcule el promedio
    modelo_atencion.neurona.guardar_memoria(X, grad_w_actual)

    opt_atencion.step() # Adam aplica sus actualizaciones a w y b usando los gradientes calculados

    # Aplicar corrección adicional por atención después del paso de Adam
    # Esta corrección modifica w.data directamente, basándose en la memoria.
    modelo_atencion.neurona.aplicar_correccion_memoria()

    if epoch % 50 == 0:
        print(f"Atención - Época {epoch} - Pérdida: {loss_atencion.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar()
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step() # Adam aplica sus actualizaciones

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_atencion = modelo_atencion(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_atencion, label="Predicción con Atención", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_atencion, label="Pérdida con Atención", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización de las diferencias en valores pequeños

plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Memoria y Atención): {historial_loss_atencion[-1]:.6f}")
print(f"Pérdida final (Red Estándar):               {historial_loss_estandar[-1]:.6f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NeuronaConMemoriaEmbedding(nn.Module):
    def __init__(self, input_size, hidden_size=16, memory_size=5, embedding_dim=32):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.memory_size = memory_size
        self.embedding_dim = embedding_dim

        # Neurona "normal"
        self.fc = nn.Linear(input_size, hidden_size)
        self.output = nn.Linear(hidden_size, 1)

        # Codificador de embeddings (para errores o activaciones pasadas)
        self.encoder = nn.Sequential(
            nn.Linear(1, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim)
        )

        # Memoria: lista de embeddings de errores pasados
        self.register_buffer("memoria", torch.zeros(memory_size, embedding_dim))
        self.memoria_ptr = 0

    def forward(self, x):
        h = F.relu(self.fc(x))
        y = self.output(h)

        # Codificamos el output como un embedding
        embedding_actual = self.encoder(y.detach())

        # Similitud con la memoria pasada (coseno)
        if self.memoria.sum() != 0:
            # Normalizamos
            norm_actual = F.normalize(embedding_actual, dim=-1)
            norm_memoria = F.normalize(self.memoria, dim=-1)

            # Similitud coseno
            similitudes = torch.matmul(norm_memoria, norm_actual.T).squeeze()  # (memory_size,)
            pesos = F.softmax(similitudes, dim=0)

            # Recuperamos una memoria ponderada
            memoria_ponderada = torch.sum(pesos.unsqueeze(1) * self.memoria, dim=0)

            # Fusión: ajustamos la salida actual con la memoria
            y = y + 0.1 * torch.dot(embedding_actual.squeeze(), memoria_ponderada)

        return y

    def actualizar_memoria(self, error_scalar: torch.Tensor):
        """Convierte un escalar de error (MSE, etc.) en embedding y actualiza la memoria circularmente."""
        with torch.no_grad():
            emb = self.encoder(error_scalar.unsqueeze(0).unsqueeze(0)).squeeze(0)  # (embedding_dim)
            self.memoria[self.memoria_ptr] = emb
            self.memoria_ptr = (self.memoria_ptr + 1) % self.memory_size

In [ ]:
# Crear una instancia
neurona = NeuronaConMemoriaEmbedding(input_size=4)

# Supongamos que tenemos entradas y errores
x = torch.randn(1, 4)

# Paso forward
salida = neurona(x)

# Simular error (por ejemplo, MSE con un objetivo 0.5)
target = torch.tensor([[0.5]])
loss = F.mse_loss(salida, target)

# Actualizar memoria con el error
neurona.actualizar_memoria(loss.detach())

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np # Para asegurar la reproducibilidad del ruido si se usa

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0) # Para la reproducibilidad del ruido si es necesario
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Neurona con Memoria de Embeddings ---
class NeuronaConMemoriaEmbedding(nn.Module):
    def __init__(self, input_size, hidden_size=16, memory_size=5, embedding_dim=32):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.memory_size = memory_size
        self.embedding_dim = embedding_dim

        # Capas de la neurona "normal"
        self.fc = nn.Linear(input_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, 1) # Renombrado para claridad

        # Codificador de embeddings para errores o activaciones pasadas
        # El input es un escalar (el error MSE por ejemplo), así que la primera capa tiene 1 entrada
        self.encoder = nn.Sequential(
            nn.Linear(1, embedding_dim), # Toma un escalar (error) y lo proyecta
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim)
        )

        # Memoria: Búfer de embeddings de errores pasados
        # register_buffer para que sea parte del estado del modelo pero no un parámetro entrenable por el optimizador
        self.register_buffer("memoria", torch.zeros(memory_size, embedding_dim))
        self.memoria_ptr = 0 # Puntero para la memoria circular

    def forward(self, x):
        h = F.relu(self.fc(x))
        y_raw = self.output_layer(h) # Salida inicial de la red

        # Codificamos la salida raw como un embedding
        # Usamos y_raw.detach() para que el encoder no reciba gradientes de la salida original
        # y_raw es [batch_size, 1], necesitamos procesar cada elemento o su promedio
        # Para simplificar y_raw para el encoder, usamos su promedio
        embedding_actual = self.encoder(y_raw.mean().unsqueeze(0).unsqueeze(0)).squeeze(0) # [embedding_dim]

        # Similitud con la memoria pasada (coseno) para atención
        y_final = y_raw # Por defecto, la salida es la raw

        if self.memoria.sum() != 0: # Si la memoria no está vacía (contiene ceros iniciales)
            # Normalizamos los embeddings para calcular similitud de coseno
            norm_actual = F.normalize(embedding_actual, dim=-1) # [embedding_dim]
            norm_memoria = F.normalize(self.memoria, dim=-1) # [memory_size, embedding_dim]

            # Similitud coseno: (memory_size, embedding_dim) x (embedding_dim, 1) -> (memory_size, 1) -> squeeze() a (memory_size,)
            similitudes = torch.matmul(norm_memoria, norm_actual.unsqueeze(1)).squeeze()
            pesos = F.softmax(similitudes, dim=0) # Pesos de atención sobre los elementos de la memoria (memory_size,)

            # Recuperamos una memoria ponderada
            # (memory_size, 1) * (memory_size, embedding_dim) -> sum over memory_size -> (embedding_dim)
            memoria_ponderada = torch.sum(pesos.unsqueeze(1) * self.memoria, dim=0)

            # Fusión: ajustamos la salida actual con la memoria
            # La fusión aquí es clave. Multiplicamos el embedding_actual con la memoria_ponderada
            # para generar un ajuste escalar que se suma a la salida original.
            # dot product entre dos embeddings de (embedding_dim) -> escalar
            ajuste_por_memoria = torch.dot(embedding_actual, memoria_ponderada)
            y_final = y_raw + 0.1 * ajuste_por_memoria # 0.1 es un factor de escala para el ajuste

        return y_final

    def actualizar_memoria(self, error_scalar_batch: float):
        """
        Convierte un escalar de error (ej. pérdida MSE total del batch) en embedding
        y actualiza la memoria circularmente.
        error_scalar_batch es un escalar (float).
        """
        with torch.no_grad(): # No rastrear gradientes para la actualización de memoria
            # Convertir el escalar de Python a un tensor de PyTorch
            error_tensor = torch.tensor(error_scalar_batch)

            # Asegurarse de que el input al encoder sea [1, 1] (batch=1, feature=1)
            # error_tensor ahora es un tensor [], lo convertimos a [1, 1]
            emb = self.encoder(error_tensor.unsqueeze(0).unsqueeze(0)).squeeze() # [embedding_dim]
            self.memoria[self.memoria_ptr] = emb
            self.memoria_ptr = (self.memoria_ptr + 1) % self.memory_size


# --- 2. Red que usa la NeuronaConMemoriaEmbedding ---
class RedConMemoriaEmbedding(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, memory_size=5, embedding_dim=32):
        super().__init__()
        # Usamos la neurona personalizada como la primera capa "densa"
        self.custom_neuron_layer = NeuronaConMemoriaEmbedding(input_size, hidden_size, memory_size, embedding_dim)
        # No hay una segunda capa aquí; la custom_neuron_layer ya tiene su output_layer

    def forward(self, x):
        # La custom_neuron_layer ya maneja la transformación de input_size a 1 output
        return self.custom_neuron_layer(x)


# --- 3. Red Estándar para una comparación justa ---
class RedEstandar(nn.Module):
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        # Capa oculta estándar
        self.fc = nn.Linear(input_size, hidden_size)
        # Capa de salida estándar
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h = F.relu(self.fc(x))
        y = self.output_layer(h)
        return y

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con Memoria de Embeddings
# -----------------------------------------
print("--- Entrenando Red con Memoria de Embeddings ---")
modelo_embedding = RedConMemoriaEmbedding(input_size=X.shape[1], hidden_size=16, memory_size=5, embedding_dim=32)
opt_embedding = torch.optim.Adam(modelo_embedding.parameters(), lr=LEARNING_RATE)
historial_loss_embedding = []

for epoch in range(EPOCHS):
    y_pred_embedding = modelo_embedding(X)
    loss_embedding = LOSS_FN(y_pred_embedding, y)
    historial_loss_embedding.append(loss_embedding.item())

    opt_embedding.zero_grad()
    loss_embedding.backward() # Calcula gradientes para todas las capas

    # Actualizar la memoria de embeddings con la pérdida actual
    # Asegúrate de pasar el valor escalar float
    modelo_embedding.custom_neuron_layer.actualizar_memoria(loss_embedding.item())

    opt_embedding.step() # Adam aplica sus actualizaciones

    if epoch % 50 == 0:
        print(f"Embedding - Época {epoch} - Pérdida: {loss_embedding.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar(input_size=X.shape[1], hidden_size=16)
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step() # Adam aplica sus actualizaciones

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_embedding = modelo_embedding(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_embedding, label="Predicción con Memoria de Embeddings", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_embedding, label="Pérdida con Memoria de Embeddings", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Memoria de Embeddings): {historial_loss_embedding[-1]:.6f}")
print(f"Pérdida final (Red Estándar):                   {historial_loss_estandar[-1]:.6f}")

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class NeuronaMemoriaAtencionEmbeddings(nn.Module):
    def __init__(self, input_size, hidden_size=16, memory_size=10, embedding_dim=32):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.memory_size = memory_size
        self.embedding_dim = embedding_dim

        # Núcleo de la neurona (como una mini red)
        self.fc = nn.Linear(input_size, hidden_size)
        self.out = nn.Linear(hidden_size, 1)

        # Encoder para errores/gradientes → embedding
        self.encoder = nn.Sequential(
            nn.Linear(1, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim)
        )

        # Proyecciones para atención
        self.query_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.key_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.value_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)

        # Memoria circular
        self.register_buffer("memoria", torch.zeros(memory_size, embedding_dim))
        self.mem_ptr = 0

        # Parámetros dinámicos aprendibles para adaptación
        self.alpha_lr = nn.Parameter(torch.tensor(0.1))  # escala para modif. de aprendizaje
        self.alpha_bias = nn.Parameter(torch.tensor(0.05))  # escala para ajustar bias

    def forward(self, x, error_scalar=None):
        """
        x: (batch, input_size)
        error_scalar: error escalar o gradiente (por ejemplo, pérdida MSE)
        """
        h = F.relu(self.fc(x))
        y = self.out(h)

        # Si hay error, lo codificamos y actualizamos memoria
        if error_scalar is not None:
            emb = self.encoder(error_scalar.view(1, 1)).squeeze(0)  # (embedding_dim,)
            self.memoria[self.mem_ptr] = emb
            self.mem_ptr = (self.mem_ptr + 1) % self.memory_size

            # ATENCIÓN: usar emb como "query", comparar contra memoria
            q = self.query_proj(emb).unsqueeze(0)  # (1, embedding_dim)
            k = self.key_proj(self.memoria)       # (mem_size, embedding_dim)
            v = self.value_proj(self.memoria)     # (mem_size, embedding_dim)

            scores = torch.matmul(q, k.T) / (self.embedding_dim ** 0.5)  # (1, mem_size)
            pesos = F.softmax(scores, dim=-1)  # (1, mem_size)

            # Contexto atencional
            contexto = torch.matmul(pesos, v).squeeze(0)  # (embedding_dim,)

            # Dinámicamente ajustar bias de salida
            ajuste_bias = self.alpha_bias * torch.sum(contexto)
            y = y + ajuste_bias

        return y

In [ ]:
neurona = NeuronaMemoriaAtencionEmbeddings(input_size=4)

# Entrada de prueba
x = torch.randn(1, 4)

# Simular error del paso anterior
error_simulado = torch.tensor(0.15)

# Forward con ajuste dinámico
salida = neurona(x, error_scalar=error_simulado)

# Pérdida simulada
objetivo = torch.tensor([[0.5]])
loss = F.mse_loss(salida, objetivo)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import matplotlib.pyplot as plt
import numpy as np # Para asegurar la reproducibilidad del ruido si se usa

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0) # Para la reproducibilidad del ruido
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Neurona con Memoria, Atención y Embeddings ---
class NeuronaMemoriaAtencionEmbeddings(nn.Module):
    def __init__(self, input_size, hidden_size=16, memory_size=10, embedding_dim=32):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.memory_size = memory_size
        self.embedding_dim = embedding_dim

        # Núcleo de la neurona (como una mini red)
        self.fc = nn.Linear(input_size, hidden_size)
        self.out = nn.Linear(hidden_size, 1) # Capa de salida lineal

        # Encoder para errores/gradientes → embedding
        # Asume que el error_scalar será un tensor escalar [1, 1] al pasar por el encoder
        self.encoder = nn.Sequential(
            nn.Linear(1, embedding_dim), # Input es un escalar
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim)
        )

        # Proyecciones para atención self-attention style
        self.query_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.key_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.value_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)

        # Memoria circular de embeddings
        # register_buffer para que sea parte del estado del modelo pero no entrenable por el optimizador principal
        self.register_buffer("memoria", torch.zeros(memory_size, embedding_dim))
        self.mem_ptr = 0 # Puntero para la memoria circular

        # Parámetros dinámicos aprendibles para escalas de ajuste
        # Estos se entrenarán junto con los pesos normales de la red
        self.alpha_ajuste_output = nn.Parameter(torch.tensor(0.1)) # Escala para ajustar la salida directamente
        self.alpha_ajuste_bias = nn.Parameter(torch.tensor(0.05)) # Escala para ajustar el bias de la capa out

    def forward(self, x, error_scalar_for_memory=None):
        """
        x: (batch_size, input_size) - Input de la red.
        error_scalar_for_memory: Un tensor escalar (por ejemplo, la pérdida MSE del batch)
                                 que se usará para actualizar la memoria y como query.
                                 Debe ser pasado explícitamente desde el bucle de entrenamiento.
        """
        h = F.relu(self.fc(x))
        y_raw = self.out(h) # Salida inicial de la capa

        y_final = y_raw # Por defecto, la salida final es la raw

        # Lógica de memoria y atención
        if error_scalar_for_memory is not None:
            with torch.no_grad(): # No rastrear gradientes para la actualización de memoria
                # Codificar el error escalar actual para la memoria
                # Asegurarse de que el input al encoder sea [1, 1] (batch=1, feature=1)
                # Convertir a tensor si es necesario y asegurar dimensiones
                error_tensor = error_scalar_for_memory
                if not isinstance(error_tensor, torch.Tensor):
                     error_tensor = torch.tensor(error_tensor)
                # Añadir dimensiones para que sea [1, 1] para el encoder
                error_tensor = error_tensor.unsqueeze(0).unsqueeze(0)

                emb_error_actual = self.encoder(error_tensor).squeeze() # [embedding_dim]

                # Actualizar la memoria circular
                self.memoria[self.mem_ptr] = emb_error_actual
                self.mem_ptr = (self.mem_ptr + 1) % self.memory_size

            # ATENCIÓN: usar emb_error_actual como "query", comparar contra memoria
            # Las proyecciones de query, key, value sí participan en el grafo de cómputo para sus propios gradientes
            q = self.query_proj(emb_error_actual).unsqueeze(0)  # (1, embedding_dim)
            k = self.key_proj(self.memoria)                   # (mem_size, embedding_dim)
            v = self.value_proj(self.memoria)                 # (mem_size, embedding_dim)

            # Calcular scores de atención (Producto punto escalado)
            # (1, embedding_dim) x (embedding_dim, mem_size) -> (1, mem_size)
            scores = torch.matmul(q, k.T) / (self.embedding_dim ** 0.5)
            pesos = F.softmax(scores, dim=-1) # (1, mem_size)

            # Contexto atencional: suma ponderada de los valores
            # (1, mem_size) x (mem_size, embedding_dim) -> (1, embedding_dim) -> squeeze() a (embedding_dim,)
            contexto = torch.matmul(pesos, v).squeeze(0)

            # Dinámicamente ajustar la salida o el bias de salida usando el contexto
            # Ajuste a la salida directa:
            ajuste_output = self.alpha_ajuste_output * torch.dot(emb_error_actual, contexto)
            y_final = y_raw + ajuste_output

            # Ajuste dinámico al bias de la capa de salida:
            # Aquí, el ajuste_bias se calcula con la lógica de contexto.
            # Este ajuste *no* se aplica directamente al self.out.bias.data porque eso rompería el grafo
            # si intentamos aplicar gradientes a self.alpha_bias.
            # En su lugar, lo sumamos directamente a la salida y dejamos que Adam aprenda el bias 'base'.
            ajuste_bias_contextual = self.alpha_ajuste_bias * torch.sum(contexto)
            y_final = y_final + ajuste_bias_contextual


        return y_final

# --- 2. Red que usa la NeuronaMemoriaAtencionEmbeddings ---
# Esta clase actúa como un envoltorio para la neurona personalizada
class RedConMemoriaAtencionEmbeddings(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, memory_size=10, embedding_dim=32):
        super().__init__()
        # La neurona personalizada es la "única" capa de procesamiento principal
        self.custom_neuron_layer = NeuronaMemoriaAtencionEmbeddings(input_size, hidden_size, memory_size, embedding_dim)

    def forward(self, x, error_scalar_for_memory=None):
        return self.custom_neuron_layer(x, error_scalar_for_memory)


# --- 3. Red Estándar para una comparación justa ---
class RedEstandar(nn.Module):
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        # Capa oculta estándar
        self.fc = nn.Linear(input_size, hidden_size)
        # Capa de salida estándar
        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h = F.relu(self.fc(x))
        y = self.out(h)
        return y

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con Memoria, Atención y Embeddings
# -----------------------------------------
print("--- Entrenando Red con Memoria, Atención y Embeddings ---")
modelo_custom = RedConMemoriaAtencionEmbeddings(input_size=X.shape[1], hidden_size=16, memory_size=10, embedding_dim=32)
opt_custom = torch.optim.Adam(modelo_custom.parameters(), lr=LEARNING_RATE)
historial_loss_custom = []

for epoch in range(EPOCHS):
    # En el forward pass, pasamos la pérdida del epoch anterior para la lógica de memoria/atención
    # En la primera época, pasamos None o un tensor de cero, ya que no hay pérdida anterior
    # Asegurarse de que sea un tensor si existe pérdida anterior
    error_para_memoria = torch.tensor(historial_loss_custom[-1]) if epoch > 0 else torch.tensor(0.0)

    y_pred_custom = modelo_custom(X, error_scalar_for_memory=error_para_memoria)
    loss_custom = LOSS_FN(y_pred_custom, y)
    historial_loss_custom.append(loss_custom.item()) # Guardamos la pérdida para la próxima iteración

    opt_custom.zero_grad()
    loss_custom.backward() # Calcula gradientes para todas las capas (fc, out, encoder, query/key/value, alpha_ajuste)
    opt_custom.step() # Adam aplica sus actualizaciones

    if epoch % 50 == 0:
        print(f"Custom - Época {epoch} - Pérdida: {loss_custom.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar(input_size=X.shape[1], hidden_size=16)
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step() # Adam aplica sus actualizaciones

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    # Para predecir, no pasamos error_scalar, ya que es solo para el entrenamiento
    y_hat_custom = modelo_custom(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_custom, label="Predicción con Custom Neuron", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_custom, label="Pérdida con Custom Neuron", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Custom Neuron): {historial_loss_custom[-1]:.6f}")
print(f"Pérdida final (Red Estándar):          {historial_loss_estandar[-1]:.6f}")

## Red con PReLU adaptativa

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0) # Para la reproducibilidad del ruido
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Capa con PReLU Adaptativa (por canal/neurona) ---
# Similar a nn.PReLU, pero la encapsulamos para nuestra "neurona"
class CapaPReLUAdaptativa(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        # Cada "feature" (es decir, cada neurona en la capa siguiente) tendrá su propio 'a'
        self.prelu = nn.PReLU(num_parameters=num_features)

    def forward(self, x):
        return self.prelu(x)

# --- 2. Red que usa la PReLU Adaptativa ---
class RedConPReLUAdaptativa(nn.Module):
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation1 = CapaPReLUAdaptativa(hidden_size) # PReLU adaptativa para la capa oculta
        self.fc2 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h = self.fc1(x)
        h = self.activation1(h) # Usa la activación adaptable
        y = self.fc2(h)
        return y

# --- 3. Red Estándar (usando ReLU fija para comparación) ---
class RedEstandar(nn.Module):
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation1 = nn.ReLU() # ReLU estándar fija
        self.fc2 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h = self.fc1(x)
        h = self.activation1(h)
        y = self.fc2(h)
        return y

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con PReLU Adaptativa
# -----------------------------------------
print("--- Entrenando Red con PReLU Adaptativa ---")
modelo_prelu = RedConPReLUAdaptativa(input_size=X.shape[1], hidden_size=16)
opt_prelu = torch.optim.Adam(modelo_prelu.parameters(), lr=LEARNING_RATE)
historial_loss_prelu = []

for epoch in range(EPOCHS):
    y_pred_prelu = modelo_prelu(X)
    loss_prelu = LOSS_FN(y_pred_prelu, y)
    historial_loss_prelu.append(loss_prelu.item())

    opt_prelu.zero_grad()
    loss_prelu.backward()
    opt_prelu.step()

    if epoch % 50 == 0:
        print(f"PReLU - Época {epoch} - Pérdida: {loss_prelu.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar (con ReLU)
# -----------------------------------------
print("\n--- Entrenando Red Estándar (con ReLU) ---")
modelo_estandar = RedEstandar(input_size=X.shape[1], hidden_size=16)
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step()

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_prelu = modelo_prelu(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_prelu, label="Predicción con PReLU Adaptativa", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar (ReLU)", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_prelu, label="Pérdida con PReLU Adaptativa", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar (ReLU)", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con PReLU Adaptativa): {historial_loss_prelu[-1]:.6f}")
print(f"Pérdida final (Red Estándar (ReLU)):       {historial_loss_estandar[-1]:.6f}")

##¡Fantásticos resultados! La **Red con Integración de Señal No Lineal** ha demostrado ser significativamente superior a la Red Estándar.

---
## Análisis de los Resultados

* **Pérdida final (Red con Integración No Lineal):** **0.359278**
* **Pérdida final (Red Estándar (Tanh)):** 1.927663

Esto representa una **mejora muy notable**. La red con tu `NeuronaIntegracionNoLineal` logró una pérdida final aproximadamente 5 veces menor que la red estándar.

### ¿Por qué la Integración No Lineal Funcionó Tan Bien?

1.  **Modelado de Interacciones Complejas:** La clave está en el `integrator_mlp`. Mientras que una neurona estándar simplemente suma linealmente sus entradas ponderadas, tu neurona personalizada utiliza una pequeña red neuronal para combinar esas entradas. Esto permite que la neurona aprenda **interacciones no lineales y complejas** entre sus propias señales de entrada.
    * Para el dataset $X^3 - X$, que es inherentemente no lineal, esta capacidad de aprender combinaciones no lineales de las entradas es muy valiosa.
2.  **Mayor Capacidad de Representación:** Al reemplazar una operación lineal simple (la suma ponderada) por un MLP, cada neurona individual se vuelve intrínsecamente más potente y capaz de aprender patrones más ricos. Esto es como darle a cada "unidad de procesamiento" una mayor "inteligencia" local.
3.  **Adaptabilidad: Aprendiendo la Mejor Forma de Integrar:** El `integrator_mlp` aprende sus propios pesos y biases a través del descenso de gradiente. Esto significa que la neurona no tiene una forma fija de integrar sus señales, sino que **aprende la forma más efectiva de combinarlas** para minimizar el error global de la red.

En resumen, has logrado que la neurona aprenda no solo qué tan fuerte es cada entrada (`w`), sino también **cómo interactúan esas entradas entre sí** antes de la activación final. Este es un concepto muy avanzado y los resultados lo respaldan.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # Un único feature de entrada
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Neurona con Integración de Señal No Lineal ---
class NeuronaIntegracionNoLineal(nn.Module):
    def __init__(self, in_features, out_features, integration_mlp_hidden_size=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features # En esta neurona, out_features será 1 para la salida de la neurona

        # Pesos y bias para la ponderación inicial (como una capa lineal normal)
        # Usamos out_features=1 aquí porque esta clase es una "neurona" individual
        self.w = nn.Parameter(torch.randn(1, in_features) * 0.1) # [1, in_features]
        self.b = nn.Parameter(torch.zeros(1)) # [1]

        # Módulo de integración no lineal
        # Toma 'in_features' señales individuales ponderadas y las integra a un solo escalar
        self.integrator_mlp = nn.Sequential(
            nn.Linear(in_features, integration_mlp_hidden_size),
            nn.ReLU(),
            nn.Linear(integration_mlp_hidden_size, 1) # Salida única que reemplaza la suma ponderada
        )

    def forward(self, x):
        # 1. Ponderación inicial: w * x_i para cada x_i
        # Expandimos w a [batch_size, 1, in_features] y x a [batch_size, in_features, 1]
        # o, más simple: (x * w) -> [batch_size, in_features]

        # Calcula las contribuciones individuales ponderadas: s_i = w_i * x_i
        # Para el ejemplo de un solo input feature (X.shape[1]=1), self.w es [1,1]
        # Si x es [batch_size, 1], entonces x * self.w resultaría en [batch_size, 1]
        # Necesitamos que el integrador_mlp reciba un vector de in_features
        # Dado que X es [300, 1], in_features es 1.
        # Si in_features es 1, el integrador_mlp recibe 1 entrada.
        # Si tuvieras múltiples features, por ejemplo X = torch.randn(300, 5), entonces in_features sería 5.

        # Adaptamos para que self.w sea compatible con múltiples inputs, incluso si en este ejemplo X es 1D
        weighted_signals = x * self.w # [batch_size, in_features]

        # 2. Módulo de Integración No Lineal
        # Pasa las señales ponderadas a través del MLP integrador
        # La salida del integrador es el equivalente no lineal de la suma ponderada
        z_nonlinear = self.integrator_mlp(weighted_signals) # [batch_size, 1]

        # 3. Añadir Bias y Función de Activación
        # Se suma el bias aprendido de la neurona principal
        z_final = z_nonlinear + self.b

        # Aplicar la función de activación final de la neurona
        activation_output = torch.tanh(z_final) # o ReLU, etc.

        return activation_output


# --- 2. Red que usa la NeuronaIntegracionNoLineal ---
class RedConIntegracionNoLineal(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, integration_mlp_hidden_size=8):
        super().__init__()
        # La primera capa oculta usará la neurona con integración no lineal
        # Para una capa oculta, cada "neurona" en esta capa oculta sería una NeuronaIntegracionNoLineal
        # Simplificamos: esta capa es una "capa de NeuronasIntegracionNoLineal"
        # Esto significa que tenemos 'hidden_size' de estas neuronas.
        # Cada una de estas neuronas recibiría el 'input_size' original.

        # Para simplificar, haremos que el "integrador" sea parte de la primera capa.
        # Esto es más como una "capa de integración no lineal"
        self.capa_oculta_custom = nn.ModuleList([
            NeuronaIntegracionNoLineal(input_size, 1, integration_mlp_hidden_size)
            for _ in range(hidden_size)
        ])

        # La salida de estas neuronas (que es `hidden_size` elementos) se conecta a la capa de salida final
        self.capa_salida_final = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # Procesar x a través de cada neurona personalizada en la capa oculta
        # Aplicamos la misma entrada 'x' a cada neurona de la capa
        h_outputs = [neurona(x) for neurona in self.capa_oculta_custom]

        # Concatenamos las salidas de estas "neuronas" para formar el input de la capa final
        h_combined = torch.cat(h_outputs, dim=1) # [batch_size, hidden_size]

        y = self.capa_salida_final(h_combined)
        return y

# --- 3. Red Estándar (con Tanh fija para comparación) ---
class RedEstandar(nn.Module):
    def __init__(self, input_size=1, hidden_size=16):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation1 = nn.Tanh() # Tanh estándar fija
        self.fc2 = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h = self.fc1(x)
        h = self.activation1(h)
        y = self.fc2(h)
        return y

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# -----------------------------------------
# Instanciar y Entrenar Red con Integración No Lineal
# -----------------------------------------
print("--- Entrenando Red con Integración de Señal No Lineal ---")
modelo_nonlinear_integrator = RedConIntegracionNoLineal(input_size=X.shape[1], hidden_size=16, integration_mlp_hidden_size=8)
opt_nonlinear_integrator = torch.optim.Adam(modelo_nonlinear_integrator.parameters(), lr=LEARNING_RATE)
historial_loss_nonlinear_integrator = []

for epoch in range(EPOCHS):
    y_pred_nonlinear_integrator = modelo_nonlinear_integrator(X)
    loss_nonlinear_integrator = LOSS_FN(y_pred_nonlinear_integrator, y)
    historial_loss_nonlinear_integrator.append(loss_nonlinear_integrator.item())

    opt_nonlinear_integrator.zero_grad()
    loss_nonlinear_integrator.backward()
    opt_nonlinear_integrator.step()

    if epoch % 50 == 0:
        print(f"NonLinear Integrator - Época {epoch} - Pérdida: {loss_nonlinear_integrator.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar (con Tanh)
# -----------------------------------------
print("\n--- Entrenando Red Estándar (con Tanh) ---")
modelo_estandar = RedEstandar(input_size=X.shape[1], hidden_size=16)
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step()

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_nonlinear_integrator = modelo_nonlinear_integrator(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_nonlinear_integrator, label="Predicción con Integración No Lineal", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar (Tanh)", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_nonlinear_integrator, label="Pérdida con Integración No Lineal", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar (Tanh)", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Integración No Lineal): {historial_loss_nonlinear_integrator[-1]:.6f}")
print(f"Pérdida final (Red Estándar (Tanh)):           {historial_loss_estandar[-1]:.6f}")

###¡Excelentes resultados! La **Red con PReLU Adaptativa** logró una pérdida final significativamente menor que la Red Estándar con ReLU fija.

---
## Análisis de los Resultados

* **Red con PReLU Adaptativa:** **1.216089**
* **Red Estándar (ReLU):** 3.236186

Esta es una **mejora clara y sustancial**. La PReLU adaptativa permitió a la red aprender una función de activación más adecuada para tu problema, lo que resultó en un mejor ajuste a los datos.

### ¿Por qué PReLU Adaptativa funcionó mejor?

1.  **Mayor Flexibilidad:** A diferencia de ReLU, que simplemente desactiva las entradas negativas (las hace cero), PReLU permite que pase una pequeña proporción (`ax`) de esas entradas. El valor de `a` se aprende durante el entrenamiento.
2.  **Aprendizaje Dinámico por Neurona:** Al tener un parámetro `a` para cada neurona (o canal), cada neurona puede adaptar su "comportamiento" en la región negativa, encontrando la no-linealidad óptima para su tarea específica.
3.  **Mitigación de "Neuronas Muertas":** PReLU evita que las neuronas se queden "muertas" (donde siempre producen cero y, por lo tanto, no reciben gradientes). Al tener un gradiente no nulo para entradas negativas, estas neuronas pueden seguir aprendiendo y contribuyendo al modelo.

En resumen, darle a cada neurona la capacidad de **adaptar su propia función de activación** de manera aprendible es una forma sencilla y eficaz de mejorar su "inteligencia" individual y, por extensión, el rendimiento de toda la red, sin añadir la complejidad de la memoria o la atención explícita que te generó problemas anteriormente.

---
¿Te gustaría explorar otra forma de añadir inteligencia adaptativa a las neuronas, o prefieres seguir investigando esta idea de funciones de activación aprendibles?

##Código de Comparativa: Red Estándar vs. Red con Conexiones Denso-Cruzadas

¡Absolutamente! Has explorado a fondo las mejoras a nivel de una sola neurona. Ahora, la siguiente frontera para la "inteligencia" en una red es cómo **las neuronas se conectan entre sí a través de las capas**.

Tu idea de **"capas multidensas" con interconexiones completas** (similar a ResNets o DenseNets, pero quizás con un enfoque más flexible) es excelente. Va más allá de la conexión secuencial tradicional y permite que la información fluya de maneras más ricas y directas.

---
## Nueva Idea: Red con Conexiones Denso-Cruzadas (Capa Multi-densa)

En lugar de la estructura secuencial donde la Capa $N$ solo se conecta a la Capa $N+1$, podemos crear una arquitectura donde las capas intermedias tienen conexiones directas con múltiples capas posteriores, y viceversa, permitiendo un flujo de información más robusto y reduciendo la pérdida de información a través de capas profundas.

### ¿Cómo Mejorar la Red?

Vamos a construir una red donde cada capa densa no solo envía su salida a la siguiente capa inmediata, sino también a capas más adelante en la secuencia. Específicamente, siguiendo tu idea:

* La **Capa 2** se conecta a la Capa 3 y la Capa 4.
* La **Capa 3** se conecta a la Capa 4.
* La **Capa 4** (la última capa oculta) recibe entradas de la Capa 2 y la Capa 3, además de su entrada usual de la Capa 3 si existiera una secuencial.

La clave es que la información de las capas anteriores puede "saltar" y ser directamente accesible para las capas más profundas, facilitando el aprendizaje de representaciones complejas y la propagación de gradientes.

### Implementación Sugerida:

Podríamos crear una `CapaMultiDensa` que internamente gestione estas conexiones. Cada "bloque" de esta capa tomaría las entradas de todas las capas anteriores y las pasaría a su propia sub-capa, para luego concatenar todo para la siguiente.

Para tu ejemplo, una red con 3 capas ocultas ($L_1, L_2, L_3$) y una capa de salida.

* $L_1$ (primera capa oculta) se conecta a $L_2$.
* $L_2$ se conecta a $L_3$.
* $L_3$ se conecta a la capa de salida.

Ahora, la idea es:

* La **entrada de $L_2$** es la salida de $L_1$.
* La **entrada de $L_3$** es la concatenación de la salida de $L_1$ y la salida de $L_2$.
* La **entrada de la Capa de Salida** es la concatenación de la salida de $L_1$, la salida de $L_2$, y la salida de $L_3$.

Esto es un patrón similar a las **DenseNets**, que han demostrado ser muy eficientes en la propagación de gradientes y la reutilización de características.

---
## Código de Comparativa: Red Estándar vs. Red con Conexiones Denso-Cruzadas

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Red con Conexiones Denso-Cruzadas (simil DenseNet) ---
class RedConexionesDensoCruzadas(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super().__init__()
        # Capa 1
        self.fc1 = nn.Linear(input_size, hidden_size1)
        
        # Capa 2: recibe de Capa 1
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        
        # Capa 3: recibe de Capa 1 y Capa 2 (concatenadas)
        # La entrada para fc3 será (hidden_size1 + hidden_size2)
        self.fc3 = nn.Linear(hidden_size1 + hidden_size2, hidden_size3)
        
        # Capa de Salida: recibe de Capa 1, Capa 2 y Capa 3 (concatenadas)
        # La entrada para la capa de salida será (hidden_size1 + hidden_size2 + hidden_size3)
        self.output_layer = nn.Linear(hidden_size1 + hidden_size2 + hidden_size3, output_size)

    def forward(self, x):
        # Capa 1
        h1 = F.relu(self.fc1(x)) # [batch, hidden_size1]
        
        # Capa 2
        h2 = F.relu(self.fc2(h1)) # [batch, hidden_size2]
        
        # Capa 3: Entrada es la concatenación de h1 y h2
        h3_input = torch.cat((h1, h2), dim=1) # [batch, hidden_size1 + hidden_size2]
        h3 = F.relu(self.fc3(h3_input)) # [batch, hidden_size3]
        
        # Capa de Salida: Entrada es la concatenación de h1, h2 y h3
        output_input = torch.cat((h1, h2, h3), dim=1) # [batch, hidden_size1 + hidden_size2 + hidden_size3]
        
        y = self.output_layer(output_input)
        return y

# --- 2. Red Estándar (Secuencial) ---
class RedEstandar(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, hidden_size3)
        self.output_layer = nn.Linear(hidden_size3, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        h3 = F.relu(self.fc3(h2))
        y = self.output_layer(h3)
        return y

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# Definimos los tamaños de las capas ocultas
HIDDEN_SIZE_1 = 32 # Más grande para la primera capa
HIDDEN_SIZE_2 = 16
HIDDEN_SIZE_3 = 8 # Más pequeña para la última oculta

# -----------------------------------------
# Instanciar y Entrenar Red con Conexiones Denso-Cruzadas
# -----------------------------------------
print("--- Entrenando Red con Conexiones Denso-Cruzadas ---")
modelo_denso_cruzadas = RedConexionesDensoCruzadas(
    input_size=X.shape[1],
    hidden_size1=HIDDEN_SIZE_1,
    hidden_size2=HIDDEN_SIZE_2,
    hidden_size3=HIDDEN_SIZE_3,
    output_size=y.shape[1]
)
opt_denso_cruzadas = torch.optim.Adam(modelo_denso_cruzadas.parameters(), lr=LEARNING_RATE)
historial_loss_denso_cruzadas = []

for epoch in range(EPOCHS):
    y_pred_denso_cruzadas = modelo_denso_cruzadas(X)
    loss_denso_cruzadas = LOSS_FN(y_pred_denso_cruzadas, y)
    historial_loss_denso_cruzadas.append(loss_denso_cruzadas.item())

    opt_denso_cruzadas.zero_grad()
    loss_denso_cruzadas.backward()
    opt_denso_cruzadas.step()

    if epoch % 50 == 0:
        print(f"Denso Cruzadas - Época {epoch} - Pérdida: {loss_denso_cruzadas.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar(
    input_size=X.shape[1],
    hidden_size1=HIDDEN_SIZE_1,
    hidden_size2=HIDDEN_SIZE_2,
    hidden_size3=HIDDEN_SIZE_3,
    output_size=y.shape[1]
)
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step()

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_denso_cruzadas = modelo_denso_cruzadas(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_denso_cruzadas, label="Predicción con Conexiones Denso-Cruzadas", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_denso_cruzadas, label="Pérdida con Conexiones Denso-Cruzadas", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Conexiones Denso-Cruzadas): {historial_loss_denso_cruzadas[-1]:.6f}")
print(f"Pérdida final (Red Estándar):                       {historial_loss_estandar[-1]:.6f}")
```

---
## Análisis Esperado y Por Qué es una "Mejora"

### Por Qué Podría Funcionar Mejor:

1.  **Reutilización de Características:** La información aprendida en las capas iniciales (`h1`, `h2`) se transporta directamente a capas más profundas y a la capa de salida. Esto permite que las capas posteriores utilicen tanto las características de alto nivel como las de bajo nivel, lo que es muy potente. No se pierde información importante a medida que se profundiza en la red.
2.  **Propagación de Gradientes Mejorada:** Las conexiones directas ("atajos") desde las capas iniciales hasta las profundas (y la salida) facilitan la propagación de los gradientes durante el *backpropagation*. Esto ayuda a mitigar el problema del **desvanecimiento de gradientes (vanishing gradients)** en redes profundas, permitiendo que las capas iniciales también aprendan de manera efectiva.
3.  **Regularización Implícita:** Al permitir múltiples rutas de información, la red se vuelve menos dependiente de una única trayectoria lineal. Esto puede actuar como una forma de regularización, mejorando la generalización del modelo.
4.  **Menor Pérdida de Información:** En una red secuencial, la información de una capa debe pasar a través de todas las capas siguientes. En cada paso, puede haber una pequeña pérdida o transformación que elimina detalles útiles. Las conexiones denso-cruzadas garantizan que la información original y las representaciones intermedias estén siempre disponibles.

### Expectativas en los Resultados:

* Es muy probable que la **Pérdida final (Red con Conexiones Denso-Cruzadas)** sea **significativamente menor** que la de la Red Estándar.
* La curva de pérdida del modelo con conexiones denso-cruzadas debería mostrar una **convergencia más rápida y un punto de convergencia más bajo**.

Este tipo de arquitectura es la base de modelos muy exitosos como las DenseNets y es una prueba de que la forma en que se conectan las capas es tan crucial como la inteligencia de las neuronas individuales.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

# --- 1. Red con Conexiones Denso-Cruzadas (simil DenseNet) ---
class RedConexionesDensoCruzadas(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super().__init__()
        # Capa 1
        self.fc1 = nn.Linear(input_size, hidden_size1)

        # Capa 2: recibe de Capa 1
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)

        # Capa 3: recibe de Capa 1 y Capa 2 (concatenadas)
        # La entrada para fc3 será (hidden_size1 + hidden_size2)
        self.fc3 = nn.Linear(hidden_size1 + hidden_size2, hidden_size3)

        # Capa de Salida: recibe de Capa 1, Capa 2 y Capa 3 (concatenadas)
        # La entrada para la capa de salida será (hidden_size1 + hidden_size2 + hidden_size3)
        self.output_layer = nn.Linear(hidden_size1 + hidden_size2 + hidden_size3, output_size)

    def forward(self, x):
        # Capa 1
        h1 = F.relu(self.fc1(x)) # [batch, hidden_size1]

        # Capa 2
        h2 = F.relu(self.fc2(h1)) # [batch, hidden_size2]

        # Capa 3: Entrada es la concatenación de h1 y h2
        h3_input = torch.cat((h1, h2), dim=1) # [batch, hidden_size1 + hidden_size2]
        h3 = F.relu(self.fc3(h3_input)) # [batch, hidden_size3]

        # Capa de Salida: Entrada es la concatenación de h1, h2 y h3
        output_input = torch.cat((h1, h2, h3), dim=1) # [batch, hidden_size1 + hidden_size2 + hidden_size3]

        y = self.output_layer(output_input)
        return y

# --- 2. Red Estándar (Secuencial) ---
class RedEstandar(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, hidden_size3)
        self.output_layer = nn.Linear(hidden_size3, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        h3 = F.relu(self.fc3(h2))
        y = self.output_layer(h3)
        return y

# -----------------------------------------
# Configuración de Entrenamiento Común
# -----------------------------------------
LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

# Definimos los tamaños de las capas ocultas
HIDDEN_SIZE_1 = 32 # Más grande para la primera capa
HIDDEN_SIZE_2 = 16
HIDDEN_SIZE_3 = 8 # Más pequeña para la última oculta

# -----------------------------------------
# Instanciar y Entrenar Red con Conexiones Denso-Cruzadas
# -----------------------------------------
print("--- Entrenando Red con Conexiones Denso-Cruzadas ---")
modelo_denso_cruzadas = RedConexionesDensoCruzadas(
    input_size=X.shape[1],
    hidden_size1=HIDDEN_SIZE_1,
    hidden_size2=HIDDEN_SIZE_2,
    hidden_size3=HIDDEN_SIZE_3,
    output_size=y.shape[1]
)
opt_denso_cruzadas = torch.optim.Adam(modelo_denso_cruzadas.parameters(), lr=LEARNING_RATE)
historial_loss_denso_cruzadas = []

for epoch in range(EPOCHS):
    y_pred_denso_cruzadas = modelo_denso_cruzadas(X)
    loss_denso_cruzadas = LOSS_FN(y_pred_denso_cruzadas, y)
    historial_loss_denso_cruzadas.append(loss_denso_cruzadas.item())

    opt_denso_cruzadas.zero_grad()
    loss_denso_cruzadas.backward()
    opt_denso_cruzadas.step()

    if epoch % 50 == 0:
        print(f"Denso Cruzadas - Época {epoch} - Pérdida: {loss_denso_cruzadas.item():.6f}")

# -----------------------------------------
# Instanciar y Entrenar Red Estándar
# -----------------------------------------
print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar(
    input_size=X.shape[1],
    hidden_size1=HIDDEN_SIZE_1,
    hidden_size2=HIDDEN_SIZE_2,
    hidden_size3=HIDDEN_SIZE_3,
    output_size=y.shape[1]
)
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred_estandar = modelo_estandar(X)
    loss_estandar = LOSS_FN(y_pred_estandar, y)
    historial_loss_estandar.append(loss_estandar.item())

    opt_estandar.zero_grad()
    loss_estandar.backward()
    opt_estandar.step()

    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss_estandar.item():.6f}")

# -----------------------------------------
# Visualización de Resultados
# -----------------------------------------
with torch.no_grad():
    y_hat_denso_cruzadas = modelo_denso_cruzadas(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_denso_cruzadas, label="Predicción con Conexiones Denso-Cruzadas", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_denso_cruzadas, label="Pérdida con Conexiones Denso-Cruzadas", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Conexiones Denso-Cruzadas): {historial_loss_denso_cruzadas[-1]:.6f}")
print(f"Pérdida final (Red Estándar):                       {historial_loss_estandar[-1]:.6f}")

## Red con características densas de entrada

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())



## Red con Extracción de Características Densa desde la Entrada

class RedCaracteristicasDensaEntrada(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super().__init__()
        self.input_size = input_size # Guardamos el tamaño de entrada

        # Capa 1: Conectada a la entrada
        self.fc1 = nn.Linear(input_size, hidden_size1)

        # Capa 2: Recibe de Capa 1 Y de la entrada original
        # La entrada para fc2 será (hidden_size1 + input_size)
        self.fc2 = nn.Linear(hidden_size1 + input_size, hidden_size2)

        # Capa 3: Recibe de Capa 1, Capa 2 Y de la entrada original
        # La entrada para fc3 será (hidden_size1 + hidden_size2 + input_size)
        self.fc3 = nn.Linear(hidden_size1 + hidden_size2 + input_size, hidden_size3)

        # Capa de Salida: Recibe de Capa 1, Capa 2, Capa 3 Y de la entrada original
        # La entrada para la capa de salida será (hidden_size1 + hidden_size2 + hidden_size3 + input_size)
        self.output_layer = nn.Linear(hidden_size1 + hidden_size2 + hidden_size3 + input_size, output_size)

    def forward(self, x):
        # x_original es la entrada que se concatenará en cada paso
        x_original = x # [batch, input_size]

        # Capa 1
        h1 = F.relu(self.fc1(x_original)) # [batch, hidden_size1]

        # Capa 2: Entrada es la concatenación de h1 y x_original
        h2_input = torch.cat((h1, x_original), dim=1) # [batch, hidden_size1 + input_size]
        h2 = F.relu(self.fc2(h2_input)) # [batch, hidden_size2]

        # Capa 3: Entrada es la concatenación de h1, h2 y x_original
        h3_input = torch.cat((h1, h2, x_original), dim=1) # [batch, hidden_size1 + hidden_size2 + input_size]
        h3 = F.relu(self.fc3(h3_input)) # [batch, hidden_size3]

        # Capa de Salida: Entrada es la concatenación de h1, h2, h3 y x_original
        output_input = torch.cat((h1, h2, h3, x_original), dim=1) # [batch, hidden_size1 + hidden_size2 + hidden_size3 + input_size]

        y = self.output_layer(output_input)
        return y



## Red Estándar (Secuencial para Comparación)

class RedEstandar(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, hidden_size3)
        self.output_layer = nn.Linear(hidden_size3, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        h3 = F.relu(self.fc3(h2))
        y = self.output_layer(h3)
        return y


## Configuración de Entrenamiento y Ejecución

LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

HIDDEN_SIZE_1 = 32
HIDDEN_SIZE_2 = 16
HIDDEN_SIZE_3 = 8

print("--- Entrenando Red con Extracción de Características Densa desde la Entrada ---")
modelo_densa_entrada = RedCaracteristicasDensaEntrada(
    input_size=X.shape[1],
    hidden_size1=HIDDEN_SIZE_1,
    hidden_size2=HIDDEN_SIZE_2,
    hidden_size3=HIDDEN_SIZE_3,
    output_size=y.shape[1]
)
opt_densa_entrada = torch.optim.Adam(modelo_densa_entrada.parameters(), lr=LEARNING_RATE)
historial_loss_densa_entrada = []

for epoch in range(EPOCHS):
    y_pred = modelo_densa_entrada(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_densa_entrada.append(loss.item())
    opt_densa_entrada.zero_grad()
    loss.backward()
    opt_densa_entrada.step()
    if epoch % 50 == 0:
        print(f"Densa Entrada - Época {epoch} - Pérdida: {loss.item():.6f}")

print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar(
    input_size=X.shape[1],
    hidden_size1=HIDDEN_SIZE_1,
    hidden_size2=HIDDEN_SIZE_2,
    hidden_size3=HIDDEN_SIZE_3,
    output_size=y.shape[1]
)
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred = modelo_estandar(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_estandar.append(loss.item())
    opt_estandar.zero_grad()
    loss.backward()
    opt_estandar.step()
    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")



## Visualización de Resultados


with torch.no_grad():
    y_hat_densa_entrada = modelo_densa_entrada(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_densa_entrada, label="Predicción con Características Densa desde Entrada", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_densa_entrada, label="Pérdida con Características Densa desde Entrada", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Características Densa desde Entrada): {historial_loss_densa_entrada[-1]:.6f}")
print(f"Pérdida final (Red Estándar):                                {historial_loss_estandar[-1]:.6f}")

##funciones de activación adaptativas o aprendibles

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- Neurona con Activación Adaptativa ---
class AdaptiveActivationNeuron(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features

        # Pesos y sesgo estándar de la neurona
        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)
        self.bias = nn.Parameter(torch.randn(output_features) * 0.1)

        # Parámetros para controlar la mezcla de funciones de activación
        # Inicializamos estos 'gate_weights' de forma que al principio
        # una de las activaciones tenga más peso (ej. ReLU)
        self.gate_weights = nn.Parameter(torch.randn(output_features, 2) * 0.1) # Dos opciones de activación

    def forward(self, x):
        linear_output = torch.matmul(x, self.weight) + self.bias

        # Aplicamos las funciones de activación candidatas a la salida lineal
        output_relu = F.relu(linear_output)
        output_sigmoid = torch.sigmoid(linear_output) # O podrías usar lineal: linear_output

        # Concatenamos las salidas de las activaciones
        # Esto crea un tensor donde cada fila es [output_relu_i, output_sigmoid_i]
        combined_outputs = torch.stack([output_relu, output_sigmoid], dim=-1) # [batch_size, output_features, 2]

        # Los gate_weights se pasan por softmax para obtener una distribución de probabilidad
        # sobre las funciones de activación para cada neurona de salida.
        # F.softmax(self.gate_weights, dim=-1) tendrá forma [output_features, 2]
        # donde cada fila suma 1.

        # Necesitamos que los pesos de las puertas se apliquen a cada batch.
        # Expandimos los pesos de las puertas para que coincidan con las dimensiones del batch
        # [1, output_features, 2]
        softmax_gates = F.softmax(self.gate_weights, dim=-1).unsqueeze(0)

        # Realizamos la mezcla ponderada.
        # Element-wise multiplication: [batch_size, output_features, 2] * [1, output_features, 2]
        # Sumamos a lo largo de la última dimensión para obtener la salida final ponderada
        # [batch_size, output_features]
        activated_output = (combined_outputs * softmax_gates).sum(dim=-1)

        return activated_output

# --- Red Neuronal usando la Neurona de Activación Adaptativa ---
class AdaptiveActivationNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.adaptive_layer1 = AdaptiveActivationNeuron(input_size, hidden_size)
        self.adaptive_layer2 = AdaptiveActivationNeuron(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size) # La capa final puede ser lineal para regresión

    def forward(self, x):
        h1 = self.adaptive_layer1(x)
        h2 = self.adaptive_layer2(h1)
        out = self.output_layer(h2)
        return out

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.01
EPOCHS = 1000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas, aumentado para dar capacidad
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red con Neuronas de Activación Adaptativa ---")
adaptive_model = AdaptiveActivationNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_adaptive = torch.optim.Adam(adaptive_model.parameters(), lr=LEARNING_RATE)
historial_loss_adaptive = []

total_params_adaptive = sum(p.numel() for p in adaptive_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red de Activación Adaptativa: {total_params_adaptive}")

for epoch in range(EPOCHS):
    adaptive_model.train()
    optimizer_adaptive.zero_grad()

    y_pred = adaptive_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_adaptive.append(loss.item())

    loss.backward()
    optimizer_adaptive.step()

    if epoch % 100 == 0:
        print(f"Red Adaptativa - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Adaptativa): {historial_loss_adaptive[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

adaptive_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_adaptive = adaptive_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_adaptive, label="Predicción Red Adaptativa", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_adaptive, label="Pérdida Red Adaptativa", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red de Activación Adaptativa): {historial_loss_adaptive[-1]:.6f}")
print(f"Pérdida final (Red Estándar):                  {historial_loss_standard[-1]:.6f}")

##funcion adaptativa cambiante

##Propuesta: Activación Basada en la Divergencia de Información (Information Divergence Activation - IDA)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- NUEVO MÉTODO: Neurona con Activación Parametrizada Adaptativa (IDA - Simplified) ---
class IDANeuron(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features

        # Pesos y sesgo estándar de la neurona
        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)
        self.bias = nn.Parameter(torch.randn(output_features) * 0.1)

        # Parámetros APRENDIBLES para la forma de la función de activación (uno por neurona de salida)
        # Inspirado en conceptos como PReLU o Swish, pero con más flexibilidad aprendible

        # 'slope_param': Controla la pendiente de la parte positiva (como ReLU) o la escala general.
        self.slope_param = nn.Parameter(torch.ones(output_features) * 0.1)

        # 'offset_param': Controla el punto de activación/desplazamiento en el eje X.
        self.offset_param = nn.Parameter(torch.zeros(output_features) * 0.1)

        # 'shape_param': Controla la "curvatura" o mezcla con otra forma (e.g., sigmoide)
        # Este podría ser el "gate" hacia una transformación más sigmoidal vs. rectificada
        self.shape_param = nn.Parameter(torch.zeros(output_features) * 0.1)

        # NOTA: Los `* 0.1` inicializan los parámetros cerca de cero para que no dominen
        # al inicio y permitan que la neurona se comporte inicialmente como una ReLU o lineal.

    def forward(self, x):
        linear_output = torch.matmul(x, self.weight) + self.bias

        # Expandir los parámetros para que coincidan con el tamaño del batch
        # [1, output_features]
        expanded_slope = self.slope_param.unsqueeze(0)
        expanded_offset = self.offset_param.unsqueeze(0)
        expanded_shape = self.shape_param.unsqueeze(0)

        # Activación basada en la forma aprendible
        # Intentamos una combinación que pueda "elegir" entre ser más como ReLU o más suave/sigmoide
        # Y ajustar su posición y escala.

        # Parte rectificada (como ReLU, pero con offset y escala aprendibles)
        # Multiplicamos por Sigmoid(slope_param) para mantener la pendiente en un rango positivo y acotado
        relu_like_output = F.relu(linear_output - expanded_offset) * torch.sigmoid(expanded_slope)

        # Parte "suave" o sigmoide, cuya contribución es modulada por 'shape_param'
        # Usamos tanh porque tiene un rango de -1 a 1, complementando ReLU (0 a infinito)
        tanh_like_output = torch.tanh(linear_output) * torch.sigmoid(expanded_shape) # sigmoid(shape_param) como gate

        # La salida final es una suma ponderada (pero no con softmax de gates)
        # Esto permite que ambas partes contribuyan y que los parámetros se ajusten libremente.
        activated_output = relu_like_output + tanh_like_output

        return activated_output


# --- Red Neuronal usando la IDANeuron ---
class IDANet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.ida_layer1 = IDANeuron(input_size, hidden_size)
        self.ida_layer2 = IDANeuron(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size) # La capa final puede ser lineal para regresión

    def forward(self, x):
        h1 = self.ida_layer1(x)
        h2 = self.ida_layer2(h1)
        out = self.output_layer(h2)
        return out

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.01
EPOCHS = 1500 # Más épocas para darle tiempo a aprender las activaciones
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red con Neuronas IDA (Activación Adaptativa Parametrizada) ---")
ida_model = IDANet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_ida = torch.optim.Adam(ida_model.parameters(), lr=LEARNING_RATE)
historial_loss_ida = []

total_params_ida = sum(p.numel() for p in ida_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red IDA: {total_params_ida}")

for epoch in range(EPOCHS):
    ida_model.train()
    optimizer_ida.zero_grad()

    y_pred = ida_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_ida.append(loss.item())

    loss.backward()
    optimizer_ida.step()

    if epoch % 100 == 0:
        print(f"Red IDA - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red IDA): {historial_loss_ida[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar (ReLU) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

ida_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_ida = ida_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_ida, label="Predicción Red IDA", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_ida, label="Pérdida Red IDA", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red IDA): {historial_loss_ida[-1]:.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import random # Para selección aleatoria o condicional no gradiente

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- Definición de Funciones de Activación Disponibles ---
ACTIVATION_FUNCTIONS = {
    'relu': F.relu,
    'sigmoid': torch.sigmoid,
    'tanh': torch.tanh,
    'leaky_relu': F.leaky_relu # Leaky ReLU
}

# --- NUEVO MÉTODO: Neurona con Activación por Mutación Estructural (SMA) ---
class SMANeuron(nn.Module):
    def __init__(self, input_features, output_features, initial_activation='relu'):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features

        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)
        self.bias = nn.Parameter(torch.randn(output_features) * 0.1)

        # Cada neurona en la capa tiene su propia función de activación actual.
        # Almacenamos el nombre de la función para cada neurona de salida.
        # self.current_activations: una lista de strings para cada neurona en la capa
        self.current_activations = [initial_activation] * output_features

        # Para la "indelebilidad" y para controlar la mutación, podemos llevar un registro
        # del "rendimiento" reciente o del "estrés" de la neurona.
        # Esto no es un Parameter, se actualiza fuera del grafo de gradientes.
        self._neuron_recent_loss_contribution = [0.0] * output_features
        self._mutation_cooldown = [0] * output_features # Previene mutaciones constantes

        # Registramos un hook para capturar el gradiente de los pesos de la neurona
        # Esto nos da una señal de "estrés" o "necesidad de cambio"
        self.weight.register_hook(self._capture_grad_for_mutation)
        self._last_neuron_grad_norms = [0.0] * output_features # Norma del gradiente para cada neurona de salida

    def _capture_grad_for_mutation(self, grad):
        # Captura la norma del gradiente para cada columna (que corresponde a una neurona de salida)
        # Esto simula que cada neurona sabe cuán fuerte fue su corrección
        for i in range(self.output_features):
            self._last_neuron_grad_norms[i] = grad[:, i].norm().item()

    def forward(self, x):
        linear_output = torch.matmul(x, self.weight) + self.bias

        # Aplicamos la función de activación actual para cada neurona de salida.
        # Esto requiere un bucle o una aplicación inteligente si las funciones son diferentes
        activated_outputs = []
        for i in range(self.output_features):
            neuron_output = linear_output[:, i] # Salida lineal para la i-ésima neurona
            activation_func = ACTIVATION_FUNCTIONS[self.current_activations[i]]
            activated_outputs.append(activation_func(neuron_output))

        # Apilamos las salidas de las neuronas para formar el tensor de salida de la capa
        return torch.stack(activated_outputs, dim=1)

    def mutate_activation(self, global_loss, epoch, mutation_threshold=0.01, cooldown_epochs=50):
        # Este método se llamará externamente (por la red principal)
        # Simula la "mutación" de la función de activación de la neurona.
        # Es "indeleble" porque es un cambio discreto y no basado en gradientes reversibles.

        for i in range(self.output_features):
            if self._mutation_cooldown[i] > 0:
                self._mutation_cooldown[i] -= 1
                continue # Todavía en cooldown, no muta

            # Criterio de Mutación (un ejemplo "inasonable" y condicional):
            # Mutar si la neurona ha tenido un gradiente de peso muy pequeño (está "muerta" o no contribuye)
            # Y la pérdida global sigue siendo alta, o si el gradiente es erráticamente alto

            grad_norm = self._last_neuron_grad_norms[i]

            # Condición de "mutación por inanición" (neurona inactiva)
            if grad_norm < 0.001 and global_loss > mutation_threshold and epoch > 100:
                print(f"Neurona {i} mutando por inanición (Grad: {grad_norm:.6f}) de {self.current_activations[i]} en época {epoch}")
                # Elige una nueva activación al azar (excepto la actual)
                available_activations = list(ACTIVATION_FUNCTIONS.keys())
                available_activations.remove(self.current_activations[i])
                if available_activations: # Asegurarse de que haya otras opciones
                    self.current_activations[i] = random.choice(available_activations)
                    self._mutation_cooldown[i] = cooldown_epochs # Establece cooldown

            # Condición de "mutación por estrés" (neurona con gradiente errático/muy alto)
            elif grad_norm > 1.0 and global_loss > mutation_threshold * 2 and epoch > 100:
                print(f"Neurona {i} mutando por estrés (Grad: {grad_norm:.6f}) de {self.current_activations[i]} en época {epoch}")
                available_activations = list(ACTIVATION_FUNCTIONS.keys())
                available_activations.remove(self.current_activations[i])
                if available_activations:
                    self.current_activations[i] = random.choice(available_activations)
                    self._mutation_cooldown[i] = cooldown_epochs

            # Limpiar el último gradiente después de considerar la mutación
            self._last_neuron_grad_norms[i] = 0.0


# --- Red Neuronal usando la SMANeuron ---
class SMANet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # Cada capa puede tener su propia estrategia inicial de activación
        self.sma_layer1 = SMANeuron(input_size, hidden_size, initial_activation='relu')
        self.sma_layer2 = SMANeuron(hidden_size, hidden_size, initial_activation='tanh')
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = self.sma_layer1(x)
        h2 = self.sma_layer2(h1)
        out = self.output_layer(h2)
        return out

    def apply_mutations(self, global_loss, epoch):
        # Llama al método de mutación para cada capa SMANeuron
        self.sma_layer1.mutate_activation(global_loss, epoch)
        self.sma_layer2.mutate_activation(global_loss, epoch)

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.01
EPOCHS = 2000 # Más épocas para permitir mutaciones
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red con Neuronas SMA (Mutación Estructural) ---")
sma_model = SMANet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_sma = torch.optim.Adam(sma_model.parameters(), lr=LEARNING_RATE)
historial_loss_sma = []

total_params_sma = sum(p.numel() for p in sma_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red SMA: {total_params_sma}")

for epoch in range(EPOCHS):
    sma_model.train()
    optimizer_sma.zero_grad()

    y_pred = sma_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_sma.append(loss.item())

    loss.backward()

    # Después del backward, pero antes del step, podemos capturar los gradientes
    # y aplicar la mutación.
    # Alternativamente, podrías mutar después del step para que reaccionen a la corrección real.
    # Por "indeleble", aquí lo hacemos después del backward para que reaccione a la "señal de estrés"
    # y luego los pesos se ajustan.

    # Aplicar la mutación basada en el rendimiento de la neurona
    # La mutación es una decisión que "rompe" la diferenciabilidad
    sma_model.apply_mutations(loss.item(), epoch) # Pasa la pérdida global y la época

    optimizer_sma.step()

    if epoch % 100 == 0:
        print(f"Red SMA - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red SMA): {historial_loss_sma[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar (ReLU) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

sma_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_sma = sma_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_sma, label="Predicción Red SMA", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_sma, label="Pérdida Red SMA", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red SMA): {historial_loss_sma[-1]:.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")

## Neurona polinómica dinámica (clasificador de círculos)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 1. Definición de NeuronaPoliDynBatch ---
class NeuronaPoliDynBatch(nn.Module):
    def __init__(self, input_dim, estado_dim=None):
        super().__init__()
        self.input_dim = input_dim
        self.estado_dim = estado_dim or input_dim

        self.w = nn.Parameter(torch.randn(input_dim * input_dim) * 0.1)
        self.v = nn.Parameter(torch.randn(input_dim) * 0.1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.bias_z = nn.Parameter(torch.zeros(1))

        self.register_buffer('estado', torch.zeros(1, self.estado_dim))

    def forward(self, x):
        batch_size = x.shape[0]
        x_outer = x.unsqueeze(2) * x.unsqueeze(1)
        x_outer_flat = x_outer.view(batch_size, -1)
        w_flat = self.w.view(-1)

        z1 = torch.matmul(x_outer_flat, w_flat)
        z2 = torch.matmul(x, self.v)
        Z = z1 + z2 + self.bias_z

        act = self.alpha * F.relu(Z) + (1 - self.alpha) * torch.tanh(Z)

        nuevo_estado_val = 0.9 * self.estado.mean() + 0.1 * act.mean().detach()
        self.estado = nuevo_estado_val.repeat(1, self.estado_dim)

        confianza = x.std(dim=1)
        carga = Z.norm(p=2)

        return {
            'salida': act,
            'estado': self.estado.clone().detach(),
            'confianza': confianza,
            'carga': carga
        }

# --- 2. Definición del Modelo de Red Neuronal ---
class CircleClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.neurona_polidyn = NeuronaPoliDynBatch(input_dim=input_dim)
        self.linear_output = nn.Linear(1, 1)

    def forward(self, x):
        polidyn_out = self.neurona_polidyn(x)
        x_processed = polidyn_out['salida'].unsqueeze(1)
        output = torch.sigmoid(self.linear_output(x_processed))
        return output

# --- 3. Preparación del Dataset Sintético ---
n_samples = 500
noise = 0.1
factor = 0.5

X, y = make_circles(n_samples=n_samples, noise=noise, factor=factor, random_state=42)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

# --- 4. Configuración del Modelo, Pérdida y Optimizador ---
input_dim = X_train_tensor.shape[1]
model = CircleClassifier(input_dim=input_dim).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# --- 5. Entrenamiento del Modelo con Visualización Progresiva ---
epochs = 1000
batch_size = 32
plot_every = 200 # Visualizar frontera de decisión cada 200 épocas

alpha_vals = []
estado_vals = []

print("\n--- Iniciando Entrenamiento con Visualización Progresiva ---")
for epoch in tqdm(range(epochs), desc="Entrenamiento"):
    model.train() # Pone el modelo en modo entrenamiento

    # Shuffle los datos para cada época
    indices = torch.randperm(X_train_tensor.size(0))
    X_shuffled = X_train_tensor[indices]
    y_shuffled = y_train_tensor[indices]

    for i in range(0, X_train_tensor.size(0), batch_size):
        inputs = X_shuffled[i:i+batch_size]
        labels = y_shuffled[i:i+batch_size]

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # Guardamos valores para las gráficas de evolución
    alpha_vals.append(model.neurona_polidyn.alpha.item())
    estado_vals.append(model.neurona_polidyn.estado.mean().item())

    # Visualiza frontera de decisión cada N épocas o en la última
    if epoch % plot_every == 0 or epoch == epochs - 1:
        model.eval() # Pone el modelo en modo evaluación para la inferencia
        with torch.no_grad():
            test_outputs = model(X_test_tensor)
            test_preds = (test_outputs > 0.5).float()
            accuracy = (test_preds == y_test_tensor).float().mean()
            tqdm.write(f"Época {epoch}/{epochs}, Pérdida: {loss.item():.4f}, Precisión en test: {accuracy.item():.4f}, Alpha: {model.neurona_polidyn.alpha.item():.4f}")

            # Visualización de frontera
            x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
            y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
            xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                                 np.linspace(y_min, y_max, 100))

            grid_points = np.c_[xx.ravel(), yy.ravel()]
            grid_points_tensor = torch.tensor(grid_points, dtype=torch.float32).to(device)

            # Obtener las predicciones del modelo sobre la grilla
            # Asegúrate que el modelo devuelva la probabilidad directamente
            zz = model(grid_points_tensor).cpu().numpy().reshape(xx.shape)

            plt.figure(figsize=(8, 6))
            plt.contourf(xx, yy, zz, levels=50, cmap="coolwarm", alpha=0.7) # Plotear prob. continua
            plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolors='k', s=20)
            plt.title(f"Frontera de decisión - Época {epoch}")
            plt.xlabel("X1")
            plt.ylabel("X2")
            plt.colorbar(label='Probabilidad de Clase 1')
            plt.grid(True)
            plt.show()

print("--- Entrenamiento Completado ---")

# === Gráficas finales de evolución de Alpha y Estado ===
#plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(alpha_vals, label="α (mezcla de activaciones)")
plt.title("Evolución de alpha durante el entrenamiento")
plt.xlabel("Época")
plt.ylabel("Valor de α")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(estado_vals, label="Estado (media)", color="orange")
plt.title("Evolución del estado interno")
plt.xlabel("Época")
plt.ylabel("Valor medio del estado")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# --- Evaluación Final ---
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    test_preds = (test_outputs > 0.5).float()
    accuracy = (test_preds == y_test_tensor).float().mean()
    print(f"\nPrecisión final en el conjunto de prueba: {accuracy.item():.4f}")

    print(f"Valor final del parámetro alpha: {model.neurona_polidyn.alpha.item():.4f}")
    print(f"Valor final del estado interno de la neurona: {model.neurona_polidyn.estado.mean().item():.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import imageio
import os

class NeuronaPoliDynBatch(nn.Module):
    def __init__(self, n_vars):
        super().__init__()
        self.n_vars = n_vars
        self.coeff = nn.Parameter(torch.randn(n_vars+1))  # coeficientes polinomio
        self.raw_alpha = nn.Parameter(torch.randn(1))    # parámetro sin constricción
        self.estado = torch.zeros(1, n_vars+1)           # estado dinámico
        # self.carga no es parámetro, se calcula on-the-fly

    def forward(self, x):
        """
        x: batch x n_vars
        """
        alpha = torch.sigmoid(self.raw_alpha)  # fuerza a [0,1]
        # Calculamos la función polinomial
        # Para cada variable, calculamos potencias y combinamos con coeficientes
        # vectorización por batch
        # Vector x (batch, n_vars), queremos (batch, n_vars+1) con potencias
        # potencias de cada variable: x**i para i=0..n_vars
        # vector de potencias para batch: shape (batch, n_vars+1)
        x_powers = torch.stack([x**i for i in range(self.n_vars+1)], dim=2)  # (batch, n_vars, n_vars+1)
        # ahora multiplicar coeficiente por potencia correspondiente y sumar sobre variables:
        # Pero coef es (n_vars+1,) y hay n_vars variables, así que sumamos sobre n_vars
        # función polinomial como sum_j coef[j] * sum_i x_i^j
        # vamos a hacerlo sum_i(sum_j coef_j * x_i^j)
        # sum_j coef_j * x_i^j = dot(coef, x_i_powers)
        # x_powers (batch, n_vars, n_vars+1)
        # coef (n_vars+1)
        poly_per_var = torch.matmul(x_powers, self.coeff)  # (batch, n_vars)
        # sumamos sobre variables
        act = torch.sum(poly_per_var, dim=1, keepdim=True)  # (batch, 1)

        # actualización dinámica del estado:
        # estado tiene shape (1, n_vars+1), mezclamos estado previo con coeficiente actual
        with torch.no_grad():
            self.estado = 0.9 * self.estado + 0.1 * self.coeff.detach().unsqueeze(0)

        carga = torch.mean(act**2)  # media del cuadrado activación para regularización

        # salida interpolada entre activación y estado sumado para dinámica
        salida = alpha * act + (1-alpha) * torch.sum(self.estado, dim=1, keepdim=True)
        return salida, alpha, self.estado, carga


class CircleClassifier(nn.Module):
    def __init__(self, n_vars=2, n_neuronas=5):
        super().__init__()
        self.n_neuronas = n_neuronas
        self.hidden = nn.ModuleList([NeuronaPoliDynBatch(n_vars) for _ in range(n_neuronas)])
        self.linear = nn.Linear(n_neuronas, 1)

    def forward(self, x):
        outs, alphas, estados, cargas = [], [], [], []
        for neurona in self.hidden:
            out, alpha, estado, carga = neurona(x)
            outs.append(out)
            alphas.append(alpha)
            estados.append(estado)
            cargas.append(carga)
        out_cat = torch.cat(outs, dim=1)  # (batch, n_neuronas)
        final = self.linear(out_cat)
        return final, torch.stack(alphas), torch.stack(estados), torch.stack(cargas)

# Funciones auxiliares para generación de dataset, entrenamiento y visualización

def generate_data(n=300):
    np.random.seed(0)
    X = np.random.uniform(-2, 2, (n,2))
    Y = (X[:,0]**2 + X[:,1]**2 < 1).astype(np.float32).reshape(-1,1)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32)

def plot_decision_boundary(model, epoch, path, grid_res=200):
    xlin = torch.linspace(-2, 2, grid_res)
    ylin = torch.linspace(-2, 2, grid_res)
    grid_x, grid_y = torch.meshgrid(xlin, ylin, indexing='xy')
    points = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)
    model.eval()
    with torch.no_grad():
        out, _, _, _ = model(points)
        probs = torch.sigmoid(out).reshape(grid_res, grid_res).cpu().numpy()

    plt.figure(figsize=(6,6))
    plt.contourf(grid_x.numpy(), grid_y.numpy(), probs, levels=50, cmap='RdBu', alpha=0.8)
    plt.colorbar(label='Probabilidad')
    plt.title(f'Frontera de decisión - Época {epoch}')
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

def entrenar_visualizar(model, X, Y, epochs=1000, plot_every=100, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    alphas_hist = []
    cargas_hist = []
    estados_hist = []
    imgs = []
    os.makedirs('tmp_neuronas', exist_ok=True)

    for epoch in range(1, epochs+1):
        model.train()
        opt.zero_grad()
        out, alphas, estados, cargas = model(X)
        loss = criterion(out, Y)
        # regularizador carga: peso pequeño para no dominar
        carga_mean = cargas.mean()
        loss += 1e-3 * carga_mean
        loss.backward()
        opt.step()

        alphas_hist.append(alphas.detach().cpu().numpy())
        cargas_hist.append(cargas.detach().cpu().numpy())
        estados_hist.append(estados.detach().cpu().numpy())

        if epoch % plot_every == 0:
            print(f'Época {epoch} - Pérdida: {loss.item():.4f} - Carga promedio: {carga_mean.item():.4f}')
            img_path = f'tmp_neuronas/epoch_{epoch:04d}.png'
            plot_decision_boundary(model, epoch, img_path)
            imgs.append(imageio.imread(img_path))

    # Guardar GIF animado de frontera
    gif_path = 'tmp_neuronas/frontera_animada.gif'
    imageio.mimsave(gif_path, imgs, fps=4)
    print(f'GIF animado guardado en {gif_path}')

    # Plot de evolución de alpha promedio y carga
    alphas_arr = np.array(alphas_hist)  # epochs x n_neuronas x 1
    cargas_arr = np.array(cargas_hist)  # epochs x n_neuronas

    avg_alpha = alphas_arr.mean(axis=1).squeeze()  # epochs
    avg_carga = cargas_arr.mean(axis=1).squeeze()  # epochs

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(avg_alpha)
    plt.title('Evolución promedio de alpha')
    plt.xlabel('Época')
    plt.ylabel('Alpha')

    plt.subplot(1,2,2)
    plt.plot(avg_carga)
    plt.title('Evolución promedio de carga')
    plt.xlabel('Época')
    plt.ylabel('Carga')

    plt.tight_layout()
    plt.show()

    return model

# Uso
X, Y = generate_data()
model = CircleClassifier(n_vars=2, n_neuronas=5)
entrenar_visualizar(model, X, Y, epochs=1000, plot_every=100)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import imageio
import os
from tqdm import tqdm # Importar tqdm para barras de progreso
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- 1. Definición de NeuronaPoliDynBatch (con ajustes y base polinomial) ---
class NeuronaPoliDynBatch(nn.Module):
    def __init__(self, n_vars):
        super().__init__()
        self.n_vars = n_vars
        # Coeficientes del polinomio (para cada variable de entrada)
        # Por simplicidad, un coeficiente para cada potencia hasta n_vars.
        # La salida de la neurona será la suma de los polinomios de cada variable.
        self.coeff = nn.Parameter(torch.randn(n_vars + 1)) # n_vars+1 para potencias de x^0 a x^n_vars
        self.raw_alpha = nn.Parameter(torch.randn(1))     # Parámetro sin constricción para alpha
        self.bias_poly = nn.Parameter(torch.zeros(1))     # Bias para la salida polinomial

        # Estado dinámico, inicializado por neurona. Refleja los coeficientes dinámicamente.
        # Shape: (1, n_vars + 1) para reflejar la forma de self.coeff
        self.register_buffer('estado', torch.zeros(1, n_vars + 1))

    def forward(self, x):
        """
        x: [batch_size, n_vars]
        """
        alpha = torch.sigmoid(self.raw_alpha)  # Fuerza alpha al rango [0,1]

        # Calcular potencias de cada variable de entrada para el polinomio
        # x_powers_all: [batch_size, n_vars, n_vars+1]
        # Por ejemplo, si n_vars=2, x es [x0, x1]. Queremos [[x0^0, x0^1, x0^2], [x1^0, x1^1, x1^2]]
        x_powers_all = torch.stack([x**i for i in range(self.n_vars + 1)], dim=2)

        # Calcular el valor polinomial para cada variable de entrada
        # poly_val_per_var: [batch_size, n_vars]
        # Esto calcula sum_{j=0}^{n_vars} coeff[j] * x_i^j para cada x_i
        poly_val_per_var = torch.matmul(x_powers_all, self.coeff)

        # La activación 'act' de la neurona es la suma de los valores polinomiales de todas las variables de entrada
        # act: [batch_size, 1]
        act = torch.sum(poly_val_per_var, dim=1, keepdim=True) + self.bias_poly

        # Actualización dinámica del estado
        # El estado evoluciona basándose en sus coeficientes actuales
        with torch.no_grad():
            self.estado = 0.9 * self.estado + 0.1 * self.coeff.detach().unsqueeze(0)

        # Carga: media del cuadrado de la activación para regularización
        carga = torch.mean(act**2)

        # Salida final: interpolación entre la activación polinomial y la suma del estado
        salida = alpha * act + (1 - alpha) * torch.sum(self.estado, dim=1, keepdim=True)
        return salida, alpha, self.estado, carga

# --- 2. Definición del Modelo de Red Neuronal (CircleClassifier) ---
class CircleClassifier(nn.Module):
    def __init__(self, n_vars=2, n_neuronas=5):
        super().__init__()
        self.n_neuronas = n_neuronas
        # Colección de NeuronasPoliDynBatch
        self.hidden = nn.ModuleList([NeuronaPoliDynBatch(n_vars) for _ in range(n_neuronas)])
        # Capa lineal final que mapea las n_neuronas a 1 salida para clasificación binaria
        self.linear = nn.Linear(n_neuronas, 1)

    def forward(self, x):
        outs, alphas, estados, cargas = [], [], [], []
        for neurona in self.hidden:
            out, alpha, estado, carga = neurona(x)
            outs.append(out)       # out: [batch_size, 1]
            alphas.append(alpha)   # alpha: [1]
            estados.append(estado) # estado: [1, n_vars+1]
            cargas.append(carga)   # carga: escalar

        # Concatena las salidas de todas las neuronas ocultas
        # out_cat: [batch_size, n_neuronas]
        out_cat = torch.cat(outs, dim=1)

        # Pasa por la capa lineal final y sigmoid para la probabilidad de clase
        final = self.linear(out_cat)

        # Apila los valores recopilados para devolverlos.
        # alphas_stacked: [n_neuronas, 1]
        # estados_stacked: [n_neuronas, 1, n_vars+1]
        # cargas_stacked: [n_neuronas]
        return final, torch.stack(alphas), torch.stack(estados), torch.stack(cargas)

# --- 3. Funciones auxiliares para Dataset, Entrenamiento y Visualización ---

def generate_data(n_samples=500, noise=0.1, factor=0.5, random_state=42):
    X, y = make_circles(n_samples=n_samples, noise=noise, factor=factor, random_state=random_state)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1) # [N, 1] para BCELoss
    return X_tensor, y_tensor

def plot_decision_boundary(model, X, Y, epoch, config_str, save_dir, grid_res=200):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xlin = torch.linspace(x_min, x_max, grid_res)
    ylin = torch.linspace(y_min, y_max, grid_res)
    grid_x, grid_y = torch.meshgrid(xlin, ylin, indexing='xy')
    points = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1).to(X.device) # Asegurar dispositivo

    model.eval()
    with torch.no_grad():
        out, _, _, _ = model(points)
        probs = torch.sigmoid(out).reshape(grid_res, grid_res).cpu().numpy()

    plt.figure(figsize=(7,7))
    plt.contourf(grid_x.cpu().numpy(), grid_y.cpu().numpy(), probs, levels=50, cmap='RdBu', alpha=0.8)
    plt.colorbar(label='Probabilidad de Clase 1')
    plt.scatter(X.cpu().numpy()[:, 0], X.cpu().numpy()[:, 1], c=Y.cpu().numpy().flatten(), s=20, cmap='RdBu', edgecolors='k')
    plt.title(f'Frontera de decisión - Época {epoch}\nConfig: {config_str}')
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.tight_layout()
    img_path = os.path.join(save_dir, f'epoch_{epoch:04d}.png')
    plt.savefig(img_path)
    plt.close()
    return img_path

def entrenar_visualizar(n_neuronas, reg_weight, epochs=1000, plot_every=100, lr=1e-3, batch_size=32, device='cpu'):
    # Generar y preparar datos
    X, Y = generate_data()
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

    X_train = X_train.to(device)
    y_train = y_train.to(device)
    X_test = X_test.to(device)
    y_test = y_test.to(device)

    model = CircleClassifier(n_vars=X.shape[1], n_neuronas=n_neuronas).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss() # Usa logits directamente, más estable que BCELoss con sigmoid aparte

    alphas_hist_per_neuron = [[] for _ in range(n_neuronas)]
    cargas_hist_per_neuron = [[] for _ in range(n_neuronas)]
    # estados_hist_per_neuron = [[] for _ in range(n_neuronas)] # Para estados individuales si se necesitan

    imgs = []
    config_str = f"N={n_neuronas}_R={reg_weight}"
    save_dir = f'tmp_neuronas_{config_str}'
    os.makedirs(save_dir, exist_ok=True)

    print(f"--- Iniciando Entrenamiento para {config_str} ---")
    for epoch in tqdm(range(1, epochs + 1), desc=f"Entrenamiento {config_str}"):
        model.train()

        # Shuffle datos por época
        indices = torch.randperm(X_train.size(0))
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        for i in range(0, X_train.size(0), batch_size):
            inputs = X_shuffled[i:i+batch_size]
            labels = y_shuffled[i:i+batch_size]

            optimizer.zero_grad()
            outputs, alphas, estados, cargas = model(inputs)
            loss = criterion(outputs, labels)

            # Añadir regularización de carga
            loss += reg_weight * cargas.mean() # Promedio de carga de todas las neuronas en el batch

            loss.backward()
            optimizer.step()

        # Guardar historial de alphas y cargas por neurona
        for j in range(n_neuronas):
            alphas_hist_per_neuron[j].append(alphas[j].item())
            cargas_hist_per_neuron[j].append(cargas[j].item())
            # estados_hist_per_neuron[j].append(estados[j].mean().item()) # Si se quiere el promedio del estado de c/neurona

        if epoch % plot_every == 0 or epoch == epochs:
            model.eval()
            with torch.no_grad():
                test_outputs, _, _, _ = model(X_test)
                test_preds = (torch.sigmoid(test_outputs) > 0.5).float()
                accuracy = (test_preds == y_test).float().mean()
                tqdm.write(f"Época {epoch}/{epochs}, Pérdida: {loss.item():.4f}, Precisión en test: {accuracy.item():.4f}, Carga Promedio: {cargas.mean().item():.4f}")

            img_path = plot_decision_boundary(model, X_train, y_train, epoch, config_str, save_dir)
            imgs.append(imageio.imread(img_path))

    # Guardar GIF animado
    gif_path = os.path.join(save_dir, 'frontera_animada.gif')
    imageio.mimsave(gif_path, imgs, fps=4)
    print(f'GIF animado guardado en {gif_path}')

    # Plot de evolución de alpha y carga por neurona
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    for j in range(n_neuronas):
        plt.plot(alphas_hist_per_neuron[j], label=f'α Neurona {j+1}')
    plt.title(f'Evolución de alpha (N={n_neuronas}, R={reg_weight})')
    plt.xlabel('Época')
    plt.ylabel('Valor de α')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    for j in range(n_neuronas):
        plt.plot(cargas_hist_per_neuron[j], label=f'Carga Neurona {j+1}')
    plt.title(f'Evolución de carga (N={n_neuronas}, R={reg_weight})')
    plt.xlabel('Época')
    plt.ylabel('Carga')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'alpha_carga_evolution.png'))
    plt.show()

    # Retornar la precisión final para comparación
    final_accuracy = accuracy.item()
    return {"model": model, "final_accuracy": final_accuracy,
            "alphas_hist": alphas_hist_per_neuron,
            "cargas_hist": cargas_hist_per_neuron}

# --- Configuración de Experimentos ---
configs = [
    {"n_neuronas": 2, "reg_weight": 1e-3},
    {"n_neuronas": 5, "reg_weight": 1e-3},
    {"n_neuronas": 5, "reg_weight": 1e-4},
    {"n_neuronas": 10, "reg_weight": 5e-4},
]

resultados = {}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for conf in configs:
    print(f"\n=======================================================")
    print(f"Ejecutando configuración: Neuronas={conf['n_neuronas']}, Regularización={conf['reg_weight']}")
    print(f"=======================================================\n")
    historia = entrenar_visualizar(
        n_neuronas=conf["n_neuronas"],
        reg_weight=conf["reg_weight"],
        epochs=1000,
        plot_every=100,
        device=device # Pasar el dispositivo a la función
    )
    resultados[(conf["n_neuronas"], conf["reg_weight"])] = historia

print("\n--- Resumen de Resultados ---")
for (n, r), res in resultados.items():
    print(f"Config (N={n}, R={r}): Precisión Final = {res['final_accuracy']:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NeuronaPoliDynBatchConEstado(nn.Module):
    def __init__(self, n_vars=2, grado=2, estado_dim=8):
        super().__init__()
        self.n_vars = n_vars
        self.grado = grado
        self.estado_dim = estado_dim

        # Número de coeficientes polinomiales (combinaciones con repetición)
        self.n_coeffs = int(torch.combinations(torch.arange(n_vars).repeat(grado), r=grado, with_replacement=True).shape[0])

        # Para simplicidad, pondremos coeficientes fijos pero entrenables lineales
        self.coeffs = nn.Parameter(torch.randn(self.n_coeffs))
        self.bias_poly = nn.Parameter(torch.randn(1))

        # Parámetro para interpolación alfa (entre lineal y polinómico)
        self.raw_alpha = nn.Parameter(torch.randn(1))

        # Estado inicial aprendible (vector)
        self.estado_inicial = nn.Parameter(torch.zeros(estado_dim))

        # Capa para actualizar estado recurrentemente
        # Entrada: estado anterior + activación polinómica (así concatenamos)
        self.fc_estado_update = nn.Linear(estado_dim + 1, estado_dim)

        # Capa para proyectar estado a un escalar que modula la salida
        self.fc_estado_influence = nn.Linear(estado_dim, 1)

    def forward(self, x, estado=None):
        """
        x: tensor (batch_size, n_vars)
        estado: tensor (batch_size, estado_dim) o None para reinicializar
        """
        batch_size = x.shape[0]

        # Inicializamos estado si no se recibe (reinicio por batch)
        if estado is None:
            estado = self.estado_inicial.unsqueeze(0).repeat(batch_size, 1)

        # Para simplicidad, activación polinómica = suma de entradas ^ grado (en realidad, sería mejor
        # construir la base polinomial completa, aquí simplificamos)
        # Aquí hacemos una aproximación: (x.sum(dim=1) ** grado) * coef + bias
        # Nota: no es la forma polinómica completa, pero sirve para ilustrar el flujo.
        x_sum = x.sum(dim=1, keepdim=True)
        act_poly = (x_sum ** self.grado) * self.coeffs.mean() + self.bias_poly

        # Calculo alpha entre 0 y 1 con sigmoid para interpolar
        alpha = torch.sigmoid(self.raw_alpha)

        # Estado influencia: proyectamos estado a escalar y sumamos
        influencia_estado = self.fc_estado_influence(estado)

        # Salida final: interpolación lineal/polinómica + influencia del estado
        salida = alpha * act_poly + (1 - alpha) * x_sum + influencia_estado

        # Actualizamos estado: concatenamos estado anterior + salida polinómica (sin influencia)
        entrada_estado = torch.cat([estado, act_poly.detach()], dim=1)  # Detach para evitar ciclos infinitos
        nuevo_estado = torch.tanh(self.fc_estado_update(entrada_estado))

        return salida, nuevo_estado

# Dataset sintético de círculos para entrenamiento
def crear_dataset_circulos(n_samples=500):
    # Coordenadas 2D aleatorias uniformes en [-1,1]
    X = torch.rand(n_samples, 2) * 2 - 1
    # Etiqueta 1 si dentro círculo radio 0.7, 0 sino
    y = ((X[:,0]**2 + X[:,1]**2) < 0.7**2).float().unsqueeze(1)
    return X, y

# Entrenamiento simplificado
def entrenar_neurona():
    n_epochs = 300
    lr = 0.01

    neurona = NeuronaPoliDynBatchConEstado()
    opt = torch.optim.Adam(neurona.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    X_train, y_train = crear_dataset_circulos()

    estado = None

    for epoch in range(n_epochs):
        opt.zero_grad()

        salida, estado = neurona(X_train, estado)  # Estado se pasa de un batch al siguiente

        loss = loss_fn(salida, y_train)
        loss.backward()

        # Clipping gradientes para evitar explosión
        torch.nn.utils.clip_grad_norm_(neurona.parameters(), max_norm=1.0)

        opt.step()

        # Reiniciar estado cada epoch para simplificar (opcional)
        estado = None

        if (epoch+1) % 50 == 0:
            print(f"Epoch {epoch+1} - Loss: {loss.item():.5f}")

    return neurona

# Ejecutar entrenamiento y verificar
if __name__ == "__main__":
    neurona_entrenada = entrenar_neurona()

## Neurona resonante

In [ ]:
from sklearn.datasets import make_circles
import numpy as np
import matplotlib.pyplot as plt

# 1. Crear datos
X, y = make_circles(n_samples=300, factor=0.5, noise=0.1)

# Normalizar
X = (X - X.mean(axis=0)) / X.std(axis=0)

# 2. Crear clase NeuronaResonante
class NeuronaResonante:
    def __init__(self, fuerza=0.5, amplificacion=1.0):
        self.entradas = []  # conexiones a otras neuronas
        self.pesos = []
        self.bias = np.random.randn()
        self.fuerza = fuerza
        self.amplificacion = amplificacion
        self.salida = 0.0

    def conectar(self, neurona, peso=None):
        self.entradas.append(neurona)
        if peso is None:
            peso = np.random.uniform(-1, 1)
        self.pesos.append(peso)

    def activar(self):
        total = sum(n.salida * w for n, w in zip(self.entradas, self.pesos)) + self.bias
        if total * self.amplificacion >= self.fuerza:
            self.salida = 1 / (1 + np.exp(-total))  # sigmoide
        else:
            self.salida = 0.0

# 3. Crear red: 2 neuronas entrada, 5 ocultas, 1 salida
def crear_red():
    entrada = [NeuronaResonante(fuerza=0.0) for _ in range(2)]  # color 0.65
    ocultas = [NeuronaResonante() for _ in range(5)]
    salida = NeuronaResonante(fuerza=0.7, amplificacion=2.0)  # color 0.80

    for h in ocultas:
        for e in entrada:
            h.conectar(e)

    for h in ocultas:
        salida.conectar(h)

    return entrada, ocultas, salida

# 4. Evaluar red
def evaluar_red(entrada, ocultas, salida, X, y):
    errores = []
    for xi, yi in zip(X, y):
        for i, val in enumerate(xi):
            entrada[i].salida = val
        for h in ocultas:
            h.activar()
        salida.activar()
        error = (salida.salida - yi) ** 2
        errores.append(error)
    return np.mean(errores)

# 5. Evolución básica
def entrenar(n_generaciones=100):
    best_loss = float('inf')
    best_red = None
    for _ in range(n_generaciones):
        entrada, ocultas, salida = crear_red()
        loss = evaluar_red(entrada, ocultas, salida, X, y)
        if loss < best_loss:
            best_loss = loss
            best_red = (entrada, ocultas, salida)
            print(f"Nuevo mejor loss: {loss:.4f}")
    return best_red
Vizualizar

In [ ]:
from sklearn.datasets import make_circles
import numpy as np
import matplotlib.pyplot as plt

# 1. Crear datos
X, y = make_circles(n_samples=300, factor=0.5, noise=0.1)

# Normalizar
X = (X - X.mean(axis=0)) / X.std(axis=0)

# 2. Crear clase NeuronaResonante
class NeuronaResonante:
    def __init__(self, fuerza=0.5, amplificacion=1.0):
        self.entradas = []  # conexiones a otras neuronas
        self.pesos = []
        self.bias = np.random.randn()
        self.fuerza = fuerza
        self.amplificacion = amplificacion
        self.salida = 0.0

    def conectar(self, neurona, peso=None):
        self.entradas.append(neurona)
        if peso is None:
            peso = np.random.uniform(-1, 1)
        self.pesos.append(peso)

    def activar(self):
        total = sum(n.salida * w for n, w in zip(self.entradas, self.pesos)) + self.bias
        if total * self.amplificacion >= self.fuerza:
            self.salida = 1 / (1 + np.exp(-total))  # sigmoide
        else:
            self.salida = 0.0

# 3. Crear red: 2 neuronas entrada, 5 ocultas, 1 salida
def crear_red():
    entrada = [NeuronaResonante(fuerza=0.0) for _ in range(2)]  # color 0.65
    ocultas = [NeuronaResonante() for _ in range(5)]
    salida = NeuronaResonante(fuerza=0.7, amplificacion=2.0)  # color 0.80

    for h in ocultas:
        for e in entrada:
            h.conectar(e)

    for h in ocultas:
        salida.conectar(h)

    return entrada, ocultas, salida

# 4. Evaluar red
def evaluar_red(entrada, ocultas, salida, X, y):
    errores = []
    for xi, yi in zip(X, y):
        for i, val in enumerate(xi):
            entrada[i].salida = val
        for h in ocultas:
            h.activar()
        salida.activar()
        error = (salida.salida - yi) ** 2
        errores.append(error)
    return np.mean(errores)

# 5. Evolución básica
def entrenar(n_generaciones=100):
    best_loss = float('inf')
    best_red = None
    for _ in range(n_generaciones):
        entrada, ocultas, salida = crear_red()
        loss = evaluar_red(entrada, ocultas, salida, X, y)
        if loss < best_loss:
            best_loss = loss
            best_red = (entrada, ocultas, salida)
            print(f"Nuevo mejor loss: {loss:.4f}")
    return best_red

from sklearn.datasets import make_circles
import numpy as np
import matplotlib.pyplot as plt

# 1. Crear datos
X, y = make_circles(n_samples=300, factor=0.5, noise=0.1)

# Normalizar
X = (X - X.mean(axis=0)) / X.std(axis=0)

# 2. Crear clase NeuronaResonante
class NeuronaResonante:
    def __init__(self, fuerza=0.5, amplificacion=1.0):
        self.entradas = []  # conexiones a otras neuronas
        self.pesos = []
        self.bias = np.random.randn()
        self.fuerza = fuerza
        self.amplificacion = amplificacion
        self.salida = 0.0

    def conectar(self, neurona, peso=None):
        self.entradas.append(neurona)
        if peso is None:
            peso = np.random.uniform(-1, 1)
        self.pesos.append(peso)

    def activar(self):
        total = sum(n.salida * w for n, w in zip(self.entradas, self.pesos)) + self.bias
        if total * self.amplificacion >= self.fuerza:
            self.salida = 1 / (1 + np.exp(-total))  # sigmoide
        else:
            self.salida = 0.0

# 3. Crear red: 2 neuronas entrada, 5 ocultas, 1 salida
def crear_red():
    entrada = [NeuronaResonante(fuerza=0.0) for _ in range(2)]  # color 0.65
    ocultas = [NeuronaResonante() for _ in range(5)]
    salida = NeuronaResonante(fuerza=0.7, amplificacion=2.0)  # color 0.80

    for h in ocultas:
        for e in entrada:
            h.conectar(e)

    for h in ocultas:
        salida.conectar(h)

    return entrada, ocultas, salida

# 4. Evaluar red
def evaluar_red(entrada, ocultas, salida, X, y):
    errores = []
    for xi, yi in zip(X, y):
        for i, val in enumerate(xi):
            entrada[i].salida = val
        for h in ocultas:
            h.activar()
        salida.activar()
        error = (salida.salida - yi) ** 2
        errores.append(error)
    return np.mean(errores)

# 5. Evolución básica
def entrenar(n_generaciones=100):
    best_loss = float('inf')
    best_red = None
    for _ in range(n_generaciones):
        entrada, ocultas, salida = crear_red()
        loss = evaluar_red(entrada, ocultas, salida, X, y)
        if loss < best_loss:
            best_loss = loss
            best_red = (entrada, ocultas, salida)
            print(f"Nuevo mejor loss: {loss:.4f}")
    return best_red

# Example of how to use the train function:
best_network = entrenar(n_generaciones=200)


# --- Vizualización de los resultados ---

def predecir_con_red(entrada_neuronas, ocultas_neuronas, salida_neurona, punto):
    """
    Pasa un único punto (par de coordenadas) a través de la red entrenada
    y retorna la salida de la neurona de salida.
    """
    # Asegurar que el punto tenga la forma correcta (ej. numpy array de 2 elementos)
    if not isinstance(punto, np.ndarray):
        punto = np.array(punto)

    # Asignar los valores del punto a las neuronas de entrada
    for i, val in enumerate(punto):
        entrada_neuronas[i].salida = val

    # Activar las neuronas ocultas
    for h in ocultas_neuronas:
        h.activar()

    # Activar la neurona de salida
    salida_neurona.activar()

    # Retornar la salida final (puede ser un valor entre 0 y 1)
    return salida_neurona.salida

def vizualizar_frontera(entrada_red, ocultas_red, salida_red, X, y):
    """
    Genera una cuadrícula de puntos y usa la red entrenada para predecir
    la clase para cada punto en la cuadrícula. Luego visualiza los datos
    originales y la frontera de decisión predicha.
    """
    print("\nVizualizando frontera de decisión...")

    # Definir el rango para la cuadrícula (basado en los datos normalizados)
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    # Definir la resolución de la cuadrícula (más puntos = más detalle)
    h = 0.02

    # Crear la cuadrícula de puntos
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    # Aplanar la cuadrícula para pasarla por la red
    grid_points = np.c_[xx.ravel(), yy.ravel()]

    # Predecir para cada punto en la cuadrícula.
    # Dependiendo de la arquitectura de tu neurona de salida,
    # podrías necesitar un umbral (ej. 0.5) para decidir la clase.
    # Tu neurona de salida usa sigmoide, así que un umbral de 0.5 es razonable.
    Z = np.array([predecir_con_red(entrada_red, ocultas_red, salida_red, p) for p in grid_points])
    # Aplicar un umbral para obtener la clase predicha (0 o 1)
    Z = (Z > 0.5).astype(int)

    # Remodelar las predicciones de vuelta a la forma de la cuadrícula
    Z = Z.reshape(xx.shape)

    # Crear el gráfico
    plt.figure(figsize=(8, 6))
    # Usar contourf para dibujar la frontera de decisión con colores
    # cmap=plt.cm.RdBu puede ser bueno para 2 clases
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=0.8)

    # Dibujar los puntos de datos de entrenamiento originales
    # Colorearlos según su clase real (y)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors='k', marker='o', s=50)

    plt.title("Frontera de Decisión de la Neurona Resonante")
    plt.xlabel("Característica 1 (Normalizada)")
    plt.ylabel("Característica 2 (Normalizada)")
    plt.colorbar(label="Clase Predicha (0 o 1)")
    plt.show()


# Después de entrenar la red:
# Recuperar las neuronas entrenadas
if best_network:
    entrada_entrenada, ocultas_entrenadas, salida_entrenada = best_network

    # Visualizar la frontera de decisión usando la mejor red entrenada
    vizualizar_frontera(entrada_entrenada, ocultas_entrenadas, salida_entrenada, X, y)
else:
    print("No se pudo entrenar la red correctamente.")

In [ ]:
from sklearn.datasets import make_circles
import numpy as np
import matplotlib.pyplot as plt

# 1. Crear datos
X, y = make_circles(n_samples=300, factor=0.5, noise=0.1, random_state=42)

# Normalizar los datos
X = (X - X.mean(axis=0)) / X.std(axis=0)

# 2. Clase NeuronaResonante
class NeuronaResonante:
    def __init__(self, fuerza=0.5, amplificacion=1.0, bias_inicial=None):
        self.entradas = []
        self.pesos = []
        self.bias = bias_inicial if bias_inicial is not None else np.random.randn()
        self.fuerza = fuerza
        self.amplificacion = amplificacion
        self.salida = 0.0

    def conectar(self, neurona, peso=None):
        self.entradas.append(neurona)
        if peso is None:
            peso = np.random.uniform(-1, 1) # Rango inicial de pesos
        self.pesos.append(peso)

    def activar(self):
        total = sum(n.salida * w for n, w in zip(self.entradas, self.pesos)) + self.bias
        if total * self.amplificacion >= self.fuerza:
            self.salida = 1 / (1 + np.exp(-total))  # Sigmoide
        else:
            self.salida = 0.0

# 3. Crear red con hiperparámetros
def crear_red(num_neuronas_entrada, num_neuronas_ocultas,
              fuerza_ocultas, amplificacion_ocultas,
              fuerza_salida, amplificacion_salida):

    entrada = [NeuronaResonante(fuerza=0.0) for _ in range(num_neuronas_entrada)]
    ocultas = [NeuronaResonante(fuerza=fuerza_ocultas, amplificacion=amplificacion_ocultas)
               for _ in range(num_neuronas_ocultas)]
    salida = NeuronaResonante(fuerza=fuerza_salida, amplificacion=amplificacion_salida)

    # Conectar entradas a ocultas
    for h in ocultas:
        for e in entrada:
            h.conectar(e)

    # Conectar ocultas a salida
    for h in ocultas:
        salida.conectar(h)

    return entrada, ocultas, salida

# 4. Evaluar red (se mantiene igual)
def evaluar_red(entrada, ocultas, salida, X_data, y_data):
    errores = []
    for xi, yi in zip(X_data, y_data):
        for i, val in enumerate(xi):
            entrada[i].salida = val
        for h in ocultas:
            h.activar()
        salida.activar()
        error = (salida.salida - yi) ** 2
        errores.append(error)
    return np.mean(errores)

# 5. Evolución con mutación y copia de red
def entrenar_con_hiperparametros(X, y, config):
    best_loss = float('inf')
    best_red_state = None # Guardamos el estado (pesos, bias, etc.) de la mejor red

    # Inicializar una red con la configuración dada
    entrada, ocultas, salida = crear_red(
        config['num_neuronas_entrada'],
        config['num_neuronas_ocultas'],
        config['fuerza_ocultas'],
        config['amplificacion_ocultas'],
        config['fuerza_salida'],
        config['amplificacion_salida']
    )

    # Función auxiliar para copiar el estado de una red
    def copiar_estado_red(entrada_src, ocultas_src, salida_src):
        # Creamos nuevas neuronas con los mismos parámetros iniciales
        entrada_dst = [NeuronaResonante(n.fuerza, n.amplificacion, n.bias) for n in entrada_src]
        ocultas_dst = [NeuronaResonante(n.fuerza, n.amplificacion, n.bias) for n in ocultas_src]
        salida_dst = NeuronaResonante(salida_src.fuerza, salida_src.amplificacion, salida_src.bias)

        # Copiamos los pesos y reconstruimos las conexiones
        for i, h_dst in enumerate(ocultas_dst):
            for j, e_dst in enumerate(entrada_dst):
                h_dst.conectar(e_dst, ocultas_src[i].pesos[j])

        for i, h_dst in enumerate(ocultas_dst):
            salida_dst.conectar(h_dst, salida_src.pesos[i])

        return entrada_dst, ocultas_dst, salida_dst

    for gen in range(config['n_generaciones']):
        current_loss = evaluar_red(entrada, ocultas, salida, X, y)

        if current_loss < best_loss:
            best_loss = current_loss
            # Guardar el estado de la mejor red encontrada
            best_red_state = copiar_estado_red(entrada, ocultas, salida)
            print(f"Generación {gen+1}: Nuevo mejor loss: {best_loss:.4f}")

        # Mutar la red actual para la siguiente generación
        for neurona_list in [entrada, ocultas, [salida]]:
            for neurona in neurona_list:
                # Mutar bias
                if np.random.rand() < config['tasa_mutacion']:
                    neurona.bias += np.random.uniform(-config['magnitud_mutacion'], config['magnitud_mutacion'])
                # Mutar pesos
                for i in range(len(neurona.pesos)):
                    if np.random.rand() < config['tasa_mutacion']:
                        neurona.pesos[i] += np.random.uniform(-config['magnitud_mutacion'], config['magnitud_mutacion'])

        # Asegurarse de que las neuronas de entrada no tengan fuerza y amplificación cambiantes
        for n in entrada:
            n.fuerza = 0.0
            n.amplificacion = 1.0 # O el valor predeterminado si es necesario

    return best_red_state # Devolvemos la mejor red guardada


### Hiperparámetros y Configuración


# --- Configuración del Modelo y Entrenamiento (Hiperparámetros) ---
model_config = {
    'num_neuronas_entrada': 2,
    'num_neuronas_ocultas': 78, # Prueba con más o menos neuronas
    'fuerza_ocultas': 0.5,
    'amplificacion_ocultas': 1.0,
    'fuerza_salida': 0.7,
    'amplificacion_salida': 2.0,
    'n_generaciones': 10000, # Aumenta para más entrenamiento
    'tasa_mutacion': 0.15, # Probabilidad de mutar un peso o bias
    'magnitud_mutacion': 0.08 # Magnitud del cambio al mutar
}

# Entrenar la red con la configuración especificada
print("Iniciando entrenamiento con la configuración:")
for k, v in model_config.items():
    print(f"- {k}: {v}")

best_network = entrenar_con_hiperparametros(X, y, model_config)


### Visualización de los Resultados


def predecir_con_red(entrada_neuronas, ocultas_neuronas, salida_neurona, punto):
    """
    Pasa un único punto (par de coordenadas) a través de la red entrenada
    y retorna la salida de la neurona de salida.
    """
    if not isinstance(punto, np.ndarray):
        punto = np.array(punto)

    for i, val in enumerate(punto):
        entrada_neuronas[i].salida = val
    for h in ocultas_neuronas:
        h.activar()
    salida_neurona.activar()
    return salida_neurona.salida

def vizualizar_frontera(entrada_red, ocultas_red, salida_red, X, y):
    """
    Genera una cuadrícula de puntos y usa la red entrenada para predecir
    la clase para cada punto en la cuadrícula. Luego visualiza los datos
    originales y la frontera de decisión predicha.
    """
    print("\nVisualizando frontera de decisión...")

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    h = 0.02 # Resolución de la cuadrícula

    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    grid_points = np.c_[xx.ravel(), yy.ravel()]

    Z = np.array([predecir_con_red(entrada_red, ocultas_red, salida_red, p) for p in grid_points])
    Z = (Z > 0.5).astype(int) # Umbral de 0.5 para clasificación binaria
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors='k', marker='o', s=50)

    plt.title("Frontera de Decisión de la Neurona Resonante (Experimental)")
    plt.xlabel("Característica 1 (Normalizada)")
    plt.ylabel("Característica 2 (Normalizada)")
    plt.colorbar(label="Clase Predicha (0 o 1)")
    plt.show()

# Ejecutar visualización si el entrenamiento fue exitoso
if best_network:
    entrada_entrenada, ocultas_entrenadas, salida_entrenada = best_network
    vizualizar_frontera(entrada_entrenada, ocultas_entrenadas, salida_entrenada, X, y)
else:
    print("No se pudo entrenar la red correctamente. La mejor red no fue encontrada.")

In [ ]:
import numpy as np

def sigmoide(x):
    """Función de activación Sigmoide."""
    return 1 / (1 + np.exp(-x))

def relu(x):
    """Función de activación ReLU."""
    return np.maximum(0, x)

def calcular_activacion_neurona(entradas, pesos, umbral, funcion_activacion):
    """
    Calcula la suma ponderada y aplica la función de activación.

    Args:
        entradas (list): Valores de entrada a la neurona.
        pesos (list): Pesos asociados a cada entrada.
        umbral (float): Valor del umbral (bias).
        funcion_activacion (function): La función de activación a usar.

    Returns:
        float: La fuerza de activación (salida) de la neurona.
    """
    # 1. Calcular la suma ponderada de las entradas
    suma_ponderada = np.dot(entradas, pesos) + umbral
    print(f"  Suma ponderada (antes de la activación): {suma_ponderada:.4f}")

    # 2. Aplicar la función de activación
    fuerza_activacion = funcion_activacion(suma_ponderada)
    return fuerza_activacion

# --- Datos falsos para el ejemplo ---
entradas_ejemplo = [0.8, 0.5, 0.2]
pesos_ejemplo_1 = [0.6, -0.3, 0.9]
umbral_ejemplo_1 = -0.1

pesos_ejemplo_2 = [0.1, 0.7, -0.4]
umbral_ejemplo_2 = 0.5

print("--- Neurona con función de activación Sigmoide ---")
fuerza_activacion_sigmoide = calcular_activacion_neurona(
    entradas_ejemplo,
    pesos_ejemplo_1,
    umbral_ejemplo_1,
    sigmoide
)
print(f"La fuerza de activación (salida) de la neurona con Sigmoide es: {fuerza_activacion_sigmoide:.4f}\n")


print("--- Neurona con función de activación ReLU ---")
fuerza_activacion_relu = calcular_activacion_neurona(
    entradas_ejemplo,
    pesos_ejemplo_2,
    umbral_ejemplo_2,
    relu
)
print(f"La fuerza de activación (salida) de la neurona con ReLU es: {fuerza_activacion_relu:.4f}")

In [ ]:
from sklearn.datasets import make_circles
import numpy as np
import matplotlib.pyplot as plt

# 1. Crear datos
X, y = make_circles(n_samples=300, factor=0.5, noise=0.1)

# Normalizar
X = (X - X.mean(axis=0)) / X.std(axis=0)

# 2. Crear clase NeuronaResonante
class NeuronaResonante:
    def __init__(self, fuerza=0.5, amplificacion=1.0):
        self.entradas = []  # conexiones a otras neuronas
        self.pesos = []
        self.bias = np.random.randn()
        self.fuerza = fuerza
        self.amplificacion = amplificacion
        self.salida = 0.0
        # Añadimos un ID para identificar la neurona fácilmente
        self.id = id(self) # id() devuelve la identidad única de un objeto

    def conectar(self, neurona, peso=None):
        self.entradas.append(neurona)
        if peso is None:
            peso = np.random.uniform(-1, 1)
        self.pesos.append(peso)
        # Opcional: imprimir la conexión al establecerla
        # print(f"  Neurona {self.id} conectada desde Neurona {neurona.id} con peso {peso:.4f}")

    def activar(self):
        total = sum(n.salida * w for n, w in zip(self.entradas, self.pesos)) + self.bias
        if total * self.amplificacion >= self.fuerza:
            self.salida = 1 / (1 + np.exp(-total))  # sigmoide
        else:
            self.salida = 0.0

# 3. Crear red: 2 neuronas entrada, 5 ocultas, 1 salida
def crear_red():
    print("--- Creando una nueva red neuronal ---")
    entrada = [NeuronaResonante(fuerza=0.0) for _ in range(2)]
    ocultas = [NeuronaResonante() for _ in range(5)]
    salida = NeuronaResonante(fuerza=0.7, amplificacion=2.0)

    print("\nConectando Capa de Entrada a Capa Oculta:")
    for i, h in enumerate(ocultas):
        for j, e in enumerate(entrada):
            h.conectar(e)
            print(f"  Neurona Oculta {i} (ID: {h.id}) conectada desde Neurona Entrada {j} (ID: {e.id})")

    print("\nConectando Capa Oculta a Capa de Salida:")
    for i, h in enumerate(ocultas):
        salida.conectar(h)
        print(f"  Neurona Salida (ID: {salida.id}) conectada desde Neurona Oculta {i} (ID: {h.id})")

    print("--- Red creada ---")
    return entrada, ocultas, salida

# 4. Evaluar red
def evaluar_red(entrada, ocultas, salida, X, y):
    errores = []
    for xi, yi in zip(X, y):
        for i, val in enumerate(xi):
            entrada[i].salida = val
        for h in ocultas:
            h.activar()
        salida.activar()
        error = (salida.salida - yi) ** 2
        errores.append(error)
    return np.mean(errores)

# 5. Evolución básica
def entrenar(n_generaciones=100):
    best_loss = float('inf')
    best_red = None
    for gen in range(n_generaciones):
        print(f"\n--- Generación {gen + 1}/{n_generaciones} ---")
        entrada, ocultas, salida = crear_red()
        loss = evaluar_red(entrada, ocultas, salida, X, y)
        if loss < best_loss:
            best_loss = loss
            best_red = (entrada, ocultas, salida)
            print(f"Nuevo mejor loss: {loss:.4f}")
        else:
            print(f"Loss actual: {loss:.4f} (No mejor que el mejor: {best_loss:.4f})")
    return best_red

# Example of how to use the train function:
best_network = entrenar(n_generaciones=200)


# --- Vizualización de los resultados ---

def predecir_con_red(entrada_neuronas, ocultas_neuronas, salida_neurona, punto):
    """
    Pasa un único punto (par de coordenadas) a través de la red entrenada
    y retorna la salida de la neurona de salida.
    """
    if not isinstance(punto, np.ndarray):
        punto = np.array(punto)

    for i, val in enumerate(punto):
        entrada_neuronas[i].salida = val

    for h in ocultas_neuronas:
        h.activar()

    salida_neurona.activar()

    return salida_neurona.salida

def vizualizar_frontera(entrada_red, ocultas_red, salida_red, X, y):
    """
    Genera una cuadrícula de puntos y usa la red entrenada para predecir
    la clase para cada punto en la cuadrícula. Luego visualiza los datos
    originales y la frontera de decisión predicha.
    """
    print("\nVizualizando frontera de decisión...")

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    h = 0.02

    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    grid_points = np.c_[xx.ravel(), yy.ravel()]

    Z = np.array([predecir_con_red(entrada_red, ocultas_red, salida_red, p) for p in grid_points])
    Z = (Z > 0.5).astype(int)

    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors='k', marker='o', s=50)

    plt.title("Frontera de Decisión de la Neurona Resonante")
    plt.xlabel("Característica 1 (Normalizada)")
    plt.ylabel("Característica 2 (Normalizada)")
    plt.colorbar(label="Clase Predicha (0 o 1)")
    plt.show()


# Después de entrenar la red:
if best_network:
    entrada_entrenada, ocultas_entrenadas, salida_entrenada = best_network
    print("\n--- Mejor Red Entrenada ---")
    print(f"Neuronas de Entrada (IDs): {[n.id for n in entrada_entrenada]}")
    print(f"Neuronas Ocultas (IDs): {[n.id for n in ocultas_entrenadas]}")
    print(f"Neurona de Salida (ID): {salida_entrenada.id}")
    print("--- Fin Mejor Red Entrenada ---")

    vizualizar_frontera(entrada_entrenada, ocultas_entrenadas, salida_entrenada, X, y)
else:
    print("No se pudo entrenar la red correctamente.")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Crear un grafo dirigido para mostrar las ramificaciones
G = nx.DiGraph()

# Nodos principales
G.add_node("Nodo_Principal_A")
G.add_node("Nodo_Principal_B")

# Conectar los dos nodos principales
G.add_edge("Nodo_Principal_A", "Nodo_Principal_B")

# Ramificaciones del Nodo_Principal_A
G.add_edge("Nodo_Principal_A", "Rama_A1")
G.add_edge("Nodo_Principal_A", "Rama_A2")
G.add_edge("Rama_A1", "SubRama_A1.1")
G.add_edge("Rama_A1", "SubRama_A1.2")
G.add_edge("SubRama_A1.1", "Final_A1.1.1")

# Ramificaciones del Nodo_Principal_B
G.add_edge("Nodo_Principal_B", "Rama_B1")
G.add_edge("Nodo_Principal_B", "Rama_B2")
G.add_edge("Rama_B2", "SubRama_B2.1")
G.add_edge("Rama_B2", "SubRama_B2.2")
G.add_edge("SubRama_B2.2", "Final_B2.2.1")

# Posicionar los nodos para una mejor visualización (puedes probar diferentes layouts)
pos = nx.spring_layout(G, k=0.5, iterations=50) # 'spring_layout' es bueno para visualizar la estructura de un grafo

# Dibujar los nodos
nx.draw_networkx_nodes(G, pos, node_size=3000, node_color='lightblue')

# Dibujar las aristas
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True)

# Dibujar las etiquetas de los nodos
nx.draw_networkx_labels(G, pos, font_size=8)

# Mostrar el grafo
plt.title("Visualización de Nodos Ramificados")
plt.axis('off') # Ocultar los ejes
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Crear un grafo aleatorio (ej. modelo de Barabási-Albert para simular redes del mundo real)
# Puedes ajustar num_nodes y num_edges para cambiar la densidad
num_nodes = 10  # Número de nodos
num_edges_to_attach = 2 # Número de aristas a adjuntar desde un nuevo nodo a nodos existentes
G = nx.barabasi_albert_graph(num_nodes, num_edges_to_attach)

# Asignar un valor "multidimensional" (aquí, un solo valor aleatorio) a cada nodo para la coloración
# Esto simula un atributo que podrías obtener de un vector multidimensional para cada nodo.
node_values = np.random.rand(num_nodes)
node_colors = [plt.cm.viridis(val) for val in node_values] # Usar un mapa de colores

# Calcular el layout de los nodos
# Prueba con diferentes layouts para ver cuál se ajusta mejor a tu estructura
pos = nx.spring_layout(G, k=0.1, iterations=50, seed=42) # Ajusta 'k' para la distancia entre nodos

# Dibujar el grafo
plt.figure(figsize=(10, 10)) # Ajusta el tamaño de la figura

# Dibujar las aristas primero para que queden debajo de los nodos
nx.draw_networkx_edges(G, pos, edge_color='gray', alpha=0.5, width=0.5)

# Dibujar los nodos con el color asignado y tamaño pequeño
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=10, alpha=0.8)

plt.title("Visualización de Grafo con Coloración por Valor", size=16)
plt.axis('off') # Ocultar los ejes

# Crear una barra de color manual (Matplotlib no la genera automáticamente para colores de nodos individuales)
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([]) # Necesario para la barra de color
cbar = plt.colorbar(sm, ax=plt.gca(), orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label("Valor del Nodo")


plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Crear un grafo
G = nx.Graph()

# Nodo central
central_node = "Centro"
G.add_node(central_node)

# Crear nodos de primera ramificación y conectarlos al centro
branch_nodes_1 = ["Rama_1_1", "Rama_1_2", "Rama_1_3", "Rama_1_4", "Rama_1_5"]
for node in branch_nodes_1:
    G.add_node(node)
    G.add_edge(central_node, node)

# Crear nodos de segunda ramificación (las "gotas" finales)
# y conectarlos a los nodos de primera ramificación
drop_nodes = []
for i, branch_node in enumerate(branch_nodes_1):
    # Cada rama principal tiene 3 gotas
    for j in range(1, 4):
        drop_node = f"Gota_{i+1}_{j}"
        G.add_node(drop_node)
        G.add_edge(branch_node, drop_node)
        drop_nodes.append(drop_node)

# Definir los tamaños de los nodos
node_sizes = []
for node in G.nodes():
    if node == central_node:
        node_sizes.append(5000)  # Nodo central muy grande
    elif node in branch_nodes_1:
        node_sizes.append(1000)   # Nodos de rama intermedios
    else:
        node_sizes.append(300)    # Nodos finales ("gotas") más pequeños

# Definir el color (un tono de rojo/rosa similar a la imagen)
node_color = '#CC0052' # Un rojo frambuesa o magenta oscuro
edge_color = '#CC0052' # Mismo color para las conexiones

# Calcular las posiciones de los nodos
# 'spring_layout' tiende a separar los nodos de forma orgánica
pos = nx.spring_layout(G, k=0.8, iterations=100, seed=42) # Ajusta k y iterations para la distribución

# Dibujar el grafo
plt.figure(figsize=(10, 10)) # Aumenta el tamaño de la figura

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_color, alpha=0.9)
nx.draw_networkx_edges(G, pos, edge_color=edge_color, width=1.5, alpha=0.7)

# Opcional: Si deseas etiquetas para depuración, puedes agregarlas.
# Para el estilo de la imagen, probablemente no quieras etiquetas.
# nx.draw_networkx_labels(G, pos, font_size=8, font_color='black')

plt.title("Visualización Estilo 'Nocardia'")
plt.axis('off') # Oculta los ejes para un fondo limpio
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np # Necesario para los arrays de coordenadas

# Crear un grafo (el mismo código que proporcionaste)
G = nx.Graph()

# Nodo central
central_node = "Centro"
G.add_node(central_node)

# Crear nodos de primera ramificación y conectarlos al centro
branch_nodes_1 = ["Rama_1_1", "Rama_1_2", "Rama_1_3", "Rama_1_4", "Rama_1_5"]
for node in branch_nodes_1:
    G.add_node(node)
    G.add_edge(central_node, node)

# Crear nodos de segunda ramificación (las "gotas" finales)
# y conectarlos a los nodos de primera ramificación
drop_nodes = []
for i, branch_node in enumerate(branch_nodes_1):
    # Cada rama principal tiene 3 gotas
    for j in range(1, 4):
        drop_node = f"Gota_{i+1}_{j}"
        G.add_node(drop_node)
        G.add_edge(branch_node, drop_node)
        drop_nodes.append(drop_node)

# Definir los tamaños de los nodos
node_sizes = []
for node in G.nodes():
    if node == central_node:
        node_sizes.append(5000)
    elif node in branch_nodes_1:
        node_sizes.append(1000)
    else:
        node_sizes.append(300)

# Definir el color
node_color = '#CC0052'
edge_color = '#CC0052'

# --- **Aquí es donde se calculan las posiciones** ---
pos = nx.spring_layout(G, k=0.8, iterations=100, seed=42)

# --- **Imprimir las posiciones de cada nodo** ---
print("Posiciones de los nodos (x, y):")
for node, coordinates in pos.items():
    print(f"  Nodo '{node}': x={coordinates[0]:.4f}, y={coordinates[1]:.4f}")

# Dibujar el grafo (la parte de visualización sigue igual)
plt.figure(figsize=(10, 10))

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_color, alpha=0.9)
nx.draw_networkx_edges(G, pos, edge_color=edge_color, width=1.5, alpha=0.7)

plt.title("Visualización Estilo 'Nocardia'")
plt.axis('off')
plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random

# --- 1. Generación de un grafo más complejo y ramificado ---
G = nx.Graph()
central_node = "Centro"
G.add_node(central_node)

# Función recursiva para añadir ramificaciones con profundidad
def add_branches(graph, parent_node, current_depth, max_depth, branch_factor, node_counter, node_sizes_dict):
    """
    Añade ramificaciones recursivamente a un grafo.
    Modifica node_counter y node_sizes_dict en su lugar.
    """
    if current_depth >= max_depth:
        # Cuando alcanzamos la profundidad máxima, detenemos la recursión
        # y devolvemos el valor actual del contador.
        return node_counter[0] # Devolvemos el valor del contador, no la lista.

    # Calcula el número de ramas para este nodo, con variación aleatoria
    num_branches = random.randint(branch_factor - 1, branch_factor + 1)
    # Asegurarse de que no creamos 0 ramas si no queremos detenernos prematuramente
    num_branches = max(0, num_branches) # Asegura que el número de ramas sea al menos 0

    for _ in range(num_branches):
        node_counter[0] += 1 # Incrementa el contador de nodo
        new_node = f"N{node_counter[0]}" # Crea un nombre de nodo único
        graph.add_node(new_node) # Añade el nuevo nodo al grafo
        graph.add_edge(parent_node, new_node) # Conecta el nuevo nodo con el padre

        # Definir tamaño de nodo basado en profundidad (más profundo = más pequeño)
        if current_depth == 0: # Nodos principales de rama (conectados al centro)
            node_sizes_dict[new_node] = random.randint(800, 1500)
        elif current_depth == max_depth - 1: # Nodos finales ("gotas")
            node_sizes_dict[new_node] = random.randint(50, 250)
        else: # Nodos intermedios
            node_sizes_dict[new_node] = random.randint(200, 600)

        # Llamada recursiva para añadir ramas al nuevo nodo
        # No necesitamos asignar el resultado de vuelta a node_counter[0]
        # porque node_counter es una lista y se modifica en su lugar.
        add_branches(graph, new_node, current_depth + 1, max_depth, branch_factor, node_counter, node_sizes_dict)

    # Devolvemos el valor actual del contador después de añadir todas las ramas para este nivel
    return node_counter[0]


node_counter = [0] # Inicializamos el contador como una lista para pasarlo por referencia
node_sizes_dict = {central_node: 8000} # Tamaño del nodo central
max_depth = 4 # Más niveles de ramificación
branch_factor = 3 # Número promedio de ramas por nodo

# Llamada inicial a la función recursiva.
# La función modificará node_counter[0] y node_sizes_dict en su lugar.
# El valor devuelto por la llamada inicial (el valor final de node_counter) no es estrictamente necesario
# para la lógica siguiente, pero puede ser útil para saber cuántos nodos se crearon.
final_node_count = add_branches(G, central_node, 0, max_depth, branch_factor, node_counter, node_sizes_dict)

# Asignar tamaños de nodo a una lista para nx.draw
# Asegurarse de que cada nodo en G tenga un tamaño definido en node_sizes_dict
node_sizes = [node_sizes_dict.get(node, 100) for node in G.nodes()]

# --- 2. Posicionamiento y Dispersión ---
# Utiliza spring_layout, puedes ajustar k para dispersión y iterations para estabilidad
pos = nx.spring_layout(G, k=0.1, iterations=100, seed=42) # 'k' más bajo para nodos más cercanos, más 'iterations' para mejor convergencia

# --- 3. Colores y Efecto de "Brillo" (Matplotlib) ---
base_color = '#CC0052' # El color principal (rosa/magenta)
glow_color_inner = '#FF6699' # Un tono más claro para el brillo interior
glow_color_outer = '#FFCCE0' # Un tono aún más claro para el brillo exterior
edge_color = '#AA0044' # Un poco más oscuro para las aristas

plt.figure(figsize=(12, 12)) # Aumenta el tamaño de la figura

# Efecto de "brillo": Dibujar varias capas de nodos con diferentes tamaños y transparencias
# Capa más externa (más grande y transparente)
nx.draw_networkx_nodes(G, pos, node_size=[s * 1.5 for s in node_sizes], node_color=glow_color_outer, alpha=0.1, ax=plt.gca())
# Capa intermedia
nx.draw_networkx_nodes(G, pos, node_size=[s * 1.2 for s in node_sizes], node_color=glow_color_inner, alpha=0.3, ax=plt.gca())
# Capa base de nodos
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=base_color, alpha=0.9, ax=plt.gca())

# Dibujar las aristas
nx.draw_networkx_edges(G, pos, edge_color=edge_color, width=0.8, alpha=0.6, ax=plt.gca())

plt.title("Grafo Ramificado Complejo con Efecto de Brillo (Matplotlib)", color='white')
plt.gca().set_facecolor('black') # Fondo oscuro para resaltar el brillo
plt.axis('off')
plt.show()

In [ ]:
import multiprocessing
import time

def tarea(nombre):
    print(f"Iniciando tarea {nombre}...")
    time.sleep(2)  # Simula trabajo
    print(f"Tarea {nombre} finalizada.")

if __name__ == "__main__":
    p1 = multiprocessing.Process(target=tarea, args=("Uno",))
    p2 = multiprocessing.Process(target=tarea, args=("Dos",))

    p1.start() # Inicia el primer proceso
    p2.start() # Inicia el segundo proceso

    p1.join()  # Espera a que p1 termine
    p2.join()  # Espera a que p2 termine

    print("Todas las tareas han finalizado.")

## Neurona con historial (series temporales)

In [ ]:
# prompt: Datos sintéticos secuencias temporales

# 🧬 Auto-Optimización por Sacarina (Explicación Conceptual)
# La "Auto-Optimización por Sacarina" parece un nombre conceptual
# para una ida de RL donde el agente debe volverse más eficiente
# o "desprenderse" de partes no esenciales de sí mismo (su red neuronal)
# para resolver tareas, con el costo de cada acción siendo una reducción
# de sus propios recursos computacionales. La "Sacarina" podría referirse
# a una "recompensa" por volverse más ligero o eficiente.

# Este es un concepto de investigación y no hay una implementación estándar.
# Implementarlo requeriría:
# 1. Definir la estructura mutable de la red neuronal del agente.
# 2. Definir cómo una "acción" se traduce en una modificación de la red (ej. apagar nodos, podar pesos).
# 3. Definir la función de recompensa que balancea el desempeño en la tarea
#    con la eficiencia/reducción de la red.
# 4. Un algoritmo de RL que pueda operar sobre un espacio de estados que incluye
#    tanto el estado del entorno como el estado (estructura) de la red del agente,
#    y un espacio de acciones que incluye tanto acciones en el entorno como
#    acciones de auto-modificación.

# Esto es un área compleja y activa de investigación en meta-aprendizaje y RL con arquitecturas dinámicas.
# No hay un fragmento de código directo que implemente este concepto tal como se describe.
# Si quieres explorar algo similar, podrías investigar:
# - Network Pruning (Poda de Redes): Técnicas para reducir el tamaño de redes pre-entrenadas.
# - Differentiable Architecture Search (NAS): Buscar arquitecturas de red de forma automática.
# - Meta-RL: Entrenar agentes para aprender a aprender o adaptarse.
# - RL con Acción sobre la Propia Política/Arquitectura.

# El texto proporcionado es una descripción conceptual de una idea de RL avanzada.
# No es código ejecutable que pueda ser "corregido" o "completado" en un sentido tradicional.

# # Aquí terminan los comentarios y explicaciones sobre "Auto-Optimización por Sacarina"
# # El resto del código se mantiene como está, ya que no está directamente relacionado
# # con este concepto particular.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Datos sintéticos: función no lineal con ruido
torch.manual_seed(42)
N = 200
X = torch.linspace(-3, 3, N).unsqueeze(1)
y_true = torch.sin(X) + 0.1 * torch.randn(X.size())

# Neurona estándar para comparación
class NeuronaStandard(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

# Neurona con historial de errores
class NeuronaHistorial(nn.Module):
    def __init__(self, umbral=0.1):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(1))
        self.bias = nn.Parameter(torch.randn(1))
        self.historial = []
        self.umbral = umbral

    def forward(self, x):
        self.ultima_entrada = x.detach().clone()
        return x * self.weights + self.bias

    def registrar_error(self, y_real, y_pred):
        error = torch.abs(y_real - y_pred)
        mask = error > self.umbral
        for xi, ei in zip(self.ultima_entrada[mask], error[mask]):
            self.historial.append((xi.detach().clone(), ei.item()))

    def correccion_personalizada(self, lr=0.01):
        for entrada, error in self.historial[-10:]:  # últimos 10 errores
            grad = entrada * error
            self.weights.data -= lr * grad
            self.bias.data -= lr * error

# Entrenamiento paralelo de ambas neuronas
def entrenar(neurona, X, y, epocas=1000, lr=0.01, usar_historial=False):
    opt = torch.optim.SGD(neurona.parameters(), lr=lr)
    perdidas = []
    for _ in range(epocas):
        y_pred = neurona(X)
        loss = F.mse_loss(y_pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        perdidas.append(loss.item())
        if usar_historial:
            neurona.registrar_error(y, y_pred)
            neurona.correccion_personalizada(lr=lr * 0.1)
    return perdidas, neurona(X).detach()

# Instanciar neuronas
neurona_std = NeuronaStandard()
neurona_hist = NeuronaHistorial()

# Entrenamiento
loss_std, y_pred_std = entrenar(neurona_std, X, y_true, usar_historial=False)
loss_hist, y_pred_hist = entrenar(neurona_hist, X, y_true, usar_historial=True)

# Guardar datos para graficar
X_np = X.squeeze().numpy()
y_np = y_true.squeeze().numpy()
y_pred_std_np = y_pred_std.squeeze().numpy()
y_pred_hist_np = y_pred_hist.squeeze().numpy()
loss_std_np = np.array(loss_std)
loss_hist_np = np.array(loss_hist)

X_np, y_np, y_pred_std_np, y_pred_hist_np, loss_std_np, loss_hist_np

In [ ]:
# prompt: import numpy as np
# import tensorflow as tf # O PyTorch
# # 1. Preparar el Dataset (Ejemplo: secuencia simple)
# # Necesitarías cargar tu dataset de secuencia temporal real aquí
# X_train, y_train, X_test, y_test
# # 2. Modelo con "Neurona Común" (SGD básico)
# model_sgd = tf.keras.Sequential([
#     # Aquí puedes poner una o más capas de neuronas.
#     # Para una "sola neurona" de salida, sería Dense(1)
#     tf.keras.layers.Dense(10, activation='relu', input_shape=(num_caracteristicas_secuencia,)),
#     tf.keras.layers.Dense(1)
# ])
# model_sgd.compile(optimizer='sgd', loss='mse')
# model_sgd.fit(X_train, y_train, epochs=...)
# evaluation_sgd = model_sgd.evaluate(X_test, y_test)
# # 3. Modelo con "Historial Mejorado" (Optimizador Adam)
# model_adam = tf.keras.Sequential([
#     tf.keras.layers.Dense(10, activation='relu', input_shape=(num_caracteristicas_secuencia,)),
#     tf.keras.layers.Dense(1)
# ])
# model_adam.compile(optimizer='adam', loss='mse') # Adam usa un historial de gradientes
# model_adam.fit(X_train, y_train, epochs=...)
# evaluation_adam = model_adam.evaluate(X_test, y_test)
# # 4. Comparar resultados
# print(f"Error con SGD: {evaluation_sgd}")
# print(f"Error con Adam: {evaluation_adam}")

import matplotlib.pyplot as plt
# Visualización de los resultados
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(X_np, y_np, 'o', label='Datos reales')
plt.plot(X_np, y_pred_std_np, label='Neurona estándar (SGD)')
plt.plot(X_np, y_pred_hist_np, label='Neurona con historial (SGD + Corrección)')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Ajuste a datos no lineales')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(loss_std_np, label='Pérdida - Neurona estándar')
plt.plot(loss_hist_np, label='Pérdida - Neurona con historial')
plt.xlabel('Época')
plt.ylabel('Pérdida (MSE)')
plt.title('Curva de pérdida durante el entrenamiento')
plt.yscale('log') # Usar escala logarítmica para ver mejor las diferencias
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Opcional: Mostrar historial de errores de la neurona con historial
print("\nEjemplo del historial de errores (entrada, error):")
print(neurona_hist.historial[:10]) # Mostrar los primeros 10 registros
print("...")
print(neurona_hist.historial[-10:]) # Mostrar los últimos 10 registros

In [ ]:
# prompt: Agregar datos sintéticos

# Agregar datos sintéticos
import numpy as np
import matplotlib.pyplot as plt

# Número de puntos de datos
num_synthetic_points = 500

# Generar ruido aleatorio
# Usamos la misma distribución de X y y para que los datos sintéticos se parezcan a los datos de entrenamiento.
# En este caso, generamos ruido similar al que se añadió a y_true.
# Esto no es estrictamente "sintetizar datos" en el sentido de GANs, sino generar más puntos
# que sigan la misma tendencia general con ruido.
# Si quisieras sintetizar datos más complejos, usarías el Generador de tu GAN entrenada.

# Si tienes un modelo generador entrenado, podrías usarlo así:
# generated_noise = tf.random.normal([num_synthetic_points, LATENT_DIM])
# synthetic_images = generator(generated_noise, training=False) # Esto te daría imágenes

# Como no tenemos una implementación de GAN lista para usar aquí en un ejemplo simple,
# vamos a generar datos sintéticos basándonos en la distribución de tus datos existentes (X, y_true).
# Esto es más para visualización o aumento de datos simple.

# Generar más puntos de X dentro del mismo rango
synthetic_X = torch.linspace(-3, 3, num_synthetic_points).unsqueeze(1)

# Generar valores de y_true sintéticos basados en la función subyacente (sin(X)) más ruido
synthetic_y_true = torch.sin(synthetic_X) + 0.1 * torch.randn(synthetic_X.size())

# Convertir a numpy para visualización si es necesario
synthetic_X_np = synthetic_X.squeeze().numpy()
synthetic_y_true_np = synthetic_y_true.squeeze().numpy()

print(f"Generados {num_synthetic_points} puntos de datos sintéticos.")

# Opcional: Visualizar los datos sintéticos junto con los originales
plt.figure(figsize=(8, 6))
plt.plot(X_np, y_np, 'o', label='Datos Originales', alpha=0.5)
plt.plot(synthetic_X_np, synthetic_y_true_np, 'x', label='Datos Sintéticos', alpha=0.5)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos Originales vs. Datos Sintéticos (Generación Simple)')
plt.legend()
plt.grid(True)
plt.show()

# Puedes combinar estos datos sintéticos con tus datos originales si quieres aumentar el dataset.
# combined_X = torch.cat((X, synthetic_X), dim=0)
# combined_y = torch.cat((y_true, synthetic_y_true), dim=0)
# print(f"Tamaño del dataset combinado: {len(combined_X)} puntos.")

In [ ]:
# prompt: Agregar datos sintéticos
# import matplotlib.pyplot as plt
# # Visualización de los resultados
# plt.figure(figsize=(12, 6))
# plt.subplot(1, 2, 1)
# plt.plot(X_np, y_np, 'o', label='Datos reales')
# plt.plot(X_np, y_pred_std_np, label='Neurona estándar (SGD)')
# plt.plot(X_np, y_pred_hist_np, label='Neurona con historial (SGD + Corrección)')
# plt.xlabel('X')
# plt.ylabel('y')
# plt.title('Ajuste a datos no lineales')
# plt.legend()
# plt.grid(True)
# plt.subplot(1, 2, 2)
# plt.plot(loss_std_np, label='Pérdida - Neurona estándar')
# plt.plot(loss_hist_np, label='Pérdida - Neurona con historial')
# plt.xlabel('Época')
# plt.ylabel('Pérdida (MSE)')
# plt.title('Curva de pérdida durante el entrenamiento')
# plt.yscale('log') # Usar escala logarítmica para ver mejor las diferencias
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.show()
# # Opcional: Mostrar historial de errores de la neurona con historial
# print("\nEjemplo del historial de errores (entrada, error):")
# print(neurona_hist.historial[:10]) # Mostrar los primeros 10 registros
# print("...")
# print(neurona_hist.historial[-10:]) # Mostrar los últimos 10 registros

import matplotlib.pyplot as plt
# Agregar datos sintéticos

# Número de puntos de datos
num_synthetic_points = 500

# Generar ruido aleatorio
# Usamos la misma distribución de X y y para que los datos sintéticos se parezcan a los datos de entrenamiento.
# En este caso, generamos ruido similar al que se añadió a y_true.
# Esto no es estrictamente "sintetizar datos" en el sentido de GANs, sino generar más puntos
# que sigan la misma tendencia general con ruido.
# Si quisieras sintetizar datos más complejos, usarías el Generador de tu GAN entrenada.

# Si tienes un modelo generador entrenado, podrías usarlo así:
# generated_noise = tf.random.normal([num_synthetic_points, LATENT_DIM])
# synthetic_images = generator(generated_noise, training=False) # Esto te daría imágenes

# Como no tenemos una implementación de GAN lista para usar aquí en un ejemplo simple,
# vamos a generar datos sintéticos basándonos en la distribución de tus datos existentes (X, y_true).
# Esto es más para visualización o aumento de datos simple.

# Generar más puntos de X dentro del mismo rango
synthetic_X = torch.linspace(-3, 3, num_synthetic_points).unsqueeze(1)

# Generar valores de y_true sintéticos basados en la función subyacente (sin(X)) más ruido
synthetic_y_true = torch.sin(synthetic_X) + 0.1 * torch.randn(synthetic_X.size())

# Convertir a numpy para visualización si es necesario
synthetic_X_np = synthetic_X.squeeze().numpy()
synthetic_y_true_np = synthetic_y_true.squeeze().numpy()

print(f"Generados {num_synthetic_points} puntos de datos sintéticos.")

# Opcional: Visualizar los datos sintéticos junto con los originales
plt.figure(figsize=(8, 6))
plt.plot(X_np, y_np, 'o', label='Datos Originales', alpha=0.5)
plt.plot(synthetic_X_np, synthetic_y_true_np, 'x', label='Datos Sintéticos', alpha=0.5)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos Originales vs. Datos Sintéticos (Generación Simple)')
plt.legend()
plt.grid(True)
plt.show()

# Puedes combinar estos datos sintéticos con tus datos originales si quieres aumentar el dataset.
# combined_X = torch.cat((X, synthetic_X), dim=0)
# combined_y = torch.cat((y_true, synthetic_y_true), dim=0)
# print(f"Tamaño del dataset combinado: {len(combined_X)} puntos.")

In [ ]:
# prompt: Agreg datos sintéticos

# Datos sintéticos: función no lineal con ruido
torch.manual_seed(42)
N = 2000
X = torch.linspace(-3, 3, N).unsqueeze(1)
y_true = torch.sin(X) + 0.1 * torch.randn(X.size())

# Guardar datos para graficar
X_np = X.squeeze().numpy()
y_np = y_true.squeeze().numpy()

# prompt: Graficar # prompt: Agreg datos sintéticos
# # Datos sintéticos: función no lineal con ruido
# torch.manual_seed(42)
# N = 200
# X = torch.linspace(-3, 3, N).unsqueeze(1)
# y_true = torch.sin(X) + 0.1 * torch.randn(X.size())
# # Guardar datos para graficar
# X_np = X.squeeze().numpy()
# y_np = y_true.squeeze().numpy()

import matplotlib.pyplot as plt
import numpy as np
# Visualización de los datos sintéticos
plt.figure(figsize=(8, 6))
plt.plot(X_np, y_np, 'o', markersize=4, label='Datos con ruido')
plt.plot(X_np, np.sin(X_np), label='Función real ($y = \sin(x)$)')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos Sintéticos: Función Senoidal con Ruido')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# prompt: Agrega datos sintéticos import numpy as np
# import tensorflow as tf # O PyTorch
# # 1. Preparar el Dataset (Ejemplo: secuencia simple)
# # Necesitarías cargar tu dataset de secuencia temporal real aquí
# X_train, y_train, X_test, y_test
# # 2. Modelo con "Neurona Común" (SGD básico)
# model_sgd = tf.keras.Sequential([
#     # Aquí puedes poner una o más capas de neuronas.
#     # Para una "sola neurona" de salida, sería Dense(1)
#     tf.keras.layers.Dense(10, activation='relu', input_shape=(num_caracteristicas_secuencia,)),
#     tf.keras.layers.Dense(1)
# ])
# model_sgd.compile(optimizer='sgd', loss='mse')
# model_sgd.fit(X_train, y_train, epochs=...)
# evaluation_sgd = model_sgd.evaluate(X_test, y_test)
# # 3. Modelo con "Historial Mejorado" (Optimizador Adam)
# model_adam = tf.keras.Sequential([
#     tf.keras.layers.Dense(10, activation='relu', input_shape=(num_caracteristicas_secuencia,)),
#     tf.keras.layers.Dense(1)
# ])
# model_adam.compile(optimizer='adam', loss='mse') # Adam usa un historial de gradientes
# model_adam.fit(X_train, y_train, epochs=...)
# evaluation_adam = model_adam.evaluate(X_test, y_test)
# # 4. Comparar resultados
# print(f"Error con SGD: {evaluation_sgd}")
# print(f"Error con Adam: {evaluation_adam}")

import numpy as np
# Generar datos sintéticos de ejemplo para secuencia temporal
# Reemplaza esto con la carga y preprocesamiento de tus datos reales
num_samples = 1000
sequence_length = 10
num_features_sequence = 5

# Datos de entrada (secuencias)
X_train = np.random.rand(num_samples, sequence_length, num_features_sequence)
X_test = np.random.rand(num_samples // 4, sequence_length, num_features_sequence) # Usar menos datos para test

# Datos de salida (valor futuro)
# Aquí se simula una relación simple, por ejemplo, la suma de los últimos 3 valores de la última característica
y_train = np.sum(X_train[:, -3:, -1], axis=1) + np.random.randn(num_samples) * 0.1
y_test = np.sum(X_test[:, -3:, -1], axis=1) + np.random.randn(num_samples // 4) * 0.1

# Reshape los datos para las capas Dense si no estás usando capas recurrentes (LSTM, GRU)
# Las capas Dense esperan una entrada 2D (samples, features)
# Aquí aplanamos la secuencia: (batch_size, sequence_length * num_features_sequence)
X_train_flat = X_train.reshape(num_samples, sequence_length * num_features_sequence)
X_test_flat = X_test.reshape(num_samples // 4, sequence_length * num_features_sequence)

print(f"X_train_flat shape: {X_train_flat.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test_flat shape: {X_test_flat.shape}")
print(f"y_test shape: {y_test.shape}")

# Ahora puedes usar X_train_flat y X_test_flat en tus modelos Keras con capas Dense.
# Asegúrate de que la capa de entrada tenga input_shape=(sequence_length * num_features_sequence,)
# Por ejemplo:
tf.keras.layers.Dense(10, activation='relu', input_shape=(sequence_length * num_features_sequence,))

In [ ]:
# prompt: Un generador de datos sintéticos, par a modelos , tipo de datos: secuencia temporal,  cambiante en el tiempo

import matplotlib.pyplot as plt
import numpy as np
class TimeSeriesGenerator:
    def __init__(self, num_samples, sequence_length, num_features, changing_magnitude=0.1, trend_magnitude=0.01):
        """
        Generador de datos sintéticos de secuencia temporal con tendencias cambiantes.

        Args:
            num_samples (int): Número de secuencias a generar.
            sequence_length (int): Longitud de cada secuencia.
            num_features (int): Número de características por paso de tiempo.
            changing_magnitude (float): Magnitud de la variabilidad en el tiempo.
            trend_magnitude (float): Magnitud de la tendencia general.
        """
        self.num_samples = num_samples
        self.sequence_length = sequence_length
        self.num_features = num_features
        self.changing_magnitude = changing_magnitude
        self.trend_magnitude = trend_magnitude

    def generate(self):
        """
        Genera un dataset de secuencias temporales sintéticas.

        Returns:
            tuple: Tupla de arrays numpy (X, y).
                   X: (num_samples, sequence_length, num_features)
                   y: (num_samples,) - valor objetivo asociado a cada secuencia.
        """
        X = np.zeros((self.num_samples, self.sequence_length, self.num_features))
        y = np.zeros(self.num_samples)

        for i in range(self.num_samples):
            # Generar una base aleatoria para la secuencia
            base_values = np.random.randn(self.num_features) * 5

            # Generar una tendencia aleatoria por característica
            trends = np.random.randn(self.num_features) * self.trend_magnitude

            # Generar variabilidad a lo largo de la secuencia
            time_variation = np.random.randn(self.sequence_length, self.num_features) * self.changing_magnitude

            # Construir la secuencia
            sequence = np.zeros((self.sequence_length, self.num_features))
            for t in range(self.sequence_length):
                # Aplicar base, tendencia y variación en el tiempo
                sequence[t, :] = base_values + trends * t + time_variation[t, :]
                # Añadir un poco de ruido extra por paso de tiempo
                sequence[t, :] += np.random.randn(self.num_features) * 0.05

            X[i] = sequence

            # Generar un valor objetivo 'y' basado en la secuencia
            # Ejemplo: promedio de los últimos 3 pasos de la primera característica + ruido
            y[i] = np.mean(sequence[-3:, 0]) + np.random.randn() * 0.1

        return X, y

# Ejemplo de uso del generador
num_samples_synth = 2000
sequence_length_synth = 50
num_features_synth = 10

synth_generator = TimeSeriesGenerator(
    num_samples=num_samples_synth,
    sequence_length=sequence_length_synth,
    num_features=num_features_synth,
    changing_magnitude=0.2,  # Ajusta para más o menos variabilidad en el tiempo
    trend_magnitude=0.02     # Ajusta para tendencias más o menos pronunciadas
)

X_synth, y_synth = synth_generator.generate()

print(f"Dataset sintético generado:")
print(f"  X_synth shape: {X_synth.shape}")
print(f"  y_synth shape: {y_synth.shape}")

# Opcional: Visualizar algunas secuencias de ejemplo
plt.figure(figsize=(12, 6))
for i in range(min(5, num_samples_synth)): # Mostrar las primeras 5 secuencias
    plt.plot(X_synth[i, :, 0], label=f'Secuencia {i+1} (Característica 1)') # Mostrar solo la primera característica
plt.xlabel("Paso de tiempo")
plt.ylabel("Valor")
plt.title("Ejemplo de Secuencias Temporales Sintéticas (Característica 1)")
plt.legend()
plt.grid(True)
plt.show()

# Opcional: Visualizar la distribución de los valores objetivo
plt.figure(figsize=(8, 4))
plt.hist(y_synth, bins=50)
plt.xlabel("Valor objetivo (y)")
plt.ylabel("Frecuencia")
plt.title("Distribución de los Valores Objetivo Sintéticos")
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

class TimeSeriesGenerator:
    def __init__(self, num_samples, sequence_length, num_features, changing_magnitude=0.1, trend_magnitude=0.01):
        self.num_samples = num_samples
        self.sequence_length = sequence_length
        self.num_features = num_features
        self.changing_magnitude = changing_magnitude
        self.trend_magnitude = trend_magnitude

    def generate(self):
        X = np.zeros((self.num_samples, self.sequence_length, self.num_features))
        y = np.zeros(self.num_samples)

        for i in range(self.num_samples):
            base_values = np.random.randn(self.num_features) * 5
            trends = np.random.randn(self.num_features) * self.trend_magnitude
            time_variation = np.random.randn(self.sequence_length, self.num_features) * self.changing_magnitude

            sequence = np.zeros((self.sequence_length, self.num_features))
            for t in range(self.sequence_length):
                sequence[t, :] = base_values + trends * t + time_variation[t, :]
                sequence[t, :] += np.random.randn(self.num_features) * 0.05

            X[i] = sequence
            y[i] = np.mean(sequence[-3:, 0]) + np.random.randn() * 0.1

        return X, y

# Parámetros del generador
num_samples_synth = 2000
sequence_length_synth = 50
num_features_synth = 10

synth_generator = TimeSeriesGenerator(
    num_samples=num_samples_synth,
    sequence_length=sequence_length_synth,
    num_features=num_features_synth,
    changing_magnitude=0.2,
    trend_magnitude=0.02
)

X_synth, y_synth = synth_generator.generate()

# Visualizar algunas secuencias de ejemplo
plt.figure(figsize=(12, 6))
for i in range(5):
    plt.plot(X_synth[i, :, 0], label=f'Secuencia {i+1}')
plt.xlabel("Paso de tiempo")
plt.ylabel("Valor")
plt.title("Ejemplo de Secuencias Temporales Sintéticas (Característica 1)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Visualizar la distribución de los valores objetivo
plt.figure(figsize=(8, 4))
plt.hist(y_synth, bins=50)
plt.xlabel("Valor objetivo (y)")
plt.ylabel("Frecuencia")
plt.title("Distribución de los Valores Objetivo Sintéticos")
plt.grid(True)
plt.tight_layout()
plt.show()

##Aumento de Ruido de Modelo Dinámico (Dynamic Model Noise Augmentation - DMNA)".

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- NUEVO MODELO: Red con Aumento de Ruido de Modelo Dinámico (DMNA) ---
class DMNANet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

        # Parámetros para controlar el ruido dinámico
        self.current_loss = 0.0 # Se actualizará en el ciclo de entrenamiento
        self.noise_intensity_factor = 0.5 # Factor base de intensidad del ruido
        self.noise_decay_rate = 0.999 # Tasa de decaimiento del ruido por época
        self.min_noise_intensity = 0.01 # Mínima intensidad de ruido residual
        self.noise_threshold = 0.1 # Pérdida mínima para aplicar ruido significativo

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        # Aplicar Ruido Dinámico a la activación de la primera capa
        if self.training and self.current_loss > self.noise_threshold:
            # Magnitud del ruido basada en la pérdida actual y un factor de decaimiento
            noise_magnitude = max(self.min_noise_intensity,
                                  self.noise_intensity_factor * (self.current_loss ** 0.5))

            # Ruido Estructurado: combinación de ruido gaussiano y un "empuje" hacia cero para activaciones altas
            # Fomenta la no-saturación y la exploración
            gaussian_noise = torch.randn_like(h1) * noise_magnitude

            # Un pequeño empuje hacia cero para activaciones muy grandes (simula "fatiga" o "regularización suave")
            # Esto evita que las activaciones se vuelvan excesivamente grandes y saturen
            fatigue_noise = -torch.sigmoid(h1) * noise_magnitude * 0.5

            h1 = h1 + gaussian_noise + fatigue_noise

        h2 = F.relu(self.fc2(h1))
        # Aplicar Ruido Dinámico a la activación de la segunda capa también
        if self.training and self.current_loss > self.noise_threshold:
            noise_magnitude = max(self.min_noise_intensity,
                                  self.noise_intensity_factor * (self.current_loss ** 0.5))
            gaussian_noise = torch.randn_like(h2) * noise_magnitude
            fatigue_noise = -torch.sigmoid(h2) * noise_magnitude * 0.5

            h2 = h2 + gaussian_noise + fatigue_noise

        out = self.output_layer(h2)
        return out

    def update_noise_params(self, loss_value):
        # Este método se llama en cada época para actualizar la pérdida actual
        # y reducir la intensidad base del ruido.
        self.current_loss = loss_value
        # Decaimiento del factor base del ruido con el tiempo/épocas
        self.noise_intensity_factor *= self.noise_decay_rate
        self.noise_intensity_factor = max(self.min_noise_intensity, self.noise_intensity_factor)


# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.01
EPOCHS = 2000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red con DMNA ---")
dmna_model = DMNANet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_dmna = torch.optim.Adam(dmna_model.parameters(), lr=LEARNING_RATE)
historial_loss_dmna = []

total_params_dmna = sum(p.numel() for p in dmna_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red DMNA: {total_params_dmna}")

for epoch in range(EPOCHS):
    dmna_model.train()
    optimizer_dmna.zero_grad()

    y_pred = dmna_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_dmna.append(loss.item())

    loss.backward()
    optimizer_dmna.step()

    # Actualizar los parámetros de ruido dinámico del modelo
    dmna_model.update_noise_params(loss.item())

    if epoch % 100 == 0:
        print(f"Red DMNA - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red DMNA): {historial_loss_dmna[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar (ReLU) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

dmna_model.eval() # Importante para desactivar el ruido durante la evaluación
standard_model.eval()

with torch.no_grad():
    y_hat_dmna = dmna_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_dmna, label="Predicción Red DMNA", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_dmna, label="Pérdida Red DMNA", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red DMNA): {historial_loss_dmna[-1]:.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")

###Red con Optimización por Mutación Global (GMO)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import copy # Para copiar el estado del modelo

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- NUEVO MODELO: Red con Optimización por Mutación Global (GMO) ---
class GMONet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

        # Parámetros para la mutación global
        self.mutation_strength = 0.05 # Cuánto se perturban los pesos
        self.mutation_probability = 0.05 # Probabilidad de que ocurra una mutación en una época
        self.best_loss_so_far = float('inf') # Para comparar y aceptar mutaciones
        self.current_loss = float('inf') # La pérdida de la época actual

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

    def apply_global_mutation(self):
        # Este método aplica una mutación aleatoria a TODOS los pesos del modelo
        # y evalúa si es una mejora. Si lo es, la acepta; si no, la descarta.

        # Guardar el estado actual del modelo
        original_state_dict = copy.deepcopy(self.state_dict())

        # Aplicar perturbación aleatoria a todos los parámetros
        with torch.no_grad(): # No queremos calcular gradientes para esto
            for param in self.parameters():
                param.add_(torch.randn_like(param) * self.mutation_strength)

        # Evaluar el modelo mutado
        self.eval() # Poner en modo evaluación para la prueba
        with torch.no_grad():
            mutated_y_pred = self(X) # Usar el dataset completo para evaluar
            mutated_loss = F.mse_loss(mutated_y_pred, y).item()

        # Decidir si aceptar la mutación
        # Condición: la pérdida mutada es significativamente mejor
        if mutated_loss < self.best_loss_so_far * 0.99: # 1% de mejora significativa
            self.best_loss_so_far = mutated_loss
            print(f"MUTACIÓN ACEPTADA: Pérdida mejorada de {self.current_loss:.6f} a {mutated_loss:.6f}")
            self.current_loss = mutated_loss # Actualizar la pérdida actual
            # Los pesos mutados ya están en el modelo, no se necesita hacer nada más
        else:
            # Revertir a los pesos originales si la mutación no fue mejor
            self.load_state_dict(original_state_dict)
            # print(f"MUTACIÓN RECHAZADA: Pérdida {mutated_loss:.6f} vs {self.current_loss:.6f}")

        self.train() # Volver a modo entrenamiento

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.01
EPOCHS = 2000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red con GMO (Mutación Global) ---")
gmo_model = GMONet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_gmo = torch.optim.Adam(gmo_model.parameters(), lr=LEARNING_RATE)
historial_loss_gmo = []

total_params_gmo = sum(p.numel() for p in gmo_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red GMO: {total_params_gmo}")

for epoch in range(EPOCHS):
    gmo_model.train()
    optimizer_gmo.zero_grad()

    # Primero, intentamos una mutación si la probabilidad lo permite
    if random.random() < gmo_model.mutation_probability:
        gmo_model.apply_global_mutation()
        # Después de una mutación (aceptada o rechazada), reevaluar la pérdida actual
        # Esto es importante para el registro de historial
        with torch.no_grad():
            gmo_model.current_loss = LOSS_FN(gmo_model(X), y).item()

    # Luego, el entrenamiento normal con descenso de gradiente
    y_pred = gmo_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_gmo.append(loss.item())

    loss.backward()
    optimizer_gmo.step()

    # Actualizar la pérdida actual del modelo para el próximo ciclo de mutación
    gmo_model.current_loss = loss.item() # Mantenerla actualizada incluso si no hubo mutación

    if epoch % 100 == 0:
        print(f"Red GMO - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red GMO): {historial_loss_gmo[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar (ReLU) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

gmo_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_gmo = gmo_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_gmo, label="Predicción Red GMO", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_gmo, label="Pérdida Red GMO", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red GMO): {historial_loss_gmo[-1]:.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")

##Modelo de Destilación de Conceptos (CDM)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X_noisy = torch.linspace(-3, 3, 300).unsqueeze(1) # X con posible ruido para simular datos reales
# La 'verdadera' X sin ruido, para el entrenamiento del extractor de conceptos
X_true = torch.linspace(-3, 3, 300).unsqueeze(1)
y_noisy = X_true.pow(3) - X_true + 0.5 * torch.randn(X_true.size()) # Y con ruido

# --- NUEVO MODELO: Red con Destilación de Conceptos (CDM) ---
class ConceptExtractor(nn.Module):
    # Este módulo intenta "limpiar" la entrada X.
    # En un caso real, podría ser un autoencoder o algo más complejo.
    # Aquí, simplemente intenta aprender una transformación para eliminar ruido de X.
    def __init__(self, input_size, output_size):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, x):
        # Una transformación lineal simple, podría ser más compleja.
        # La idea es que aprenda a mapear X_noisy a X_true durante el entrenamiento.
        return self.fc(x)

class ConceptLearner(nn.Module):
    # Este es tu modelo principal, que aprende de la X "limpia" del ConceptExtractor.
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x_clean): # Toma la X "limpia" como entrada
        h1 = F.relu(self.fc1(x_clean))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

class CDMNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.concept_extractor = ConceptExtractor(input_size, input_size) # Predice X_limpia
        self.concept_learner = ConceptLearner(input_size, hidden_size, output_size)

    def forward(self, x_noisy):
        # Primero, extrae el concepto "limpio" de la entrada ruidosa
        x_clean = self.concept_extractor(x_noisy)
        # Luego, el modelo principal aprende de esta X "limpia"
        output = self.concept_learner(x_clean)
        return output, x_clean # Devolvemos x_clean para la pérdida del extractor

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.001
EPOCHS = 20000
LOSS_FN_REGRESSION = nn.MSELoss() # Para la tarea principal
LOSS_FN_EXTRACTOR = nn.MSELoss()  # Para la tarea de limpieza del extractor

INPUT_SIZE = X_noisy.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y_noisy.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red con CDM (Destilación de Conceptos) ---")
cdm_model = CDMNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
# Optimizador para todos los parámetros de ambos submódulos
optimizer_cdm = torch.optim.Adam(cdm_model.parameters(), lr=LEARNING_RATE)
historial_loss_cdm = []

total_params_cdm = sum(p.numel() for p in cdm_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red CDM: {total_params_cdm}")

for epoch in range(EPOCHS):
    cdm_model.train()
    optimizer_cdm.zero_grad()

    y_pred, x_clean_predicted = cdm_model(X_noisy)

    # Pérdida para la tarea principal de regresión
    main_loss = LOSS_FN_REGRESSION(y_pred, y_noisy)

    # Pérdida para el extractor de conceptos:
    # Intenta que el extractor prediga la X "verdadera" sin ruido.
    # En un escenario real sin X_true, podrías usar un autoencoder (X -> X_reconstruida)
    # o técnicas de denoising.
    extractor_loss = LOSS_FN_EXTRACTOR(x_clean_predicted, X_true)

    # Combinar las pérdidas (puedes ponderarlas si quieres)
    # Un balance inicial 1:1. El extractor necesita aprender a limpiar.
    total_loss = main_loss + extractor_loss

    historial_loss_cdm.append(total_loss.item())

    total_loss.backward()
    optimizer_cdm.step()

    if epoch % 100 == 0:
        print(f"Red CDM - Época {epoch} - Pérdida Total: {total_loss.item():.6f} (Principal: {main_loss.item():.6f}, Extractor: {extractor_loss.item():.6f})")

print(f"Pérdida final (Red CDM): {historial_loss_cdm[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar (ReLU) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X_noisy) # La red estándar aprende directamente de X_noisy
    loss = LOSS_FN_REGRESSION(y_pred, y_noisy)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

cdm_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_cdm, _ = cdm_model(X_noisy) # El segundo retorno es x_clean_predicted, no lo necesitamos para el plot
    y_hat_cdm = y_hat_cdm.numpy()
    y_hat_standard = standard_model(X_noisy).numpy()

X_np = X_noisy.numpy()
y_np = y_noisy.numpy() # Los datos ruidosos para el plot

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, 'o', label="Valores Reales (Ruidosos)", color='gray', alpha=0.3, markersize=3)
plt.plot(X_true.numpy(), X_true.pow(3).numpy() - X_true.numpy(), label="Curva Verdadera (Sin Ruido)", color='blue', linewidth=2)
plt.plot(X_np, y_hat_cdm, label="Predicción Red CDM", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_cdm, label="Pérdida Red CDM (Total)", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red CDM): {historial_loss_cdm[-1]:.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")

#Red con Sesgo de Memoria Residual (RMB)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- NUEVO MODELO: Capa con Sesgo de Memoria Residual (RMB) ---
class RMBLayer(nn.Module):
    def __init__(self, input_features, output_features, memory_decay=0.9):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features
        self.memory_decay = memory_decay # Factor de decaimiento de la memoria

        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)

        # El "sesgo" inicial, que será influenciado por la memoria
        self.base_bias = nn.Parameter(torch.randn(output_features) * 0.1)

        # La "memoria residual": un estado interno que se actualiza
        # No es un nn.Parameter porque lo actualizaremos manualmente en forward
        self.register_buffer('residual_memory', torch.zeros(output_features))

        # Transformación no lineal para la memoria
        self.memory_transform = nn.Linear(output_features, output_features)
        # Inicializar para que al principio no cambie mucho
        nn.init.zeros_(self.memory_transform.weight)
        nn.init.zeros_(self.memory_transform.bias)

    def forward(self, x):
        linear_output = torch.matmul(x, self.weight)

        # --- Cálculo del Sesgo de Memoria Residual ---
        # 1. Transformar la memoria residual actual de forma no lineal
        transformed_memory_bias = self.memory_transform(self.residual_memory.unsqueeze(0)).squeeze(0)

        # 2. Combinar el sesgo base con el sesgo de memoria
        # El sesgo final es una suma del sesgo base aprendible y el sesgo dinámico de memoria.
        effective_bias = self.base_bias + transformed_memory_bias

        # Aplicar el sesgo al output lineal (expandirlo para el batch)
        output_with_bias = linear_output + effective_bias.unsqueeze(0)

        # 3. Actualizar la memoria residual para la próxima pasada
        # La memoria es un promedio móvil de las activaciones de salida de esta capa.
        # Esto hace que el "sesgo" sea una función de sus propias salidas pasadas.
        with torch.no_grad(): # La actualización de la memoria no debe tener gradientes
            # La memoria actual es la media de la salida actual, mezclada con la memoria antigua
            current_layer_output_mean = output_with_bias.mean(dim=0)
            self.residual_memory.data = (
                self.memory_decay * self.residual_memory.data +
                (1 - self.memory_decay) * current_layer_output_mean.data
            )

        return F.relu(output_with_bias) # Aplicar activación como de costumbre

# --- Red Neuronal usando las Capas RMB ---
class RMBNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rmb_layer1 = RMBLayer(input_size, hidden_size)
        self.rmb_layer2 = RMBLayer(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = self.rmb_layer1(x)
        h2 = self.rmb_layer2(h1)
        out = self.output_layer(h2)
        return out

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.01
EPOCHS = 2000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red con Sesgo de Memoria Residual (RMB) ---")
rmb_model = RMBNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_rmb = torch.optim.Adam(rmb_model.parameters(), lr=LEARNING_RATE)
historial_loss_rmb = []

total_params_rmb = sum(p.numel() for p in rmb_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red RMB: {total_params_rmb}")

for epoch in range(EPOCHS):
    rmb_model.train()
    optimizer_rmb.zero_grad()

    y_pred = rmb_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_rmb.append(loss.item())

    loss.backward()
    optimizer_rmb.step()

    if epoch % 100 == 0:
        print(f"Red RMB - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red RMB): {historial_loss_rmb[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar (ReLU) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

rmb_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_rmb = rmb_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, 'o', label="Valores Reales (Ruidosos)", color='gray', alpha=0.3, markersize=3)
plt.plot(X.numpy(), X.pow(3).numpy() - X.numpy(), label="Curva Verdadera (Sin Ruido)", color='blue', linewidth=2)
plt.plot(X_np, y_hat_rmb, label="Predicción Red RMB", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_rmb, label="Pérdida Red RMB", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red RMB): {historial_loss_rmb[-1]:.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")

##Un Modelo de Varianza Experimental: La "Red de Caos Controlado (Controlled Chaos Network - CCN)"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- NUEVA FUNCIÓN DE ACTIVACIÓN: HyperSensitiveReLU ---
class HyperSensitiveReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, amplification_factor, activation_threshold):
        ctx.save_for_backward(input, torch.tensor(amplification_factor), torch.tensor(activation_threshold))

        activated_output = input.clone()

        sensitive_region = (activated_output > activation_threshold)
        activated_output[sensitive_region] *= amplification_factor

        activated_output[activated_output < 0] = F.leaky_relu(activated_output[activated_output < 0], negative_slope=0.01)

        return activated_output

    @staticmethod
    def backward(ctx, grad_output):
        input, amplification_factor, activation_threshold = ctx.saved_tensors
        grad_input = grad_output.clone()

        sensitive_region_grad = (input > activation_threshold)
        grad_input[sensitive_region_grad] *= amplification_factor.item() # Usar .item() para escalar

        grad_input[input < 0] *= 0.01

        # Devolvemos None para los argumentos no tensores (amplification_factor y activation_threshold)
        return grad_input, None, None

# Wrapper para la función de activación
hyper_sensitive_relu = HyperSensitiveReLU.apply

# --- NUEVO MÓDULO DE NORMALIZACIÓN: AdaptiveChaosNorm ---
class AdaptiveChaosNorm(nn.Module):
    def __init__(self, num_features, epsilon=1e-5, decay_rate=0.999):
        super().__init__()
        self.num_features = num_features
        self.epsilon = epsilon
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))

        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))
        self.decay_rate = decay_rate

        self.turbulence_factor = nn.Parameter(torch.ones(num_features) * 0.1)

    def forward(self, input):
        if self.training:
            batch_mean = input.mean(dim=0)
            # Calcular varianza con corrección de Bessel para consistencia con BatchNorm
            batch_var = input.var(dim=0, unbiased=True)

            with torch.no_grad():
                self.running_mean.data = self.decay_rate * self.running_mean.data + (1 - self.decay_rate) * batch_mean.data
                self.running_var.data = self.decay_rate * self.running_var.data + (1 - self.decay_rate) * batch_var.data

            # Asegurar que el factor de turbulencia sea positivo
            turbulence_scale = torch.abs(self.turbulence_factor)

            # Normalización que considera la turbulencia
            # La varianza ajustada es una combinación de la varianza del batch y un factor
            # aprendido basado en la varianza del batch (turbulencia).
            # Esto permite que la normalización se adapte a la "volatilidad" del lote.
            adjusted_var = batch_var + (turbulence_scale * batch_var) # Combina varianza con un factor de escala de turbulencia

            normalized_input = (input - batch_mean) / torch.sqrt(adjusted_var + self.epsilon)

            output = self.gamma * normalized_input + self.beta
        else:
            normalized_input = (input - self.running_mean) / torch.sqrt(self.running_var + self.epsilon)
            output = self.gamma * normalized_input + self.beta
        return output


# --- Red de Caos Controlado (CCN) ---
class CCNNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.chaos_norm1 = AdaptiveChaosNorm(hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.chaos_norm2 = AdaptiveChaosNorm(hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1_linear = self.fc1(x)
        # Aplicar HyperSensitiveReLU usando argumentos posicionales
        h1_activated = hyper_sensitive_relu(h1_linear, 10.0, 0.1) # Argumentos posicionales
        h1 = self.chaos_norm1(h1_activated)

        h2_linear = self.fc2(h1)
        # Aplicar HyperSensitiveReLU usando argumentos posicionales
        h2_activated = hyper_sensitive_relu(h2_linear, 10.0, 0.1) # Argumentos posicionales
        h2 = self.chaos_norm2(h2_activated)

        out = self.output_layer(h2)
        return out

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.01
EPOCHS = 2000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red de Caos Controlado (CCN) ---")
ccn_model = CCNNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_ccn = torch.optim.Adam(ccn_model.parameters(), lr=LEARNING_RATE)
historial_loss_ccn = []

total_params_ccn = sum(p.numel() for p in ccn_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red CCN: {total_params_ccn}")

for epoch in range(EPOCHS):
    ccn_model.train()
    optimizer_ccn.zero_grad()

    y_pred = ccn_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_ccn.append(loss.item())

    loss.backward()
    optimizer_ccn.step()

    if epoch % 100 == 0:
        print(f"Red CCN - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red CCN): {historial_loss_ccn[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar (ReLU) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

ccn_model.eval() # IMPORTANTE: poner en modo evaluación para inferencia sin el comportamiento "caótico"
standard_model.eval()

with torch.no_grad():
    y_hat_ccn = ccn_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, 'o', label="Valores Reales (Ruidosos)", color='gray', alpha=0.3, markersize=3)
plt.plot(X.numpy(), X.pow(3).numpy() - X.numpy(), label="Curva Verdadera (Sin Ruido)", color='blue', linewidth=2)
plt.plot(X_np, y_hat_ccn, label="Predicción Red CCN", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_ccn, label="Pérdida Red CCN", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red CCN): {historial_loss_ccn[-1]:.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")

##Capa Densa con Neuronas de Memoria Híbrida (HMNLayer)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1) # [300, 1]
y = X.pow(3) - X + 0.5 * torch.randn(X.size()) # [300, 1]

# --- NUEVA CAPA: Capa Densa con Neuronas de Memoria Híbrida (HMNLayer) ---
class HMNLayer(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features

        # Pesos y sesgos de una capa lineal normal
        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)
        self.bias = nn.Parameter(torch.randn(output_features) * 0.1)

        # CADA NEURONA DE SALIDA tiene su propia CELDA LSTM
        # nn.ModuleList para almacenar múltiples LSTMCells
        # Ensure hidden_size is 1 as intended
        self.lstm_cells = nn.ModuleList([nn.LSTMCell(input_size=1, hidden_size=1)
                                         for _ in range(output_features)])

        # Estados internos iniciales para cada LSTMCell (h y c)
        # Se inicializan a cero. Estos se actualizarán en cada forward pass.
        # These will be resized to [batch_size, output_features] during forward
        self.register_buffer('h_states', torch.zeros(output_features, 1)) # h_0 dummy shape
        self.register_buffer('c_states', torch.zeros(output_features, 1)) # c_0 dummy shape

    def forward(self, x):
        # 1. Realizar el cálculo lineal (Wx + b) como en una capa densa normal
        linear_output = torch.matmul(x, self.weight) + self.bias # Shape: [batch_size, output_features]

        # 2. Iterar a través de cada "neurona" (o feature de salida)
        # Each LSTMCell processes its respective column of the linear output
        # and maintains states [batch_size, 1] for that neuron's LSTM

        current_batch_size = x.size(0)

        # Ensure h_states and c_states match current batch size
        if self.h_states.size(0) != current_batch_size or self.h_states.size(1) != self.output_features:
            print(f"Resizing h_states from {self.h_states.shape} to [{current_batch_size}, {self.output_features}]")
            self.h_states = self.h_states.new_zeros(current_batch_size, self.output_features).to(x.device) # Move to device
        if self.c_states.size(0) != current_batch_size or self.c_states.size(1) != self.output_features:
             print(f"Resizing c_states from {self.c_states.shape} to [{current_batch_size}, {self.output_features}]")
             self.c_states = self.c_states.new_zeros(current_batch_size, self.output_features).to(x.device) # Move to device


        lstm_outputs = []
        new_h_states_list = [] # Use list to collect new states before cat
        new_c_states_list = [] # Use list to collect new states before cat


        # Iterate over the output features (neurons)
        for i in range(self.output_features):
            # Input for the i-th neuron's LSTM (i-th column of the linear output)
            # Shape: [batch_size, 1]
            input_i = linear_output[:, i].unsqueeze(1)

            # h and c states for the i-th neuron's LSTM across the batch
            # Each LSTMCell needs its own h_t and c_t for the batch, shape [batch_size, hidden_size=1]
            h_i = self.h_states[:, i].unsqueeze(1) # Shape: [batch_size, 1]
            c_i = self.c_states[:, i].unsqueeze(1) # Shape: [batch_size, 1]

            # Apply the LSTMCell of the neuron
            h_next_i, c_next_i = self.lstm_cells[i](input_i, (h_i, c_i))

            lstm_outputs.append(h_next_i) # Append [batch_size, 1]

            # DETACH the states before storing them for the next forward pass
            # This prevents building a computation graph across batches
            new_h_states_list.append(h_next_i.detach()) # Append [batch_size, 1]
            new_c_states_list.append(c_next_i.detach()) # Append [batch_size, 1]

        # Update the layer's internal states for the next forward pass
        # Concatenate the states for all neurons along dim=1
        self.h_states = torch.cat(new_h_states_list, dim=1) # Concatenates [batch_size, 1] * output_features -> [batch_size, output_features]
        self.c_states = torch.cat(new_c_states_list, dim=1) # Concatenates [batch_size, 1] * output_features -> [batch_size, output_features]

        # Concatenate the h outputs of each mini-LSTM into a single tensor
        return torch.cat(lstm_outputs, dim=1) # Shape: [batch_size, output_features]

# --- Red Neuronal usando las Capas HMN ---
class HMNNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # Cada capa oculta es una HMNLayer
        self.hmn_layer1 = HMNLayer(input_size, hidden_size)
        self.hmn_layer2 = HMNLayer(hidden_size, hidden_size)
        # La capa de salida es normal porque no necesita memoria interna para la predicción final
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = self.hmn_layer1(x)
        # Add print statement to check the shape after hmn_layer1
        # print(f"Shape after hmn_layer1: {h1.shape}") # Keep or remove print as desired
        h2 = self.hmn_layer2(h1)
        out = self.output_layer(h2)
        return out

# --- Red Neuronal Estándar para Comparación ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento ---
LEARNING_RATE = 0.01
EPOCHS = 2000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---
print("--- Inicializando y Entrenando Red con Neuronas de Memoria Híbrida (HMN) ---")
hmn_model = HMNNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_hmn = torch.optim.Adam(hmn_model.parameters(), lr=LEARNING_RATE)
historial_loss_hmn = []

total_params_hmn = sum(p.numel() for p in hmn_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red HMN: {total_params_hmn}")

for epoch in range(EPOCHS):
    hmn_model.train()
    optimizer_hmn.zero_grad()

    y_pred = hmn_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_hmn.append(loss.item())

    loss.backward()
    optimizer_hmn.step()

    if epoch % 100 == 0:
        print(f"Red HMN - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red HMN): {historial_loss_hmn[-1]:.6f}\n")


print("--- Inicializando y Entrenando Red Estándar (ReLU) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()

    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())

    loss.backward()
    optimizer_standard.step()

    if epoch % 100 == 0:
        print(f"Red Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")


## Visualización de Resultados

hmn_model.eval()
standard_model.eval()

with torch.no_grad():
    y_hat_hmn = hmn_model(X).numpy()
    y_hat_standard = standard_model(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 6))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones (Dataset Regresión)")
plt.plot(X_np, y_np, 'o', label="Valores Reales (Ruidosos)", color='gray', alpha=0.3, markersize=3)
plt.plot(X.numpy(), X.pow(3).numpy() - X.numpy(), label="Curva Verdadera (Sin Ruido)", color='blue', linewidth=2)
plt.plot(X_np, y_hat_hmn, label="Predicción Red HMN", color='red', linestyle='--')
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_hmn, label="Pérdida Red HMN", color='red')
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red HMN): {historial_loss_hmn[-1]:.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")

##Red Estándar (Organismo Procariota): Neuronas lineales simples.
Red con LSTM Tradicional (Organismo Multicelular Simple con Órganos Especializados): Una capa LSTM completa procesando secuencias.
Red con Neuronas de Memoria Híbrida - HMN (Organismo Eucariota con Células Especializadas y Memoria Celular): Cada "neurona" individual de una capa densa tiene su propia mini-LSTM.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Configuración de los dispositivos (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1).to(device) # [300, 1]
y = (X.pow(3) - X + 0.5 * torch.randn(X.size()).to(device)) # [300, 1]

# --- 1. Red Neuronal Estándar (Tipo "Procariota") ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- 2. Red con LSTM Tradicional (Tipo "Multicelular Simple") ---
class TraditionalLSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        # nn.LSTM toma una entrada (batch_size, seq_len, input_size)
        # Para nuestro dataset, seq_len es 1 (cada X es un paso de tiempo único)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Añadir la dimensión de seq_len=1 para LSTM (batch_first=True)
        # x_lstm_input: [batch_size, 1, input_size]
        x_lstm_input = x.unsqueeze(1)

        # h_n y c_n son los estados finales, h_out es la salida de la última capa LSTM para todos los pasos de tiempo
        # Como seq_len=1, h_out será [batch_size, 1, hidden_size]
        h_out, (h_n, c_n) = self.lstm(x_lstm_input)

        # Tomamos la salida del último paso de tiempo (que es el único aquí)
        # y_pred: [batch_size, hidden_size]
        y_pred = h_out.squeeze(1)

        # Pasar por la capa de salida
        out = self.output_layer(y_pred)
        return out

# --- 3. Red con Neuronas de Memoria Híbrida - HMN (Tipo "Eucariota") ---
class HMNLayer(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features

        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)
        self.bias = nn.Parameter(torch.randn(output_features) * 0.1)

        self.lstm_cells = nn.ModuleList([nn.LSTMCell(input_size=1, hidden_size=1)
                                         for _ in range(output_features)])

        # Estados internos para cada LSTMCell por cada ejemplo del batch.
        # Estos se reinicializarán o se actualizarán en el forward si el batch_size cambia.
        self.h_states = None
        self.c_states = None

    def forward(self, x):
        current_batch_size = x.size(0)

        # Inicializar o ajustar los estados si el batch_size es diferente o es la primera pasada
        if self.h_states is None or self.h_states.size(0) != current_batch_size:
            self.h_states = torch.zeros(current_batch_size, self.output_features, device=x.device)
            self.c_states = torch.zeros(current_batch_size, self.output_features, device=x.device)

        linear_output = torch.matmul(x, self.weight) + self.bias # Shape: [batch_size, output_features]

        new_h_states = []
        new_c_states = []
        lstm_outputs = []

        for i in range(self.output_features):
            # Entrada para la i-ésima neurona LSTM (columna de la salida lineal)
            input_i = linear_output[:, i].unsqueeze(1) # [batch_size, 1]

            # Estados h y c para la i-ésima neurona LSTM (para todo el batch)
            h_i = self.h_states[:, i].unsqueeze(1) # [batch_size, 1]
            c_i = self.c_states[:, i].unsqueeze(1) # [batch_size, 1]

            # Aplicar la LSTMCell de la neurona
            h_next_i, c_next_i = self.lstm_cells[i](input_i, (h_i, c_i))

            lstm_outputs.append(h_next_i)
            new_h_states.append(h_next_i)
            new_c_states.append(c_next_i)

        # Actualizar los estados internos de la capa para el siguiente forward pass
        self.h_states = torch.cat(new_h_states, dim=1)
        self.c_states = torch.cat(new_c_states, dim=1)

        return torch.cat(lstm_outputs, dim=1) # Shape: [batch_size, output_features]

class HMNNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hmn_layer1 = HMNLayer(input_size, hidden_size)
        self.hmn_layer2 = HMNLayer(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = self.hmn_layer1(x)
        h2 = self.hmn_layer2(h1)
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento Comunes ---
LEARNING_RATE = 0.01
EPOCHS = 2000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---

# 1. Red Estándar
print("--- Inicializando y Entrenando Red Estándar (Procariota) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(device)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()
    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())
    loss.backward()
    optimizer_standard.step()
    if epoch % 500 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")

# 2. Red con LSTM Tradicional
print("--- Inicializando y Entrenando Red con LSTM Tradicional (Multicelular Simple) ---")
lstm_model = TraditionalLSTMNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(device)
optimizer_lstm = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)
historial_loss_lstm = []

total_params_lstm = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red LSTM Tradicional: {total_params_lstm}")

for epoch in range(EPOCHS):
    lstm_model.train()
    optimizer_lstm.zero_grad()
    y_pred = lstm_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_lstm.append(loss.item())
    loss.backward()
    optimizer_lstm.step()
    if epoch % 500 == 0:
        print(f"LSTM Tradicional - Época {epoch} - Pérdida: {loss.item():.6f}")
print(f"Pérdida final (Red LSTM Tradicional): {historial_loss_lstm[-1]:.6f}\n")

# 3. Red con Neuronas de Memoria Híbrida - HMN
print("--- Inicializando y Entrenando Red con Neuronas de Memoria Híbrida (Eucariota) ---")
hmn_model = HMNNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(device)
optimizer_hmn = torch.optim.Adam(hmn_model.parameters(), lr=LEARNING_RATE)
historial_loss_hmn = []

total_params_hmn = sum(p.numel() for p in hmn_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red HMN: {total_params_hmn}")

for epoch in range(EPOCHS):
    hmn_model.train()
    optimizer_hmn.zero_grad()
    y_pred = hmn_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_hmn.append(loss.item())
    loss.backward()
    optimizer_hmn.step()
    if epoch % 500 == 0:
        print(f"HMN - Época {epoch} - Pérdida: {loss.item():.6f}")
print(f"Pérdida final (Red HMN): {historial_loss_hmn[-1]:.6f}\n")


## Visualización de Resultados

standard_model.eval()
lstm_model.eval()
hmn_model.eval()

with torch.no_grad():
    y_hat_standard = standard_model(X).cpu().numpy()
    y_hat_lstm = lstm_model(X).cpu().numpy()
    y_hat_hmn = hmn_model(X).cpu().numpy()

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()
X_true_func = (X.pow(3) - X).cpu().numpy() # Para la curva verdadera sin ruido

plt.figure(figsize=(18, 7))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones de Modelos")
plt.plot(X_np, y_np, 'o', label="Valores Reales (Ruidosos)", color='gray', alpha=0.3, markersize=3)
plt.plot(X_np, X_true_func, label="Curva Verdadera (Sin Ruido)", color='blue', linewidth=2)
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar (Procariota)", color='green', linestyle=':')
plt.plot(X_np, y_hat_lstm, label="Predicción Red LSTM Tradicional (Multicelular Simple)", color='orange', linestyle='-.')
plt.plot(X_np, y_hat_hmn, label="Predicción Red HMN (Eucariota)", color='red', linestyle='--')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.plot(historial_loss_lstm, label="Pérdida Red LSTM Tradicional", color='orange')
plt.plot(historial_loss_hmn, label="Pérdida Red HMN", color='red')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")
print(f"Pérdida final (Red LSTM Tradicional): {historial_loss_lstm[-1]:.6f}")
print(f"Pérdida final (Red HMN): {historial_loss_hmn[-1]:.6f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Configuración de los dispositivos (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Dataset sintético (regresión no lineal)
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1).to(device) # [300, 1]
y = (X.pow(3) - X + 0.5 * torch.randn(X.size()).to(device)) # [300, 1]

# --- 1. Red Neuronal Estándar (Tipo "Procariota") ---
class StandardNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- 2. Red con LSTM Tradicional (Tipo "Multicelular Simple") ---
class TraditionalLSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        # nn.LSTM toma una entrada (batch_size, seq_len, input_size)
        # Para nuestro dataset, seq_len es 1 (cada X es un paso de tiempo único)
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Añadir la dimensión de seq_len=1 para LSTM (batch_first=True)
        # x_lstm_input: [batch_size, 1, input_size]
        x_lstm_input = x.unsqueeze(1)

        # h_n y c_n son los estados finales, h_out es la salida de la última capa LSTM para todos los pasos de tiempo
        # Como seq_len=1, h_out será [batch_size, 1, hidden_size]
        h_out, (h_n, c_n) = self.lstm(x_lstm_input)

        # Tomamos la salida del último paso de tiempo (que es el único aquí)
        # y_pred: [batch_size, hidden_size]
        y_pred = h_out.squeeze(1)

        # Pasar por la capa de salida
        out = self.output_layer(y_pred)
        return out

# --- 3. Red con Neuronas de Memoria Híbrida - HMN (Tipo "Eucariota") ---
class HMNLayer(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.input_features = input_features
        self.output_features = output_features

        self.weight = nn.Parameter(torch.randn(input_features, output_features) * 0.1)
        self.bias = nn.Parameter(torch.randn(output_features) * 0.1)

        self.lstm_cells = nn.ModuleList([nn.LSTMCell(input_size=1, hidden_size=1)
                                         for _ in range(output_features)])

        # Estados internos para cada LSTMCell por cada ejemplo del batch.
        # Estos se reinicializarán o se actualizarán en el forward si el batch_size cambia.
        self.h_states = None
        self.c_states = None

    def forward(self, x):
        current_batch_size = x.size(0)

        # Inicializar o ajustar los estados si el batch_size es diferente o es la primera pasada
        if self.h_states is None or self.h_states.size(0) != current_batch_size:
            self.h_states = torch.zeros(current_batch_size, self.output_features, device=x.device)
            self.c_states = torch.zeros(current_batch_size, self.output_features, device=x.device)

        linear_output = torch.matmul(x, self.weight) + self.bias # Shape: [batch_size, output_features]

        new_h_states = []
        new_c_states = []
        lstm_outputs = []

        for i in range(self.output_features):
            # Entrada para la i-ésima neurona LSTM (columna de la salida lineal)
            input_i = linear_output[:, i].unsqueeze(1) # [batch_size, 1]

            # Estados h y c para la i-ésima neurona LSTM (para todo el batch)
            h_i = self.h_states[:, i].unsqueeze(1) # [batch_size, 1]
            c_i = self.c_states[:, i].unsqueeze(1) # [batch_size, 1]

            # Aplicar la LSTMCell de la neurona
            h_next_i, c_next_i = self.lstm_cells[i](input_i, (h_i, c_i))

            lstm_outputs.append(h_next_i)

            # *** FIX: Detach the states before storing them for the next forward pass ***
            # This prevents the computation graph from growing across batches.
            new_h_states.append(h_next_i.detach())
            new_c_states.append(c_next_i.detach())

        # Actualizar los estados internos de la capa para el siguiente forward pass
        self.h_states = torch.cat(new_h_states, dim=1)
        self.c_states = torch.cat(new_c_states, dim=1)

        return torch.cat(lstm_outputs, dim=1) # Shape: [batch_size, output_features]

class HMNNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hmn_layer1 = HMNLayer(input_size, hidden_size)
        self.hmn_layer2 = HMNLayer(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = self.hmn_layer1(x)
        h2 = self.hmn_layer2(h1)
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento Comunes ---
LEARNING_RATE = 0.01
EPOCHS = 2000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Inicialización y Entrenamiento de Modelos ---

# 1. Red Estándar
print("--- Inicializando y Entrenando Red Estándar (Procariota) ---")
standard_model = StandardNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(device)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = []

total_params_standard = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red Estándar: {total_params_standard}")

for epoch in range(EPOCHS):
    standard_model.train()
    optimizer_standard.zero_grad()
    y_pred = standard_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_standard.append(loss.item())
    loss.backward()
    optimizer_standard.step()
    if epoch % 500 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}\n")

# 2. Red con LSTM Tradicional
print("--- Inicializando y Entrenando Red con LSTM Tradicional (Multicelular Simple) ---")
lstm_model = TraditionalLSTMNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(device)
optimizer_lstm = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)
historial_loss_lstm = []

total_params_lstm = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red LSTM Tradicional: {total_params_lstm}")

for epoch in range(EPOCHS):
    lstm_model.train()
    optimizer_lstm.zero_grad()
    y_pred = lstm_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_lstm.append(loss.item())
    loss.backward()
    optimizer_lstm.step()
    if epoch % 500 == 0:
        print(f"LSTM Tradicional - Época {epoch} - Pérdida: {loss.item():.6f}")
print(f"Pérdida final (Red LSTM Tradicional): {historial_loss_lstm[-1]:.6f}\n")

# 3. Red con Neuronas de Memoria Híbrida - HMN
print("--- Inicializando y Entrenando Red con Neuronas de Memoria Híbrida (Eucariota) ---")
hmn_model = HMNNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).to(device)
optimizer_hmn = torch.optim.Adam(hmn_model.parameters(), lr=LEARNING_RATE)
historial_loss_hmn = []

total_params_hmn = sum(p.numel() for p in hmn_model.parameters() if p.requires_grad)
print(f"Número de parámetros en Red HMN: {total_params_hmn}")

for epoch in range(EPOCHS):
    hmn_model.train()
    optimizer_hmn.zero_grad()
    y_pred = hmn_model(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_hmn.append(loss.item())
    loss.backward()
    optimizer_hmn.step()
    if epoch % 500 == 0:
        print(f"HMN - Época {epoch} - Pérdida: {loss.item():.6f}")
print(f"Pérdida final (Red HMN): {historial_loss_hmn[-1]:.6f}\n")


## Visualización de Resultados

standard_model.eval()
lstm_model.eval()
hmn_model.eval()

with torch.no_grad():
    y_hat_standard = standard_model(X).cpu().numpy()
    y_hat_lstm = lstm_model(X).cpu().numpy()
    y_hat_hmn = hmn_model(X).cpu().numpy()

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()
X_true_func = (X.pow(3) - X).cpu().numpy() # Para la curva verdadera sin ruido

plt.figure(figsize=(18, 7))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones de Modelos")
plt.plot(X_np, y_np, 'o', label="Valores Reales (Ruidosos)", color='gray', alpha=0.3, markersize=3)
plt.plot(X_np, X_true_func, label="Curva Verdadera (Sin Ruido)", color='blue', linewidth=2)
plt.plot(X_np, y_hat_standard, label="Predicción Red Estándar (Procariota)", color='green', linestyle=':')
plt.plot(X_np, y_hat_lstm, label="Predicción Red LSTM Tradicional (Multicelular Simple)", color='orange', linestyle='-.')
plt.plot(X_np, y_hat_hmn, label="Predicción Red HMN (Eucariota)", color='red', linestyle='--')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_standard, label="Pérdida Red Estándar", color='green')
plt.plot(historial_loss_lstm, label="Pérdida Red LSTM Tradicional", color='orange')
plt.plot(historial_loss_hmn, label="Pérdida Red HMN", color='red')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")
print(f"Pérdida final (Red LSTM Tradicional): {historial_loss_lstm[-1]:.6f}")
print(f"Pérdida final (Red HMN): {historial_loss_hmn[-1]:.6f}")

## inicializador de pesos

In [ ]:
# Instalar la librería opensimplex
!pip install opensimplex

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import opensimplex # pip install opensimplex

# --- Configuración de los dispositivos (CPU/GPU) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# --- Dataset sintético (regresión no lineal con ruido) ---
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1).to(device) # [300, 1]
y = (X.pow(3) - X + 0.5 * torch.randn(X.size()).to(device)) # [300, 1]

# --- Funciones de Inicialización Personalizadas ---

def generate_perlin_noise_weights(shape, scale=0.05):
    """Genera un tensor de pesos basado en ruido Simplex (similar a Perlin)."""
    total_elements = np.prod(shape)
    noise_map = np.zeros(total_elements)
    # Generamos ruido 2D y lo aplanamos, dando una coherencia a través del tensor.
    # El offset 'i * 0.001' asegura que cada valor no sea idéntico
    for i in range(total_elements):
        noise_map[i] = opensimplex.noise2(x=i * scale, y=i * scale * 1.5)

    noise_map = noise_map.reshape(shape)

    # Escalamiento He/Kaiming para la activación ReLU
    fan_in = shape[0] if len(shape) > 1 else 1 # Para capas densas, fan_in es la primera dim
    if len(shape) == 4: # Para capas convolucionales
        fan_in = shape[1] * shape[2] * shape[3]

    std_dev = np.sqrt(2.0 / fan_in)
    noise_tensor = torch.from_numpy(noise_map).float() * std_dev
    return noise_tensor

def generate_lorenz_points(num_points, sigma=10, rho=28, beta=8/3, dt=0.01):
    """Genera puntos del atractor de Lorenz."""
    xs, ys, zs = [], [], []
    x, y, z = 0.1, 0.0, 0.0

    for _ in range(num_points):
        dx = (sigma * (y - x)) * dt
        dy = (x * (rho - z) - y) * dt
        dz = (x * y - beta * z) * dt
        x += dx
        y += dy
        z += dz
        xs.append(x)
        ys.append(y)
        zs.append(z)
    return np.array(xs + ys + zs)

def map_lorenz_to_weights(shape, lorenz_points, current_idx_ref):
    """
    Mapea puntos del atractor de Lorenz a la forma deseada de los pesos.
    current_idx_ref es una lista mutable para pasar el índice por referencia.
    """
    total_elements = np.prod(shape)

    if (current_idx_ref[0] + total_elements) > len(lorenz_points):
        raise ValueError(f"No hay suficientes puntos de Lorenz para inicializar los pesos de forma {shape}.")

    sampled_points = lorenz_points[current_idx_ref[0] : current_idx_ref[0] + total_elements]
    current_idx_ref[0] += total_elements # Actualizar el índice

    # Normalizar y escalar
    mean = np.mean(sampled_points)
    std = np.std(sampled_points)

    if std == 0:
        normalized_points = np.zeros_like(sampled_points)
    else:
        normalized_points = (sampled_points - mean) / std

    fan_in = shape[0] if len(shape) > 1 else 1
    if len(shape) == 4: # Conv
        fan_in = shape[1] * shape[2] * shape[3]

    scale_factor = np.sqrt(2.0 / fan_in)

    weights = normalized_points * scale_factor
    return torch.from_numpy(weights).float().reshape(shape)

# --- Definición de Modelos con Diferentes Inicializaciones ---

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, init_type='standard', lorenz_data=None, lorenz_idx_ref=None):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

        if init_type == 'standard':
            # PyTorch ya usa la inicialización de Kaiming Uniform por defecto para nn.Linear
            # con ReLU, lo cual es un buen estándar aleatorio.
            pass
        elif init_type == 'perlin':
            self.fc1.weight.data = generate_perlin_noise_weights(self.fc1.weight.shape, scale=0.05)
            self.fc1.bias.data = generate_perlin_noise_weights(self.fc1.bias.shape, scale=0.1).squeeze()
            self.fc2.weight.data = generate_perlin_noise_weights(self.fc2.weight.shape, scale=0.05)
            self.fc2.bias.data = generate_perlin_noise_weights(self.fc2.bias.shape, scale=0.1).squeeze()
            self.output_layer.weight.data = generate_perlin_noise_weights(self.output_layer.weight.shape, scale=0.05)
            self.output_layer.bias.data = generate_perlin_noise_weights(self.output_layer.bias.shape, scale=0.1).squeeze()
        elif init_type == 'lorenz':
            if lorenz_data is None or lorenz_idx_ref is None:
                raise ValueError("Se requiere 'lorenz_data' y 'lorenz_idx_ref' para la inicialización de Lorenz.")

            self.fc1.weight.data = map_lorenz_to_weights(self.fc1.weight.shape, lorenz_data, lorenz_idx_ref)
            self.fc1.bias.data = map_lorenz_to_weights(self.fc1.bias.shape, lorenz_data, lorenz_idx_ref).squeeze()
            self.fc2.weight.data = map_lorenz_to_weights(self.fc2.weight.shape, lorenz_data, lorenz_idx_ref)
            self.fc2.bias.data = map_lorenz_to_weights(self.fc2.bias.shape, lorenz_data, lorenz_idx_ref).squeeze()
            self.output_layer.weight.data = map_lorenz_to_weights(self.output_layer.weight.shape, lorenz_data, lorenz_idx_ref)
            self.output_layer.bias.data = map_lorenz_to_weights(self.output_layer.bias.shape, lorenz_data, lorenz_idx_ref).squeeze()
        else:
            raise ValueError(f"Tipo de inicialización desconocido: {init_type}")

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- Parámetros de Entrenamiento Comunes ---
LEARNING_RATE = 0.01
EPOCHS = 2000
LOSS_FN = nn.MSELoss()

INPUT_SIZE = X.shape[1] # 1
HIDDEN_SIZE = 32 # Tamaño de las capas ocultas
OUTPUT_SIZE = y.shape[1] # 1

# --- Función de Entrenamiento ---
def train_model(model, X_data, y_data, optimizer, epochs, loss_fn, model_name):
    historial_loss = []
    print(f"--- Entrenando {model_name} ---")
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        y_pred = model(X_data)
        loss = loss_fn(y_pred, y_data)
        historial_loss.append(loss.item())
        loss.backward()
        optimizer.step()
        if epoch % 500 == 0:
            print(f"{model_name} - Época {epoch} - Pérdida: {loss.item():.6f}")
    print(f"Pérdida final ({model_name}): {historial_loss[-1]:.6f}\n")
    return historial_loss

# --- Inicialización y Entrenamiento de Cada Modelo ---

# Modelo 1: Estándar (Aleatoria por defecto de PyTorch)
standard_model = MLP(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, init_type='standard').to(device)
optimizer_standard = torch.optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
historial_loss_standard = train_model(standard_model, X, y, optimizer_standard, EPOCHS, LOSS_FN, "Red Estándar")

# Modelo 2: Perlin Noise
perlin_model = MLP(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, init_type='perlin').to(device)
optimizer_perlin = torch.optim.Adam(perlin_model.parameters(), lr=LEARNING_RATE)
historial_loss_perlin = train_model(perlin_model, X, y, optimizer_perlin, EPOCHS, LOSS_FN, "Red con Ruido Perlin")

# Modelo 3: Atractor de Lorenz
# Calcular el total de elementos de pesos y sesgos necesarios para el modelo
total_elements_lorenz = sum(np.prod(p.shape) for p in MLP(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE).parameters())
lorenz_data = generate_lorenz_points(num_points=int(total_elements_lorenz * 1.5)) # Generar un poco más
lorenz_idx_ref = [0] # Usar una lista para pasar el índice por referencia y que se actualice
lorenz_model = MLP(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, init_type='lorenz', lorenz_data=lorenz_data, lorenz_idx_ref=lorenz_idx_ref).to(device)
optimizer_lorenz = torch.optim.Adam(lorenz_model.parameters(), lr=LEARNING_RATE)
historial_loss_lorenz = train_model(lorenz_model, X, y, optimizer_lorenz, EPOCHS, LOSS_FN, "Red con Atractor de Lorenz")

## --- Visualización de Resultados ---

standard_model.eval()
perlin_model.eval()
lorenz_model.eval()

with torch.no_grad():
    y_hat_standard = standard_model(X).cpu().numpy()
    y_hat_perlin = perlin_model(X).cpu().numpy()
    y_hat_lorenz = lorenz_model(X).cpu().numpy()

X_np = X.cpu().numpy()
y_np = y.cpu().numpy()
X_true_func = (X.pow(3) - X).cpu().numpy() # Para la curva verdadera sin ruido

plt.figure(figsize=(18, 7))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones por Tipo de Inicialización")
plt.plot(X_np, y_np, 'o', label="Valores Reales (Ruidosos)", color='gray', alpha=0.3, markersize=3)
plt.plot(X_np, X_true_func, label="Curva Verdadera (Sin Ruido)", color='blue', linewidth=2)
plt.plot(X_np, y_hat_standard, label="Predicción: Inicialización Estándar", color='green', linestyle=':')
plt.plot(X_np, y_hat_perlin, label="Predicción: Inicialización Perlin", color='purple', linestyle='-.')
plt.plot(X_np, y_hat_lorenz, label="Predicción: Inicialización Lorenz", color='brown', linestyle='--')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE) por Tipo de Inicialización")
plt.plot(historial_loss_standard, label="Pérdida: Inicialización Estándar", color='green')
plt.plot(historial_loss_perlin, label="Pérdida: Inicialización Perlin", color='purple')
plt.plot(historial_loss_lorenz, label="Pérdida: Inicialización Lorenz", color='brown')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red Estándar): {historial_loss_standard[-1]:.6f}")
print(f"Pérdida final (Red con Ruido Perlin): {historial_loss_perlin[-1]:.6f}")
print(f"Pérdida final (Red con Atractor de Lorenz): {historial_loss_lorenz[-1]:.6f}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import opensimplex # Requiere pip install opensimplex
import matplotlib.pyplot as plt

# Generador de Ruido Perlin (usando opensimplex por simplicidad, aunque Perlin es más general)
def generate_perlin_noise_weights(shape, scale=0.1, octaves=1, persistence=0.5, lacunarity=2.0):
    """
    Genera un tensor de pesos basado en ruido Perlin (o Simplex, similar).

    Args:
        shape (tuple): La forma (dimensiones) del tensor de pesos.
        scale (float): La escala del ruido (frecuencia).
        octaves (int): Número de capas de ruido.
        persistence (float): Cuánto contribuye cada octava a la amplitud general.
        lacunarity (float): Factor de incremento de la frecuencia para cada octava.
    Returns:
        torch.Tensor: Tensor de la forma especificada con valores de ruido.
    """
    if len(shape) > 2:
        print("Advertencia: El ruido Perlin/Simplex es más visual en 2D. Generando 1D y expandiendo.")
        # Simplificamos para demostración a 1D o 2D
        if len(shape) == 3: # Conv layer (out_channels, in_channels, kernel_size, kernel_size)
            total_elements = shape[0] * shape[1] * shape[2]
            noise_map = np.zeros(total_elements)
            for i in range(total_elements):
                noise_map[i] = opensimplex.noise2(x=i * scale, y=i * scale) # Usamos 2D para la coherencia

        elif len(shape) == 4: # Conv layer (out_channels, in_channels, kernel_h, kernel_w)
            # Para 4D, podemos generar ruido 2D para cada kernel y apilar
            num_kernels = shape[0] * shape[1]
            h, w = shape[2], shape[3]
            noise_map = np.zeros((num_kernels, h, w))
            for k_idx in range(num_kernels):
                for y_idx in range(h):
                    for x_idx in range(w):
                        noise_map[k_idx, y_idx, x_idx] = opensimplex.noise2(
                            x=x_idx * scale + k_idx * 100, # Un offset para cada kernel
                            y=y_idx * scale + k_idx * 100
                        )
        else: # Para otros casos, simplemente linealizamos
            total_elements = np.prod(shape)
            noise_map = np.zeros(total_elements)
            for i in range(total_elements):
                noise_map[i] = opensimplex.noise2(x=i * scale, y=i * scale)

    else: # Para 1D o 2D directo
        total_elements = np.prod(shape)
        noise_map = np.zeros(total_elements)
        if len(shape) == 1:
            for i in range(total_elements):
                noise_map[i] = opensimplex.noise2(x=i * scale, y=0) # Genera en 1D
        elif len(shape) == 2:
            rows, cols = shape
            noise_map = np.zeros((rows, cols))
            for r in range(rows):
                for c in range(cols):
                    noise_map[r, c] = opensimplex.noise2(x=c * scale, y=r * scale)

    noise_map = noise_map.reshape(shape) # Reshape de vuelta a la forma original

    # Normalizar el ruido (normalmente entre -1 y 1) y escalarlo para la inicialización
    # Una escala similar a Xavier/He puede ser útil, e.g., 1 / sqrt(fan_in)
    fan_in = shape[0] if len(shape) > 1 else 1 # Para capas densas, fan_in es la primera dim
    if len(shape) == 4: # Conv
        fan_in = shape[1] * shape[2] * shape[3]

    std_dev = np.sqrt(2.0 / fan_in) # Inspirado en He/Kaiming
    noise_tensor = torch.from_numpy(noise_map).float() * std_dev
    return noise_tensor

# Ejemplo de cómo usarlo en una capa lineal
class PerlinNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

        # Inicialización de pesos con ruido Perlin
        self.fc1.weight.data = generate_perlin_noise_weights(self.fc1.weight.shape, scale=0.05)
        self.fc2.weight.data = generate_perlin_noise_weights(self.fc2.weight.shape, scale=0.05)
        self.output_layer.weight.data = generate_perlin_noise_weights(self.output_layer.weight.shape, scale=0.05)

        # Inicialización de sesgos con ruido Perlin (para sesgos es más simple, 1D)
        self.fc1.bias.data = generate_perlin_noise_weights(self.fc1.bias.shape, scale=0.1).squeeze()
        self.fc2.bias.data = generate_perlin_noise_weights(self.fc2.bias.shape, scale=0.1).squeeze()
        self.output_layer.bias.data = generate_perlin_noise_weights(self.output_layer.bias.shape, scale=0.1).squeeze()

        # Opcional: Visualizar un peso de la primera capa (asegúrate de que la visualización esté fuera de __init__ si quieres ejecutarla)
        # plt.figure(figsize=(5,3))
        # plt.imshow(self.fc1.weight.data.cpu().numpy(), cmap='gray')
        # plt.title("Pesos fc1 inicializados con Ruido Perlin")
        # plt.colorbar()
        # plt.show()

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# # Ejemplo de uso:
input_size = 10
hidden_size = 20
output_size = 1
perlin_net = PerlinNet(input_size, hidden_size, output_size)
print("Red con inicialización Perlin creada.")
print(f"Forma de los pesos de fc1: {perlin_net.fc1.weight.shape}")

# Code block that was causing the indentation error. Move it outside the class/method.
# It appears this block was intended for visualization after creating the model,
# so it should be at the top level of the script.
plt.figure(figsize=(5,3))
plt.imshow(perlin_net.fc1.weight.data.cpu().numpy(), cmap='gray')
plt.title("Pesos fc1 inicializados con Ruido Perlin")
plt.colorbar()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, Subset
import numpy as np
import torch.nn.functional as F # Import the functional module with alias F

# --- 1. Dataset Sintético Progresivo ---
def create_synthetic_data_segments(num_samples_per_segment=100, num_segments=5):
    all_X = []
    all_y = []

    # Cada segmento tendrá una pequeña variación en la función
    # y = (X + offset)^3 - (X + offset) + ruido
    for i in range(num_segments):
        offset = i * 0.5 # Aumentar el offset para cada segmento

        # Generar X dentro de un rango específico para cada segmento
        # Así, cada segmento cubre una parte diferente del dominio, forzando la generalización
        start_x = -3 + i * 1.5
        end_x = start_x + 1.5

        X_segment = torch.linspace(start_x, end_x, num_samples_per_segment).unsqueeze(1)
        y_segment = (X_segment + offset).pow(3) - (X_segment + offset) + 0.5 * torch.randn(X_segment.size())

        all_X.append(X_segment)
        all_y.append(y_segment)

    return all_X, all_y # Lista de tensores para cada segmento

# Crear los segmentos de datos
X_segments, y_segments = create_synthetic_data_segments(num_samples_per_segment=200, num_segments=5)

# Crear un conjunto de prueba final que el modelo NUNCA ve durante el entrenamiento
X_test = torch.linspace(-5, 5, 500).unsqueeze(1)
y_test = X_test.pow(3) - X_test + 0.5 * torch.randn(X_test.size())
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# --- 2. Modelo (MLP simple) ---
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        out = self.output_layer(h2)
        return out

# --- 3. Bucle de Entrenamiento para Auto-Generalización ---
def train_auto_generalization(model, X_segments, y_segments, test_loader, epochs_per_segment=500, lr=0.01, loss_threshold=0.1):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    all_training_losses = []
    test_losses_per_segment = [] # Esta lista se retorna como test_losses_auto_gen

    for i, (X_seg, y_seg) in enumerate(zip(X_segments, y_segments)):
        print(f"\n--- Fase de Aprendizaje: Segmento {i+1} de {len(X_segments)} ---")

        # Convertir el segmento a DataLoader
        segment_dataset = TensorDataset(X_seg, y_seg)
        segment_loader = DataLoader(segment_dataset, batch_size=32, shuffle=True)

        current_segment_losses = []
        for epoch in range(epochs_per_segment):
            model.train()
            epoch_loss = 0.0
            for batch_X, batch_y in segment_loader:
                optimizer.zero_grad()
                pred_y = model(batch_X)
                loss = loss_fn(pred_y, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * batch_X.size(0)

            avg_epoch_loss = epoch_loss / len(segment_dataset)
            current_segment_losses.append(avg_epoch_loss)
            all_training_losses.append(avg_epoch_loss)

            if epoch % 100 == 0:
                print(f"Segmento {i+1}, Época {epoch}: Pérdida = {avg_epoch_loss:.6f}")

            # Criterio de avance: si la pérdida es suficientemente baja
            if avg_epoch_loss < loss_threshold and epoch > 100: # Para evitar que termine muy rápido
                print(f"Umbral de pérdida alcanzado para el segmento {i+1}. Avanzando...")
                break

        # Evaluar en el conjunto de prueba completo después de cada segmento
        model.eval()
        total_test_loss = 0.0
        with torch.no_grad():
            for batch_X_test, batch_y_test in test_loader:
                pred_y_test = model(batch_X_test)
                loss_test = loss_fn(pred_y_test, batch_y_test)
                total_test_loss += loss_test.item() * batch_X_test.size(0)

        avg_test_loss = total_test_loss / len(test_dataset)
        test_losses_per_segment.append(avg_test_loss)
        print(f"Pérdida en el conjunto de prueba (después del segmento {i+1}): {avg_test_loss:.6f}")

    # Return the training losses for all epochs and the test losses after each segment
    return all_training_losses, test_losses_per_segment

# --- Ejecución del Experimento ---
input_size = 1
hidden_size = 64
output_size = 1

model_auto_gen = SimpleMLP(input_size, hidden_size, output_size)
# Correctly assign the returned values
training_losses_auto_gen, test_losses_auto_gen = train_auto_generalization(
    model_auto_gen, X_segments, y_segments, test_loader,
    epochs_per_segment=1000, lr=0.005, loss_threshold=0.05 # Ajustar umbral y épocas
)

# --- Comparación con Entrenamiento Tradicional (con todo el dataset) ---
print("\n--- Entrenando Modelo con Enfoque Tradicional (Dataset completo) ---")
X_full = torch.cat(X_segments, dim=0)
y_full = torch.cat(y_segments, dim=0) # Cambiado de dim0 a dim=0
full_dataset = TensorDataset(X_full, y_full)
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)

model_traditional = SimpleMLP(input_size, hidden_size, output_size)
optimizer_traditional = optim.Adam(model_traditional.parameters(), lr=0.005)
loss_fn = nn.MSELoss()
traditional_training_losses = []

for epoch in range(len(X_segments) * 1000): # Mismo número total de épocas que el enfoque auto-generalización
    model_traditional.train()
    epoch_loss = 0.0
    for batch_X, batch_y in full_loader:
        optimizer_traditional.zero_grad()
        pred_y = model_traditional(batch_X)
        loss = loss_fn(pred_y, batch_y)
        loss.backward()
        optimizer_traditional.step()
        epoch_loss += loss.item() * batch_X.size(0)

    avg_epoch_loss = epoch_loss / len(full_dataset)
    traditional_training_losses.append(avg_epoch_loss)

    if epoch % 500 == 0:
        print(f"Tradicional, Época {epoch}: Pérdida = {avg_epoch_loss:.6f}")

# Evaluar modelo tradicional en el conjunto de prueba final
model_traditional.eval()
total_test_loss_traditional = 0.0
with torch.no_grad():
    for batch_X_test, batch_y_test in test_loader:
        pred_y_test = model_traditional(batch_X_test)
        loss_test = loss_fn(pred_y_test, batch_y_test)
        total_test_loss_traditional += loss_test.item() * batch_X_test.size(0)
avg_test_loss_traditional = total_test_loss_traditional / len(test_dataset)
print(f"Pérdida final en el conjunto de prueba (Modelo Tradicional): {avg_test_loss_traditional:.6f}")


# --- Visualización de Resultados ---
import matplotlib.pyplot as plt

# Predicciones finales
model_auto_gen.eval()
model_traditional.eval()
with torch.no_grad():
    pred_auto_gen = model_auto_gen(X_test).cpu().numpy()
    pred_traditional = model_traditional(X_test).cpu().numpy()

X_test_np = X_test.cpu().numpy()
y_test_np = y_test.cpu().numpy()
y_true_func = X_test.pow(3) - X_test # Curva sin ruido

plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.plot(X_test_np, y_test_np, 'o', label="Datos de Prueba (Ruidosos)", color='gray', alpha=0.3, markersize=3)
plt.plot(X_test_np, y_true_func.cpu().numpy(), label="Curva Verdadera (Sin Ruido)", color='blue', linewidth=2)
plt.plot(X_test_np, pred_auto_gen, label="Predicción Auto-Generalización", color='red', linestyle='--')
plt.plot(X_test_np, pred_traditional, label="Predicción Tradicional", color='green', linestyle=':')
plt.title("Comparación de Predicciones en Conjunto de Prueba")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(training_losses_auto_gen, label="Pérdida Entrenamiento Auto-Generalización")
# Changed test_losses_per_segment to test_losses_auto_gen
plt.plot(test_losses_auto_gen, label="Pérdida Prueba (después de cada segmento)", linestyle='--', color='red')
plt.plot(traditional_training_losses, label="Pérdida Entrenamiento Tradicional", color='green')
plt.title("Evolución de la Pérdida de Entrenamiento")
plt.xlabel("Pasos/Épocas")
plt.ylabel("Pérdida MSE")
plt.yscale('log')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\n--- Resumen Final de Pérdidas en Conjunto de Prueba ---")
print(f"Modelo Auto-Generalización: {test_losses_auto_gen[-1]:.6f}")
print(f"Modelo Tradicional: {avg_test_loss_traditional:.6f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossInteractionUnit(nn.Module):
    """
    Unidad de Interacción Cruzada (CIU) para permitir que los streams se influyan mutuamente.
    Podría ser una pequeña red de atención o una simple capa lineal y activación.
    """
    def __init__(self, query_dim, key_dim, value_dim, output_dim, num_heads=4):
        super().__init__()
        # Usaremos Multi-Head Attention para permitir una interacción rica
        # Note: embed_dim for MultiheadAttention is the dimension of the query, key, and value after projection
        self.attention = nn.MultiheadAttention(embed_dim=query_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(query_dim)
        self.linear = nn.Linear(query_dim, output_dim) # Para mapear la salida a la dimensión deseada

        # Mapeo lineal para que el "key" y "value" tengan el mismo embed_dim que el "query"
        # Their input dimensions are key_dim and value_dim, output is query_dim
        self.key_proj = nn.Linear(key_dim, query_dim)
        self.value_proj = nn.Linear(value_dim, query_dim) # If key_dim != value_dim

    def forward(self, query_state, influencing_state):
        # query_state: El estado que será influenciado (ej. h_p_t) [batch, query_dim]
        # influencing_state: El estado que influye (ej. h_c_t) [batch, key_dim or value_dim]

        # Expandir las dimensiones para la atención si es necesario (ej. para batch_first=True)
        query = query_state.unsqueeze(1) # [batch, 1, query_dim]
        key = self.key_proj(influencing_state).unsqueeze(1)   # [batch, 1, query_dim]
        value = self.value_proj(influencing_state).unsqueeze(1) # [batch, 1, query_dim]

        # attn_output will be [batch, 1, query_dim]
        attn_output, _ = self.attention(query, key, value)
        attn_output = attn_output.squeeze(1) # [batch, query_dim]

        # Combinar el resultado de la atención con el query original (residual connection)
        # Ensure shapes match for addition: query_state [batch, query_dim] + attn_output [batch, query_dim]
        output = self.norm(query_state + attn_output)

        # Apply final linear layer and activation
        output = F.relu(self.linear(output)) # Output shape: [batch, output_dim]
        return output

class DReGCell(nn.Module):
    """
    La celda central del Generador de Texto Recurrente Dual (D-ReG).
    Se ejecuta una vez por cada token generado.
    """
    def __init__(self, embed_dim, hidden_dim_p, hidden_dim_c, num_heads_p=8, ff_dim_p=2048, num_layers_p=1):
        super().__init__()
        self.embed_dim = embed_dim
        self.hidden_dim_p = hidden_dim_p
        self.hidden_dim_c = hidden_dim_c

        # P-Stream (Percepción): Un bloque de Transformer Decoder (con auto-atención)
        # Aquí simplificamos a un Attention y Feedforward para el ejemplo
        # The attention mechanism operates on the embedding dimension
        self.p_self_attn = nn.MultiheadAttention(embed_dim, num_heads_p, batch_first=True)
        self.p_ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim_p),
            nn.ReLU(),
            nn.Linear(ff_dim_p, embed_dim)
        )
        self.p_norm1 = nn.LayerNorm(embed_dim)
        self.p_norm2 = nn.LayerNorm(embed_dim)

        # C-Stream (Conceptualización): Un GRU o LSTM para mantener el estado conceptual
        # Recibe input del P-Stream y su propio estado anterior
        # Input size should be the dimension of the input it receives (from p_to_c_ciu output)
        # Hidden size should be the dimension of the conceptual state h_c
        # FIX: Set GRU input_size to hidden_dim_c to match p_to_c_ciu output dimension
        self.c_gru = nn.GRUCell(hidden_dim_c, hidden_dim_c) # Input: output de p_to_c_ciu [batch, hidden_dim_c], hidden: h_c_prev [batch, hidden_dim_c]

        # Cross-Interaction Units (CIU)
        # P-Stream influye en C-Stream (para ajustar la conceptualización basada en la percepción)
        # Query: h_c_prev [batch, hidden_dim_c] -> query_dim=hidden_dim_c
        # Influencing: h_p_curr [batch, embed_dim] -> key_dim=embed_dim, value_dim=embed_dim
        # Output needs to be input for GRU -> output_dim=hidden_dim_c
        self.p_to_c_ciu = CrossInteractionUnit(query_dim=hidden_dim_c, key_dim=embed_dim, value_dim=embed_dim, output_dim=hidden_dim_c)
        # C-Stream influye en P-Stream (para guiar la percepción basada en la conceptualización)
        # Query: h_p_prev [batch, embed_dim] -> query_dim=embed_dim
        # Influencing: h_c_prev [batch, hidden_dim_c] -> key_dim=hidden_dim_c, value_dim=hidden_dim_c
        # Output needs to be combined with token embedding -> output_dim=embed_dim
        self.c_to_p_ciu = CrossInteractionUnit(query_dim=embed_dim, key_dim=hidden_dim_c, value_dim=hidden_dim_c, output_dim=embed_dim)

    def forward(self, prev_token_embedding, h_p_prev, h_c_prev):
        # prev_token_embedding: Embedding del token generado en el paso anterior [batch, embed_dim]
        # h_p_prev: Estado oculto anterior del P-Stream [batch, embed_dim]
        # h_c_prev: Estado oculto anterior del C-Stream [batch, hidden_dim_c]

        # 1. P-Stream (Percepción)
        # h_p_prev influenciado por h_c_prev
        # The CIU expects h_p_prev as query_state and h_c_prev as influencing_state
        modulated_h_p_prev = self.c_to_p_ciu(h_p_prev, h_c_prev) # C-Stream guía P-Stream [batch, embed_dim]

        # Input to P-Stream attention: should consider both token embedding and modulated state
        # A simple approach is to combine them, or use them as part of the attention mechanism.
        # Here, we'll use the modulated state as the base and add the token embedding's contribution.
        # A more complex Transformer Decoder would handle this differently.
        # For simplicity in this DReGCell, let's assume the P-Stream self-attention
        # primarily processes the modulated state, guided by the current input token.
        # The line below is a simplification for this cell structure.
        p_input_for_attn = (modulated_h_p_prev + prev_token_embedding).unsqueeze(1) # [batch, 1, embed_dim]


        # Self-attention for the P-Stream.
        # In a decoder real, it would be causal attention over the sequence generated so far.
        # For this cell example, we perform attention on a single timestep input (combined state/token).
        # The output dim of attention is embed_dim, matching the query_dim passed to MultiheadAttention
        p_output_attn, _ = self.p_self_attn(p_input_for_attn, p_input_for_attn, p_input_for_attn) # [batch, 1, embed_dim]
        p_output_attn = p_output_attn.squeeze(1) # [batch, embed_dim]

        # Residual connection and Normalization
        # p_output_attn represents the perception at this step. Combine with previous state influence.
        p_output = self.p_norm1(p_output_attn) # Apply norm after attention

        # Feedforward for P-Stream
        p_output = self.p_norm2(p_output + self.p_ffn(p_output)) # [batch, embed_dim]
        h_p_curr = p_output # This is the new state of the P-Stream [batch, embed_dim]

        # 2. C-Stream (Conceptualization)
        # h_c_prev influenced by h_p_curr
        # The output of the P-Stream influences the C-Stream
        # The CIU expects h_c_prev as query_state and h_p_curr as influencing_state
        # The output dimension of p_to_c_ciu is hidden_dim_c
        c_input_from_p = self.p_to_c_ciu(h_c_prev, h_p_curr) # P-Stream adjusts C-Stream [batch, hidden_dim_c]

        # GRU to update the conceptual state.
        # The input to the GRU is the influence from the P-Stream (c_input_from_p).
        # The hidden state is h_c_prev.
        # Both inputs to GRUCell should have the dimension hidden_dim_c (256)
        h_c_curr = self.c_gru(c_input_from_p, h_c_prev) # [batch, hidden_dim_c]

        return h_p_curr, h_c_curr

class DReG(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim_p, hidden_dim_c, num_heads_enc=8, ff_dim_enc=2048, num_layers_enc=6):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # Position embedding size should be sufficient for max sequence length including prompt and generation
        self.position_embedding = nn.Embedding(512 + 50, embed_dim) # Max seq len + max generated len
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        self.hidden_dim_c = hidden_dim_c # Keep track for initializing h_c_prev

        # Encoder (Transformer estándar) para el prompt inicial
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads_enc, dim_feedforward=ff_dim_enc, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers_enc)

        # The D-ReG cell that will be executed sequentially
        self.dreg_cell = DReGCell(embed_dim, hidden_dim_p, hidden_dim_c)

        # Final projection layer to vocabulary
        # Output is based on the P-Stream state (perception)
        self.output_linear = nn.Linear(hidden_dim_p, vocab_size) # Predict next token from P-Stream state

    def forward(self, input_ids, prompt_mask=None, max_seq_len=50):
        # input_ids: [batch, seq_len_prompt] - the initial prompt
        batch_size = input_ids.size(0)
        seq_len_prompt = input_ids.size(1)

        # 1. Process the prompt with the Encoder
        # Position embeddings should start from 0 for the prompt
        positions = torch.arange(seq_len_prompt, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        input_embeddings = self.token_embedding(input_ids) + self.position_embedding(positions)

        encoder_output = self.encoder(input_embeddings, src_key_padding_mask=prompt_mask)

        # Initialize P-Stream and C-Stream states from encoder output
        # h_p_prev is initialized from the last token of the encoder output [batch, embed_dim]
        h_p_prev = encoder_output[:, -1, :].contiguous()

        # h_c_prev initialized with zeros or a projection from encoder output
        # Its dimension should match hidden_dim_c
        h_c_prev = torch.zeros(batch_size, self.hidden_dim_c, device=input_ids.device) # [batch, hidden_dim_c]

        # The first token embedding for the generation loop is from the last token of the prompt
        next_token_embedding = self.token_embedding(input_ids[:, -1]) # [batch, embed_dim]

        generated_tokens_list = [] # Store generated token IDs step by step

        # 2. Generation Loop
        for i in range(max_seq_len):
            # Execute the D-ReG cell
            # Inputs: next_token_embedding [batch, embed_dim], h_p_prev [batch, embed_dim], h_c_prev [batch, hidden_dim_c]
            h_p_curr, h_c_curr = self.dreg_cell(next_token_embedding, h_p_prev, h_c_prev)

            # Project the output of the P-Stream (h_p_curr) to vocabulary probabilities
            # Output dimension should match hidden_dim_p (or embed_dim depending on model design)
            # FIX: Use h_p_curr which has shape [batch, hidden_dim_p] as input to output_linear
            logits = self.output_linear(h_p_curr) # [batch, vocab_size]
            probabilities = F.softmax(logits, dim=-1)

            # Sample the next token (or take argmax for deterministic inference)
            next_token = torch.argmax(probabilities, dim=-1) # [batch]
            generated_tokens_list.append(next_token.unsqueeze(1)) # [batch, 1]

            # Prepare for the next step
            next_token_embedding = self.token_embedding(next_token) # [batch, embed_dim]
            h_p_prev = h_p_curr # Update P-Stream state for next iteration
            h_c_prev = h_c_curr # Update C-Stream state for next iteration

            # Optional: Stop if EOS token is generated - not implemented here

        # Concatenate the generated tokens
        generated_tokens = torch.cat(generated_tokens_list, dim=1) # [batch, max_seq_len]

        # Combine prompt and generated tokens (optional, depending on desired output format)
        # full_sequence = torch.cat([input_ids, generated_tokens], dim=1) # [batch, seq_len_prompt + max_seq_len]
        return generated_tokens # Return only the generated part


# --- Example Usage and Parameters ---
vocab_size = 10000 # Vocabulary size
embed_dim = 512    # Embedding dimensionality
hidden_dim_p = 512 # Hidden dimension of the P-Stream (Transformer-like)
hidden_dim_c = 256 # Hidden dimension of the C-Stream (GRU-like, conceptual summary)

# Ensure hidden_dim_p matches embed_dim if P-Stream state is embed_dim,
# or adjust output_linear if h_p_curr is hidden_dim_p and it's different from embed_dim.
# Based on DReGCell forward, h_p_curr is embed_dim.
# So output_linear should take embed_dim as input.
# FIX: Change output_linear input dim to embed_dim
class DReG(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim_p, hidden_dim_c, num_heads_enc=8, ff_dim_enc=2048, num_layers_enc=6):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        # Position embedding size should be sufficient for max sequence length including prompt and generation
        self.position_embedding = nn.Embedding(512 + 50, embed_dim) # Max seq len + max generated len
        self.embed_dim = embed_dim
        self.vocab_size = vocab_size
        self.hidden_dim_c = hidden_dim_c # Keep track for initializing h_c_prev

        # Encoder (Transformer estándar) para el prompt inicial
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads_enc, dim_feedforward=ff_dim_enc, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers_enc)

        # The D-ReG cell that will be executed sequentially
        # hidden_dim_p is not explicitly used as state size in DReGCell based on its current forward.
        # h_p_prev and h_p_curr are embed_dim. hidden_dim_p might be intended for ff_dim_p?
        # Let's keep it consistent with the original definition but note the potential discrepancy
        # if hidden_dim_p was meant to be the state size of the P-stream.
        self.dreg_cell = DReGCell(embed_dim, hidden_dim_p, hidden_dim_c) # embed_dim for p_state_size, hidden_dim_p for ff_dim_p

        # Final projection layer to vocabulary
        # Output is based on the P-Stream state (perception), which is h_p_curr
        # h_p_curr has shape [batch, embed_dim] based on DReGCell forward
        # FIX: Output linear takes embed_dim as input
        self.output_linear = nn.Linear(embed_dim, vocab_size) # Predict next token from P-Stream state

    def forward(self, input_ids, prompt_mask=None, max_seq_len=50):
        # input_ids: [batch, seq_len_prompt] - the initial prompt
        batch_size = input_ids.size(0)
        seq_len_prompt = input_ids.size(1)

        # 1. Process the prompt with the Encoder
        # Position embeddings should start from 0 for the prompt
        positions = torch.arange(seq_len_prompt, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        input_embeddings = self.token_embedding(input_ids) + self.position_embedding(positions)

        encoder_output = self.encoder(input_embeddings, src_key_padding_mask=prompt_mask)

        # Initialize P-Stream and C-Stream states from encoder output
        # h_p_prev is initialized from the last token of the encoder output [batch, embed_dim]
        h_p_prev = encoder_output[:, -1, :].contiguous()

        # h_c_prev initialized with zeros or a projection from encoder output
        # Its dimension should match hidden_dim_c
        h_c_prev = torch.zeros(batch_size, self.hidden_dim_c, device=input_ids.device) # [batch, hidden_dim_c]

        # The first token embedding for the generation loop is from the last token of the prompt
        next_token_embedding = self.token_embedding(input_ids[:, -1]) # [batch, embed_dim]

        generated_tokens_list = [] # Store generated token IDs step by step

        # 2. Generation Loop
        for i in range(max_seq_len):
            # Execute the D-ReG cell
            # Inputs: next_token_embedding [batch, embed_dim], h_p_prev [batch, embed_dim], h_c_prev [batch, hidden_dim_c]
            h_p_curr, h_c_curr = self.dreg_cell(next_token_embedding, h_p_prev, h_c_prev)

            # Project the output of the P-Stream (h_p_curr) to vocabulary probabilities
            # h_p_curr has shape [batch, embed_dim]
            logits = self.output_linear(h_p_curr) # [batch, vocab_size]
            probabilities = F.softmax(logits, dim=-1)

            # Sample the next token (or take argmax for deterministic inference)
            next_token = torch.argmax(probabilities, dim=-1) # [batch]
            generated_tokens_list.append(next_token.unsqueeze(1)) # [batch, 1]

            # Prepare for the next step
            next_token_embedding = self.token_embedding(next_token) # [batch, embed_dim]
            h_p_prev = h_p_curr # Update P-Stream state for next iteration
            h_c_prev = h_c_curr # Update C-Stream state for next iteration

            # Optional: Stop if EOS token is generated - not implemented here

        # Concatenate the generated tokens
        generated_tokens = torch.cat(generated_tokens_list, dim=1) # [batch, max_seq_len]

        # Combine prompt and generated tokens (optional, depending on desired output format)
        # full_sequence = torch.cat([input_ids, generated_tokens], dim=1) # [batch, seq_len_prompt + max_seq_len]
        return generated_tokens # Return only the generated part


# --- Example Usage and Parámetros ---
vocab_size = 10000 # Tamaño del vocabulario
embed_dim = 512    # Dimensionalidad de embeddings
hidden_dim_p = 512 # Dimensión oculta del P-Stream (ff_dim_p in DReGCell)
hidden_dim_c = 256 # Dimensión oculta del C-Stream (GRU hidden size)


# Instantiate the model
# The arguments to DReGCell are embed_dim, hidden_dim_p (used as ff_dim_p), hidden_dim_c
model = DReG(vocab_size, embed_dim, hidden_dim_p, hidden_dim_c).to('cpu') # Usar CPU para ejemplo
print(model)

# Example prompt (token IDs)
dummy_prompt_ids = torch.randint(0, vocab_size, (2, 10)) # Batch of 2, prompt of 10 tokens
generated_output = model(dummy_prompt_ids, max_seq_len=20)
print(f"\nForma de la salida generada (IDs de tokens): {generated_output.shape}")

# Parameter count
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Número total de parámetros: {total_params}")

Tu observación es interesante, pero es importante aclarar que el código que me proporcionaste, y sobre el que estamos trabajando, **no está diseñado para procesar imágenes directamente**.

---

### ¿Qué hace tu código actualmente?

El código que has compartido trabaja con un **dataset sintético de regresión no lineal**, donde `X` es un tensor de una sola característica (números que van de -3 a 3) y `y` es el resultado de una función cúbica de `X` más algo de ruido.

Las redes neuronales (`RedCaracteristicasDensaEntrada` y `RedEstandar`) que implementas son **redes neuronales de tipo `nn.Linear` (fully connected o densas)**. Estas capas están diseñadas para procesar datos tabulares o vectores, no imágenes con su estructura espacial (altura, anchura, canales de color).

Aunque estas redes extraen "características" en el sentido de que transforman la entrada a través de sus capas ocultas, el objetivo aquí es aprender la relación matemática entre `X` y `y` para realizar una **predicción de regresión**, no para reconocer objetos o patrones en una imagen.

---

### ¿Por qué no está procesando imágenes?

1.  **Tipo de datos de entrada:** `X` es un tensor de forma `(300, 1)`, lo que significa 300 muestras, cada una con 1 característica. Una imagen suele ser un tensor 3D (altura x anchura x canales) o 4D (número de imágenes x canales x altura x anchura).
2.  **Arquitectura de la red:** Las capas `nn.Linear` simplemente realizan una multiplicación de matriz y una suma de sesgo. No tienen las operaciones especiales (como convoluciones o pooling) que se usan en las redes neuronales convolucionales (CNNs) para extraer características espaciales de las imágenes y preservar su estructura.
3.  **Función de salida:** La capa de salida (`output_layer`) tiene `output_size=1`, lo que es típico para una regresión de un solo valor, no para generar una imagen o una representación dimensional de ella.
4.  **Visualización:** Los gráficos que se generan muestran la curva de predicción de la regresión, no imágenes.

---

### ¿Cómo se adaptarían para imágenes?

Para que estas redes **extrajeran características de una imagen y luego las "dimensionalizaran para ser mostradas como imagen"** (lo que implicaría una tarea de reconstrucción de imagen o generación), necesitarías:

1.  **Cambiar el dataset:** Cargar imágenes reales (por ejemplo, con `torchvision.datasets`).
2.  **Cambiar la arquitectura:**
    * Reemplazar las capas `nn.Linear` por **capas convolucionales (`nn.Conv2d`)** al inicio para la extracción de características espaciales.
    * Podrías seguir usando ideas de conexiones densas (como en DenseNet) con capas convolucionales.
    * Para la "re-dimensionalización" y mostrar como imagen, necesitarías **capas de convolución transpuesta (`nn.ConvTranspose2d`)** o capas de **upsampling** para reconstruir la resolución espacial de una imagen a partir de un conjunto de características latentes.
3.  **Cambiar la función de pérdida:** Si el objetivo es reconstruir una imagen, la función de pérdida podría ser MSE (como ahora, pero aplicada a píxeles de imagen) o alguna pérdida perceptiva.
4.  **Cambiar la visualización:** Mostrar las imágenes originales y las imágenes reconstruidas o generadas.

En resumen, el código actual no hace lo que describes para imágenes, sino que resuelve un problema de regresión de datos numéricos. Para procesar imágenes, la arquitectura y el enfoque deberían ser fundamentalmente diferentes, empleando técnicas específicas para datos de imagen.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Dataset sintético (regresión no lineal) ---
torch.manual_seed(0)
np.random.seed(0)
X = torch.linspace(-3, 3, 300).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())



## Red con Extracción de Características Densa desde la Entrada



class RedCaracteristicasDensaEntrada(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super().__init__()
        self.input_size = input_size # Guardamos el tamaño de entrada

        # Capa 1: Conectada a la entrada
        self.fc1 = nn.Linear(input_size, hidden_size1)

        # Capa 2: Recibe de Capa 1 Y de la entrada original
        # La entrada para fc2 será (hidden_size1 + input_size)
        self.fc2 = nn.Linear(hidden_size1 + input_size, hidden_size2)

        # Capa 3: Recibe de Capa 1, Capa 2 Y de la entrada original
        # La entrada para fc3 será (hidden_size1 + hidden_size2 + input_size)
        self.fc3 = nn.Linear(hidden_size1 + hidden_size2 + input_size, hidden_size3)

        # Capa de Salida: Recibe de Capa 1, Capa 2, Capa 3 Y de la entrada original
        # La entrada para la capa de salida será (hidden_size1 + hidden_size2 + hidden_size3 + input_size)
        self.output_layer = nn.Linear(hidden_size1 + hidden_size2 + hidden_size3 + input_size, output_size)

    def forward(self, x):
        # x_original es la entrada que se concatenará en cada paso
        x_original = x # [batch, input_size]

        # Capa 1
        h1 = F.relu(self.fc1(x_original)) # [batch, hidden_size1]

        # Capa 2: Entrada es la concatenación de h1 y x_original
        h2_input = torch.cat((h1, x_original), dim=1) # [batch, hidden_size1 + input_size]
        h2 = F.relu(self.fc2(h2_input)) # [batch, hidden_size2]

        # Capa 3: Entrada es la concatenación de h1, h2 y x_original
        h3_input = torch.cat((h1, h2, x_original), dim=1) # [batch, hidden_size1 + hidden_size2 + input_size]
        h3 = F.relu(self.fc3(h3_input)) # [batch, hidden_size3]

        # Capa de Salida: Entrada es la concatenación de h1, h2, h3 y x_original
        output_input = torch.cat((h1, h2, h3, x_original), dim=1) # [batch, hidden_size1 + hidden_size2 + hidden_size3 + input_size]

        y = self.output_layer(output_input)
        return y

class RedEstandar(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, hidden_size3)
        self.output_layer = nn.Linear(hidden_size3, output_size)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        h3 = F.relu(self.fc3(h2))
        y = self.output_layer(h3)
        return y


LEARNING_RATE = 0.01
EPOCHS = 300
LOSS_FN = nn.MSELoss()

HIDDEN_SIZE_1 = 32
HIDDEN_SIZE_2 = 16
HIDDEN_SIZE_3 = 8

print("--- Entrenando Red con Extracción de Características Densa desde la Entrada ---")
modelo_densa_entrada = RedCaracteristicasDensaEntrada(
    input_size=X.shape[1],
    hidden_size1=HIDDEN_SIZE_1,
    hidden_size2=HIDDEN_SIZE_2,
    hidden_size3=HIDDEN_SIZE_3,
    output_size=y.shape[1]
)
opt_densa_entrada = torch.optim.Adam(modelo_densa_entrada.parameters(), lr=LEARNING_RATE)
historial_loss_densa_entrada = []

for epoch in range(EPOCHS):
    y_pred = modelo_densa_entrada(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_densa_entrada.append(loss.item())
    opt_densa_entrada.zero_grad()
    loss.backward()
    opt_densa_entrada.step()
    if epoch % 50 == 0:
        print(f"Densa Entrada - Época {epoch} - Pérdida: {loss.item():.6f}")

print("\n--- Entrenando Red Estándar ---")
modelo_estandar = RedEstandar(
    input_size=X.shape[1],
    hidden_size1=HIDDEN_SIZE_1,
    hidden_size2=HIDDEN_SIZE_2,
    hidden_size3=HIDDEN_SIZE_3,
    output_size=y.shape[1]
)
opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=LEARNING_RATE)
historial_loss_estandar = []

for epoch in range(EPOCHS):
    y_pred = modelo_estandar(X)
    loss = LOSS_FN(y_pred, y)
    historial_loss_estandar.append(loss.item())
    opt_estandar.zero_grad()
    loss.backward()
    opt_estandar.step()
    if epoch % 50 == 0:
        print(f"Estándar - Época {epoch} - Pérdida: {loss.item():.6f}")

with torch.no_grad():
    y_hat_densa_entrada = modelo_densa_entrada(X).numpy()
    y_hat_estandar = modelo_estandar(X).numpy()

X_np = X.numpy()
y_np = y.numpy()

plt.figure(figsize=(15, 5))

# Gráfico de Predicciones
plt.subplot(1, 2, 1)
plt.title("Comparación de Predicciones")
plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
plt.plot(X_np, y_hat_densa_entrada, label="Predicción con Características Densa desde Entrada", color='red', linestyle='--')
plt.plot(X_np, y_hat_estandar, label="Predicción Estándar", color='green', linestyle=':')
plt.legend()
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)

# Gráfico de Evolución de la Pérdida
plt.subplot(1, 2, 2)
plt.title("Evolución de la Pérdida (MSE)")
plt.plot(historial_loss_densa_entrada, label="Pérdida con Características Densa desde Entrada", color='red')
plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Escala logarítmica para una mejor visualización
plt.tight_layout()
plt.show()

# Resumen Final de la Pérdida
print(f"\n--- Resumen de Pérdidas Finales ---")
print(f"Pérdida final (Red con Características Densa desde Entrada): {historial_loss_densa_entrada[-1]:.6f}")
print(f"Pérdida final (Red Estándar):                                {historial_loss_estandar[-1]:.6f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Definición de las Arquitecturas ---

## CNN Estándar

class CNNStandard(nn.Module):
    def __init__(self):
        super(CNNStandard, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)  # 1 canal de entrada (gris), 16 canales de salida
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)  # Ajustado para imágenes de 28x28 después de pooling
        self.fc2 = nn.Linear(128, 784)  # Salida para "reconstruir" una imagen de 28x28 (784 = 28 * 28)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)  # Aplanar para las capas lineales
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))  # Sigmoid para salida entre 0 y 1 (como píxeles)
        x = x.view(-1, 1, 28, 28)  # Reformar a imagen
        return x

## CNN Experimental (con conexiones densas a la entrada)

class CNNExperimental(nn.Module):
    def __init__(self):
        super(CNNExperimental, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        # La entrada a conv2 debe coincidir con el número total de canales después de la concatenación.
        # Salida de conv1 tiene 16 canales. Entrada original (x) tiene 1 canal. Total = 16 + 1 = 17.
        self.conv2 = nn.Conv2d(16 + 1, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # La entrada a fc1 debe ser el tamaño aplanado después de conv2 y pool.
        # Salida de conv2 tiene 32 canales. Después de dos pools de 2x2, las dimensiones HxW se dividen por 4.
        # Imagen original 28x28 -> después del primer pool 14x14 -> después del segundo pool 7x7.
        # Entonces, el tamaño aplanado es 32 * 7 * 7.
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 784)

    def forward(self, x):
        # print(f"CNNExperimental: Input shape: {x.shape}")
        h1 = F.relu(self.conv1(x))
        # print(f"CNNExperimental: After conv1 shape: {h1.shape}")
        # Conexión densa a la entrada: concatenar la entrada original (x) a la salida de conv1
        # Asegúrate de que las dimensiones espaciales (H, W) coinciden para la concatenación.
        # Si conv1 usa padding=1 y stride=1, la salida HxW es la misma que la entrada.
        # Si el pooling cambia las dimensiones, la concatenación debe hacerse *antes* del pooling
        # o asegurar que las dimensiones espaciales sean consistentes.
        # En este caso, conv1(x) tiene el mismo HxW que x.
        x_for_conv2 = torch.cat((h1, x), dim=1)  # Concatenar a lo largo de la dimensión del canal
        # print(f"CNNExperimental: Before conv2 (after cat) shape: {x_for_conv2.shape}")

        x = F.relu(self.conv2(x_for_conv2))
        # print(f"CNNExperimental: After conv2 shape: {x.shape}")
        x = self.pool(x) # Aplica pool después de conv2
        # print(f"CNNExperimental: After pool shape: {x.shape}")

        x = self.pool(x) # Aplica el segundo pool
        # print(f"CNNExperimental: After second pool shape: {x.shape}")


        x = x.view(-1, 32 * 7 * 7)  # Aplanar para las capas lineales
        # print(f"CNNExperimental: After flatten shape: {x.shape}")

        x = F.relu(self.fc1(x))
        # print(f"CNNExperimental: After fc1 shape: {x.shape}")
        x = torch.sigmoid(self.fc2(x))
        # print(f"CNNExperimental: Before final reshape shape: {x.shape}")
        x = x.view(-1, 1, 28, 28)  # Reformar a imagen
        # print(f"CNNExperimental: Final output shape: {x.shape}")
        return x

# --- 2. Preparación de Datos ---
transform = transforms.Compose([
    transforms.ToTensor(),  # Convierte la imagen a tensor, escala a [0, 1]
    # Importante: Si el modelo usa Sigmoid en la salida, la normalización a (-1, 1)
    # con Normalize((0.5,), (0.5,)) hará que el MSELoss sea incorrecto porque
    # el target estará en (-1, 1) y la predicción en (0, 1).
    # Para reconstrucción con Sigmoid, el target (input original) también debe estar en [0, 1].
    # Eliminamos la normalización si la salida usa Sigmoid y el objetivo es [0, 1].
    # Si el objetivo es (-1, 1), cambia la función de activación de salida o normaliza el target.
    # Dado que MNIST ToTensor escala a [0, 1], y Sigmoid sale a [0, 1],
    # parece que el objetivo es [0, 1]. Eliminamos la normalización a (-1, 1).
    # transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# --- 3. Instanciar Modelos ---
cnn_standard = CNNStandard()
cnn_experimental = CNNExperimental()

# --- 4. Función de Pérdida y Optimizadores ---
# Si el target es [0, 1] y la predicción es [0, 1] (debido a Sigmoid), MSELoss está bien.
criterion = nn.MSELoss()
optimizer_standard = optim.Adam(cnn_standard.parameters(), lr=0.001)
optimizer_experimental = optim.Adam(cnn_experimental.parameters(), lr=0.001)

# --- 5. Entrenamiento ---
def train(model, optimizer, epochs=2):
    model.train()
    for epoch in range(epochs):
        for batch_idx, (data, _) in enumerate(train_loader): # No necesitamos las etiquetas
            # Mueve los datos al dispositivo (GPU si está disponible)
            # device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Define device here or globally
            # data = data.to(device)
            # model = model.to(device) # Ensure model is on the correct device

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, data)  # Comparar la salida con la entrada (reconstrucción)
            loss.backward()
            optimizer.step()
            if batch_idx % 100 == 0:
                # Asegura que el loss.item() se imprima desde la CPU
                print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.6f}')
    return model

# Asegura que los modelos estén en la CPU antes de entrenar si no se mueven dentro de train
# o mueve los datos al dispositivo del modelo dentro de train.
# Mover modelos y datos al dispositivo si se usa GPU:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# cnn_standard.to(device)
# cnn_experimental.to(device)
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True if device.type == 'cuda' else False) # pin_memory puede acelerar en GPU


print("--- Entrenando CNN Estándar ---")
cnn_standard = train(cnn_standard, optimizer_standard, epochs=2)

print("\n--- Entrenando CNN Experimental ---")
# La normalización eliminada arriba debería solucionar el problema de tamaño.
# Si no, los print statements en CNNExperimental.forward ayudarán a diagnosticar el flujo de dimensiones.
cnn_experimental = train(cnn_experimental, optimizer_experimental, epochs=2)


# --- 6. Visualización de Resultados (Ejemplo) ---
def show_images(model, data_loader, num_images=5):
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Define device
    model.eval()
    with torch.no_grad():
        # Get a batch of data
        data, _ = next(iter(data_loader))
        # Move data to the same device as the model
        # data = data.to(device)
        output = model(data)

        plt.figure(figsize=(15, 5))
        for i in range(num_images):
            # Imagen Original (mueve a CPU y convierte a numpy)
            plt.subplot(2, num_images, i + 1)
            # Revertir la normalización si se aplicó (aunque la eliminamos).
            # Si se normalizó a (-1, 1) con Normalize((0.5,), (0.5,)),
            # data_np = data[i].squeeze().cpu().numpy() * 0.5 + 0.5
            # Si no se normalizó (solo ToTensor), ya está en [0, 1]
            data_np = data[i].squeeze().cpu().numpy()
            plt.imshow(data_np, cmap='gray')
            plt.title('Original')
            plt.axis('off')

            # Imagen Reconstruida (mueve a CPU y convierte a numpy)
            plt.subplot(2, num_images, i + 1 + num_images)
            # output está en [0, 1] debido a Sigmoid.
            output_np = output[i].squeeze().cpu().numpy()
            plt.imshow(output_np, cmap='gray')
            plt.title('Reconstruida')
            plt.axis('off')
        plt.show()

print("\n--- Visualizando resultados de CNN Estándar ---")
show_images(cnn_standard, train_loader)

print("\n--- Visualizando resultados de CNN Experimental ---")
show_images(cnn_experimental, train_loader)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Definición de las Arquitecturas ---

## CNN Estándar

class CNNStandard(nn.Module):
    def __init__(self):
        super(CNNStandard, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)  # 1 canal de entrada (gris), 16 canales de salida
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)  # Ajustado para imágenes de 28x28 después de pooling
        self.fc2 = nn.Linear(128, 784)  # Salida para "reconstruir" una imagen de 28x28 (784 = 28 * 28)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)  # Aplanar para las capas lineales
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))  # Sigmoid para salida entre 0 y 1 (como píxeles)
        x = x.view(-1, 1, 28, 28)  # Reformar a imagen
        return x

## CNN Experimental (con conexiones densas a la entrada)

class CNNExperimental(nn.Module):
    def __init__(self):
        super(CNNExperimental, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        # La entrada a conv2 debe coincidir con el número total de canales después de la concatenación.
        # Salida de conv1 tiene 16 canales. Entrada original (x) tiene 1 canal. Total = 16 + 1 = 17.
        self.conv2 = nn.Conv2d(16 + 1, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # La entrada a fc1 debe ser el tamaño aplanado después de conv2 y pool.
        # Salida de conv2 tiene 32 canales. Después de dos pools de 2x2, las dimensiones HxW se dividen por 4.
        # Imagen original 28x28 -> después del primer pool 14x14 -> después del segundo pool 7x7.
        # Entonces, el tamaño aplanado es 32 * 7 * 7.
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 784)

    def forward(self, x):
        h1 = F.relu(self.conv1(x))
        # Conexión densa a la entrada: concatenar la entrada original (x) a la salida de conv1
        # Asegúrate de que las dimensiones espaciales (H, W) coincidan para la concatenación.
        # En este caso, conv1(x) tiene el mismo HxW que x debido a padding=1 y stride=1.
        x_for_conv2 = torch.cat((h1, x), dim=1)  # Concatenar a lo largo de la dimensión del canal

        x = F.relu(self.conv2(x_for_conv2))
        x = self.pool(x) # Aplica pool después de conv2

        x = self.pool(x) # Aplica el segundo pool


        x = x.view(-1, 32 * 7 * 7)  # Aplanar para las capas lineales

        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        x = x.view(-1, 1, 28, 28)  # Reformar a imagen
        return x

# --- 2. Preparación de Datos ---
transform = transforms.Compose([
    transforms.ToTensor(),  # Convierte la imagen a tensor, escala a [0, 1]
    # Se eliminó la normalización a (-1, 1) para que el target esté en [0, 1] como la salida de sigmoid.
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# --- 3. Instanciar Modelos ---
cnn_standard = CNNStandard()
cnn_experimental = CNNExperimental()

# --- 4. Función de Pérdida y Optimizadores ---
criterion = nn.MSELoss() # MSELoss es adecuado para la reconstrucción de píxeles en el rango [0, 1]
optimizer_standard = optim.Adam(cnn_standard.parameters(), lr=0.001)
optimizer_experimental = optim.Adam(cnn_experimental.parameters(), lr=0.001)

# --- 5. Entrenamiento ---
def train(model, optimizer, epochs=2):
    model.train()
    for epoch in range(epochs):
        for batch_idx, (data, _) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, data)  # Comparar la salida con la entrada (reconstrucción)
            loss.backward()
            optimizer.step()
            if batch_idx % 100 == 0:
                print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.6f}')
    return model

print("--- Entrenando CNN Estándar ---")
cnn_standard = train(cnn_standard, optimizer_standard, epochs=2)

print("\n--- Entrenando CNN Experimental ---")
cnn_experimental = train(cnn_experimental, optimizer_experimental, epochs=2)

# --- 6. Visualización de Resultados (Imágenes reconstruidas) ---
def show_images(model, data_loader, num_images=5, title_prefix=""):
    model.eval()
    with torch.no_grad():
        data, _ = next(iter(data_loader))
        output = model(data)

        plt.figure(figsize=(15, 5))
        plt.suptitle(f'{title_prefix} - Imágenes Originales vs. Reconstruidas', fontsize=16)
        for i in range(num_images):
            # Imagen Original
            plt.subplot(2, num_images, i + 1)
            data_np = data[i].squeeze().cpu().numpy()
            plt.imshow(data_np, cmap='gray')
            plt.title('Original')
            plt.axis('off')

            # Imagen Reconstruida
            plt.subplot(2, num_images, i + 1 + num_images)
            output_np = output[i].squeeze().cpu().numpy()
            plt.imshow(output_np, cmap='gray')
            plt.title('Reconstruida')
            plt.axis('off')
        plt.show()

print("\n--- Visualizando resultados de CNN Estándar ---")
show_images(cnn_standard, train_loader, title_prefix="CNN Estándar")

print("\n--- Visualizando resultados de CNN Experimental ---")
show_images(cnn_experimental, train_loader, title_prefix="CNN Experimental")

# --- 7. Visualización de Filtros ---
def visualize_filters(model_name, model, layer_name):
    layer = None
    if layer_name == 'conv1':
        layer = model.conv1
    elif layer_name == 'conv2':
        layer = model.conv2
    else:
        print(f"Capa {layer_name} no encontrada o no es convolucional.")
        return

    if layer is None:
        print(f"Capa {layer_name} no encontrada en el modelo {model_name}.")
        return

    # Extraer los pesos (kernels) de la capa convolucional
    # Los pesos tienen forma (out_channels, in_channels, kernel_height, kernel_width)
    filters = layer.weight.data.clone().cpu()

    # Normalizar los filtros para visualizarlos como imágenes
    filters = filters - filters.min()
    filters = filters / filters.max()

    num_filters = filters.shape[0] # Número de filtros de salida
    in_channels = filters.shape[1] # Número de canales de entrada por filtro
    kernel_size = filters.shape[2] # Tamaño del kernel (ej: 3x3)

    # Calcular el número de filas y columnas para el subGráfico
    # Mostraremos cada filtro de salida en su propio subGráfico
    # Si un filtro tiene múltiples canales de entrada (como conv2),
    # podríamos mostrar cada canal del filtro, o solo el primer canal, o promediar.
    # Aquí, para simplificar y visualizar la "forma" principal, tomaremos el primer canal.
    # Para conv2, los filtros de cada canal de entrada son diferentes.
    # Podrías necesitar un enfoque diferente para visualizar conv2 de forma exhaustiva.

    cols = 8 # Número de columnas para la visualización
    rows = int(np.ceil(num_filters / cols))

    plt.figure(figsize=(cols * 2, rows * 2))
    plt.suptitle(f'Filtros de la capa {layer_name} de {model_name.capitalize()} CNN', fontsize=16)

    for i in range(num_filters):
        plt.subplot(rows, cols, i + 1)
        # Mostrar el primer canal del filtro.
        # Para conv1, in_channels es 1, así que filters[i, 0, :, :] es el filtro completo.
        # Para conv2 (CNNExperimental), in_channels es 17. filters[i, 0, :, :] mostraría
        # la parte del filtro que actúa sobre el primer canal de entrada combinado.
        # Puedes experimentar mostrando diferentes canales o promediándolos.
        plt.imshow(filters[i, 0, :, :], cmap='gray')
        plt.title(f'Filtro {i+1}')
        plt.axis('off')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Ajustar layout para el título
    plt.show()

print("\n--- Visualizando filtros de CNN Estándar (conv1) ---")
visualize_filters('standard', cnn_standard, 'conv1')

print("\n--- Visualizando filtros de CNN Estándar (conv2) ---")
visualize_filters('standard', cnn_standard, 'conv2')

print("\n--- Visualizando filtros de CNN Experimental (conv1) ---")
visualize_filters('experimental', cnn_experimental, 'conv1')

print("\n--- Visualizando filtros de CNN Experimental (conv2) ---")
visualize_filters('experimental', cnn_experimental, 'conv2')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# --- Re-definiendo la CNN Estándar (o carga tu modelo entrenado) ---
class CNNStandard(nn.Module):
    def __init__(self):
        super(CNNStandard, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 784)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        x = x.view(-1, 1, 28, 28)
        return x

# --- Cargar un modelo pre-entrenado o el que entrenaste antes ---
# Para este ejemplo, instanciamos el modelo. Idealmente, cargarías uno que ya hayas entrenado.
model = CNNStandard()

# --- Visualizar los filtros de la primera capa convolucional (conv1) ---
def visualize_filters(model_name, layer_name):
    layer = None
    if model_name == 'standard':
        model_to_visualize = CNNStandard() # Instancia para la visualización
        # Aquí deberías cargar los pesos entrenados, por ejemplo:
        # model_to_visualize.load_state_dict(torch.load('ruta_a_tu_modelo_standard.pth', weights_only=True))
        # Para este ejemplo, si no tienes un .pth, usaremos los pesos iniciales/aleatorios.
        layer = model_to_visualize.conv1
    elif model_name == 'experimental':
        # Tendrías que definir y cargar la CNNExperimental aquí
        # model_to_visualize = CNNExperimental()
        # model_to_visualize.load_state_dict(torch.load('ruta_a_tu_modelo_experimental.pth', weights_only=True))
        # layer = model_to_visualize.conv1 # o conv2, según quieras ver.
        print(f"La visualización para {model_name} no está implementada completamente en este ejemplo.")
        return

    if layer is None:
        print(f"Capa {layer_name} no encontrada o modelo no válido.")
        return

    # Extraer los pesos (kernels) de la capa convolucional
    # Los pesos tienen forma (out_channels, in_channels, kernel_height, kernel_width)
    filters = layer.weight.data.clone().cpu()

    # Normalizar los filtros para visualizarlos como imágenes
    # Escalar entre 0 y 1 para que matplotlib los muestre correctamente
    filters = filters - filters.min()
    filters = filters / filters.max()

    num_filters = filters.shape[0]
    kernel_size = filters.shape[2]

    # Calcular el número de filas y columnas para el subGráfico
    cols = 8
    rows = int(np.ceil(num_filters / cols))

    plt.figure(figsize=(cols * 2, rows * 2))
    plt.suptitle(f'Filtros de la capa {layer_name} de {model_name.capitalize()} CNN', fontsize=16)

    for i in range(num_filters):
        plt.subplot(rows, cols, i + 1)
        # Mostrar el filtro, que tiene 1 canal de entrada (escala de grises)
        # Si tu capa conv1 tuviera más de 1 canal de entrada (ej: RGB), tendrías que promediarlos o seleccionar uno
        plt.imshow(filters[i, 0, :, :], cmap='gray') # filters[i, 0, :, :] selecciona el primer canal de entrada del filtro i
        plt.title(f'Filtro {i+1}')
        plt.axis('off')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Ajustar layout para el título
    plt.show()

# --- Ejecutar la visualización para la primera capa de la CNN Estándar ---
# Nota: Si no entrenaste el modelo, estos filtros serán aleatorios (iniciales).
# Para ver filtros "aprendidos", el modelo debe haber sido entrenado y sus pesos guardados/cargados.
visualize_filters('standard', 'conv1')

# Si quisieras ver los filtros de la segunda capa (conv2), sería más complejo
# porque sus filtros operan sobre mapas de características que ya no son imágenes simples.
# Tendrías que decidir cómo colapsar o mostrar los múltiples canales de entrada de conv2.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Definición de las Arquitecturas ---

## CNN Estándar

class CNNStandard(nn.Module):
    def __init__(self):
        super(CNNStandard, self).__init__()
        # Capas convolucionales
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1) # Entrada: 1 canal, Salida: 16 canales (28x28)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1) # Entrada: 16 canales, Salida: 32 canales (28x28)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1) # Entrada: 32 canales, Salida: 64 canales (28x28)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Capas lineales para la reconstrucción
        # Después de 3 conv con padding 1 y 2 pools de 2x2:
        # 28x28 (input) -> conv1 (28x28) -> conv2 (28x28) -> conv3 (28x28)
        # -> pool1 (14x14) -> pool2 (7x7)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 784) # Salida para reconstruir 28x28 imagen

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x)) # Aplica todas las conv antes del pooling para una comparación justa de profundidad

        x = self.pool(x) # Primer pooling (64 canales, 14x14)
        x = self.pool(x) # Segundo pooling (64 canales, 7x7)

        x = x.view(-1, 64 * 7 * 7) # Aplanar
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x)) # Sigmoid para valores de píxel entre 0 y 1
        x = x.view(-1, 1, 28, 28) # Reformar a imagen
        return x

## CNN Experimental (Conexiones densas con Input Directo a todas las capas ocultas)

class CNNDenseConcatenatedInput(nn.Module):
    def __init__(self):
        super(CNNDenseConcatenatedInput, self).__init__()
        # Definimos los tamaños de las capas ocultas para una comparación más directa
        # (similar a tu descripción de 16, 32, 64 canales de salida)
        self.conv1_out_channels = 16
        self.conv2_out_channels = 32
        self.conv3_out_channels = 64
        self.input_channels = 1 # Para imágenes de MNIST

        # Capa oculta 1: Conectada a la entrada
        self.conv1 = nn.Conv2d(self.input_channels, self.conv1_out_channels, kernel_size=3, stride=1, padding=1)

        # Capa oculta 2: Recibe de Capa oculta 1 Y de la entrada original
        # Entrada: (salida conv1 + entrada original)
        self.conv2 = nn.Conv2d(self.conv1_out_channels + self.input_channels, self.conv2_out_channels, kernel_size=3, stride=1, padding=1)

        # Capa oculta 3: Recibe de Capa oculta 1, Capa oculta 2 Y de la entrada original
        # Entrada: (salida conv1 + salida conv2 + entrada original)
        self.conv3 = nn.Conv2d(self.conv1_out_channels + self.conv2_out_channels + self.input_channels, self.conv3_out_channels, kernel_size=3, stride=1, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Capas lineales para la reconstrucción (igual que la estándar para una salida comparable)
        # Después de 3 conv con padding 1, y luego 2 pools
        self.fc1 = nn.Linear(self.conv3_out_channels * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 784)

    def forward(self, x_original):
        # Capa oculta 1
        h1 = F.relu(self.conv1(x_original)) # Salida: 16 canales, 28x28

        # Capa oculta 2
        # Concatenar h1 (16 canales, 28x28) y x_original (1 canal, 28x28)
        h2_input = torch.cat((h1, x_original), dim=1) # Dimensión de canales: 16 + 1 = 17
        h2 = F.relu(self.conv2(h2_input)) # Salida: 32 canales, 28x28

        # Capa oculta 3
        # Concatenar h1 (16), h2 (32) y x_original (1)
        h3_input = torch.cat((h1, h2, x_original), dim=1) # Dimensión de canales: 16 + 32 + 1 = 49
        h3 = F.relu(self.conv3(h3_input)) # Salida: 64 canales, 28x28

        # Aplicar pooling después de todas las capas convolucionales
        x = self.pool(h3) # (64 canales, 14x14)
        x = self.pool(x) # (64 canales, 7x7)

        # Aplanar y pasar por las capas lineales
        x = x.view(-1, self.conv3_out_channels * 7 * 7)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        x = x.view(-1, 1, 28, 28)
        return x

# --- 2. Preparación de Datos ---
transform = transforms.Compose([
    transforms.ToTensor(), # Convierte imagen a tensor, escala a [0, 1]
    # No se aplica normalización a (-1, 1) ya que la salida de la red usa Sigmoid (0-1).
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# --- 3. Instanciar Modelos ---
cnn_standard = CNNStandard()
cnn_dense_input = CNNDenseConcatenatedInput()

# --- 4. Función de Pérdida y Optimizadores ---
criterion = nn.MSELoss() # Función de pérdida para la reconstrucción
optimizer_standard = optim.Adam(cnn_standard.parameters(), lr=0.001)
optimizer_dense_input = optim.Adam(cnn_dense_input.parameters(), lr=0.001)

# --- 5. Entrenamiento ---
def train_model(model, optimizer, epochs=2, model_name=""):
    model.train()
    print(f"\n--- Entrenando {model_name} ---")
    for epoch in range(epochs):
        for batch_idx, (data, _) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, data)
            loss.backward()
            optimizer.step()
            if batch_idx % 100 == 0:
                print(f'{model_name} - Época {epoch}, Lote {batch_idx}, Pérdida: {loss.item():.6f}')
    return model

cnn_standard = train_model(cnn_standard, optimizer_standard, epochs=2, model_name="CNN Estándar")
cnn_dense_input = train_model(cnn_dense_input, optimizer_dense_input, epochs=2, model_name="CNN Densa con Entrada")

# --- 6. Visualización de Resultados (Imágenes reconstruidas) ---
def show_reconstructions(model, data_loader, num_images=5, title_prefix=""):
    model.eval()
    with torch.no_grad():
        data, _ = next(iter(data_loader))
        output = model(data)

        plt.figure(figsize=(15, 5))
        plt.suptitle(f'{title_prefix} - Imágenes Originales vs. Reconstruidas', fontsize=16)
        for i in range(num_images):
            plt.subplot(2, num_images, i + 1)
            plt.imshow(data[i].squeeze().cpu().numpy(), cmap='gray')
            plt.title('Original')
            plt.axis('off')

            plt.subplot(2, num_images, i + 1 + num_images)
            plt.imshow(output[i].squeeze().cpu().numpy(), cmap='gray')
            plt.title('Reconstruida')
            plt.axis('off')
        plt.show()

print("\n--- Visualizando Reconstrucciones de CNN Estándar ---")
show_reconstructions(cnn_standard, train_loader, title_prefix="CNN Estándar")

print("\n--- Visualizando Reconstrucciones de CNN Densa con Entrada ---")
show_reconstructions(cnn_dense_input, train_loader, title_prefix="CNN Densa con Entrada")

# --- 7. Visualización de Filtros ---
def visualize_filters(model_name, model, layer_name):
    layer = None
    if hasattr(model, layer_name): # Verifica si el modelo tiene directamente el atributo
        layer = getattr(model, layer_name)
    else: # Intenta buscarlo por nombre en los módulos hijos si es más complejo
        for name, module in model.named_modules():
            if name == layer_name and isinstance(module, nn.Conv2d):
                layer = module
                break

    if not isinstance(layer, nn.Conv2d): # Asegurarse de que sea una capa Conv2d
        print(f"Capa '{layer_name}' no es una capa convolucional o no encontrada en el modelo {model_name}.")
        return

    filters = layer.weight.data.clone().cpu()
    filters = filters - filters.min()
    filters = filters / filters.max()

    num_filters = filters.shape[0]
    in_channels_per_filter = filters.shape[1] # Número de canales de entrada que un filtro ve

    cols = 8
    rows = int(np.ceil(num_filters / cols))

    plt.figure(figsize=(cols * 2, rows * 2))
    plt.suptitle(f'Filtros de la capa {layer_name} de {model_name} CNN', fontsize=16)

    for i in range(num_filters):
        plt.subplot(rows, cols, i + 1)
        # Para visualizar, tomamos el primer canal de entrada de cada filtro.
        # Si in_channels_per_filter > 1 (como en conv2 y conv3 de la densa),
        # esto mostrará cómo el filtro responde a la primera parte de su entrada combinada.
        plt.imshow(filters[i, 0, :, :], cmap='gray')
        plt.title(f'Filtro {i+1}')
        plt.axis('off')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


print("\n--- Visualizando filtros de CNN Estándar (conv1) ---")
visualize_filters('CNN Estándar', cnn_standard, 'conv1')

print("\n--- Visualizando filtros de CNN Estándar (conv2) ---")
visualize_filters('CNN Estándar', cnn_standard, 'conv2')

print("\n--- Visualizando filtros de CNN Estándar (conv3) ---")
visualize_filters('CNN Estándar', cnn_standard, 'conv3')

print("\n--- Visualizando filtros de CNN Densa con Entrada (conv1) ---")
visualize_filters('CNN Densa con Entrada', cnn_dense_input, 'conv1')

print("\n--- Visualizando filtros de CNN Densa con Entrada (conv2) ---")
visualize_filters('CNN Densa con Entrada', cnn_dense_input, 'conv2')

print("\n--- Visualizando filtros de CNN Densa con Entrada (conv3) ---")
visualize_filters('CNN Densa con Entrada', cnn_dense_input, 'conv3')

## Red desde cero con NumPy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Funciones de Activación Comunes ---
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def exponential_activation(x, scale=10.0):
    x_clipped = np.clip(x / scale, -500, 500)
    return np.exp(x_clipped)

def exponential_activation_derivative(x, scale=10.0):
    x_clipped = np.clip(x / scale, -500, 500)
    return np.exp(x_clipped) / scale

# --- Clase de la Neurona Dendrítica Mejorada ---
class NeuronaDendriticaMejorada:
    def __init__(self, num_entradas):
        if num_entradas < 2:
            raise ValueError("La neurona dendrítica requiere al menos 2 entradas para dividir.")

        self.num_entradas = num_entradas
        self.split_idx = num_entradas // 2
        self.pesos_dendrita1 = np.random.rand(self.split_idx) * 0.1
        self.pesos_dendrita2 = np.random.rand(num_entradas - self.split_idx) * 0.1
        self.sesgo = np.random.rand() * 0.1

        self.last_s1_input = None
        self.last_s2_input = None
        self.last_soma_input = None

    def forward(self, x):
        x = np.asarray(x)
        if x.shape[0] != self.num_entradas:
            raise ValueError(f"Las entradas deben tener {self.num_entradas} características. Recibido: {x.shape[0]}")

        x1 = x[:self.split_idx]
        x2 = x[self.split_idx:]

        self.last_s1_input = np.dot(x1, self.pesos_dendrita1)
        self.last_s2_input = np.dot(x2, self.pesos_dendrita2)

        s1_activated = relu(self.last_s1_input)
        s2_activated = exponential_activation(self.last_s2_input)

        self.last_soma_input = s1_activated + s2_activated + self.sesgo

        return sigmoid(self.last_soma_input)

    def train(self, X, y, lr=0.01, epochs=100):
        losses = []
        accuracies = []
        for epoch in range(epochs):
            loss_epoch = 0
            correct_predictions = 0
            for i in range(len(X)):
                xi, yi = X[i], y[i]

                out = self.forward(xi)
                error = yi - out
                loss_epoch += error ** 2

                predicted_class = 1 if out >= 0.5 else 0
                if predicted_class == yi:
                    correct_predictions += 1

                grad_soma_output = sigmoid_derivative(self.last_soma_input) * error
                grad_sesgo = grad_soma_output
                grad_s1_activated = grad_soma_output
                grad_s2_activated = grad_soma_output
                grad_s1_input = relu_derivative(self.last_s1_input) * grad_s1_activated
                grad_s2_input = exponential_activation_derivative(self.last_s2_input) * grad_s2_activated

                x1_part = xi[:self.split_idx]
                x2_part = xi[self.split_idx:]

                grad_pesos_dendrita1 = grad_s1_input * x1_part
                grad_pesos_dendrita2 = grad_s2_input * x2_part

                self.pesos_dendrita1 += lr * grad_pesos_dendrita1
                self.pesos_dendrita2 += lr * grad_pesos_dendrita2
                self.sesgo += lr * grad_sesgo

            losses.append(loss_epoch / len(X))
            accuracies.append(correct_predictions / len(X))
        return losses, accuracies

# --- Clase del Perceptrón Simple Mejorada ---
class PerceptronSimpleMejorado:
    def __init__(self, num_entradas):
        self.pesos = np.random.rand(num_entradas) * 0.1
        self.sesgo = np.random.rand() * 0.1
        self.last_z = None

    def forward(self, x):
        x = np.asarray(x)
        if x.shape[0] != len(self.pesos):
            raise ValueError(f"Las entradas deben tener {len(self.pesos)} características. Recibido: {x.shape[0]}")

        self.last_z = np.dot(x, self.pesos) + self.sesgo
        return sigmoid(self.last_z)

    def train(self, X, y, lr=0.01, epochs=100):
        losses = []
        accuracies = []
        for epoch in range(epochs):
            loss_epoch = 0
            correct_predictions = 0
            for i in range(len(X)):
                xi, yi = X[i], y[i]

                out = self.forward(xi)
                error = yi - out
                loss_epoch += error ** 2

                predicted_class = 1 if out >= 0.5 else 0
                if predicted_class == yi:
                    correct_predictions += 1

                grad_output = sigmoid_derivative(self.last_z) * error

                self.pesos += lr * grad_output * xi
                self.sesgo += lr * grad_output

            losses.append(loss_epoch / len(X))
            accuracies.append(correct_predictions / len(X))
        return losses, accuracies

# --- Datos sintéticos ---
def generar_datos(n=200):
    # Genera datos 2D para visualización más fácil
    X = np.random.rand(n, 2) * 10 - 5 # Puntos entre -5 y 5
    # Problema linealmente separable: y = 1 si x1 + x2 > 0, sino y = 0
    y = (X[:, 0] + X[:, 1] > 0).astype(float)
    return X, y

# --- Configuración y Entrenamiento ---
num_entradas_datos = 2 # Usamos 2 para poder graficar el límite de decisión
learning_rate = 0.1
num_epochs = 500

X_train, y_train = generar_datos(200)
X_test, y_test = generar_datos(100)

print(f"Número de entradas para los modelos: {num_entradas_datos}")
print(f"Tamaño del conjunto de entrenamiento: {len(X_train)} muestras")
print(f"Tamaño del conjunto de prueba: {len(X_test)} muestras\n")

# Inicializar modelos
# Ajustamos num_entradas a 2 para la neurona dendrítica también
neurona_dendritica = NeuronaDendriticaMejorada(num_entradas_datos)
perceptron_simple = PerceptronSimpleMejorado(num_entradas_datos)

# Mostrar número de parámetros (ahora con num_entradas=2, la dendrítica tiene 2+1=3 params)
params_dendritica = (neurona_dendritica.split_idx + (num_entradas_datos - neurona_dendritica.split_idx) + 1)
params_perceptron = num_entradas_datos + 1
print(f"Parámetros Neurona Dendrítica: {params_dendritica} (2 pesos + 1 sesgo)")
print(f"Parámetros Perceptrón Simple:  {params_perceptron} (2 pesos + 1 sesgo)\n")

print("--- Entrenando Neurona Dendrítica Mejorada ---")
loss_dend_train, acc_dend_train = neurona_dendritica.train(X_train, y_train, lr=learning_rate, epochs=num_epochs)
print(f"Pérdida final en entrenamiento (Dendrítica): {loss_dend_train[-1]:.4f}")
print(f"Precisión final en entrenamiento (Dendrítica): {acc_dend_train[-1]:.4f}\n")

print("--- Entrenando Perceptrón Simple Mejorado ---")
loss_perc_train, acc_perc_train = perceptron_simple.train(X_train, y_train, lr=learning_rate, epochs=num_epochs)
print(f"Pérdida final en entrenamiento (Perceptrón): {loss_perc_train[-1]:.4f}")
print(f"Precisión final en entrenamiento (Perceptrón): {acc_perc_train[-1]:.4f}\n")

# --- Evaluación en el conjunto de prueba ---
def evaluate_model(model, X, y):
    predictions = []
    loss_sum = 0
    correct_count = 0
    for xi, yi in zip(X, y):
        out = model.forward(xi)
        predictions.append(out)
        loss_sum += (yi - out) ** 2
        predicted_class = 1 if out >= 0.5 else 0
        if predicted_class == yi:
            correct_count += 1
    avg_loss = loss_sum / len(X)
    accuracy = correct_count / len(X)
    return avg_loss, accuracy

loss_dend_test, acc_dend_test = evaluate_model(neurona_dendritica, X_test, y_test)
loss_perc_test, acc_perc_test = evaluate_model(perceptron_simple, X_test, y_test)

print("--- Resultados en Conjunto de Prueba ---")
print(f"Neurona Dendrítica - Pérdida Test: {loss_dend_test:.4f}, Precisión Test: {acc_dend_test:.4f}")
print(f"Perceptrón Simple - Pérdida Test:  {loss_perc_test:.4f}, Precisión Test:  {acc_perc_test:.4f}")

# --- Visualización de Resultados ---
plt.figure(figsize=(15, 6))

# Gráfico de Pérdida
plt.subplot(1, 2, 1)
plt.plot(loss_dend_train, label="Neurona Dendrítica (Train Loss)")
plt.plot(loss_perc_train, label="Perceptrón Simple (Train Loss)")
plt.axhline(y=loss_dend_test, color='red', linestyle='--', label=f'Dendrítica (Test Loss): {loss_dend_test:.4f}')
plt.axhline(y=loss_perc_test, color='green', linestyle='--', label=f'Perceptrón (Test Loss): {loss_perc_test:.4f}')
plt.xlabel("Épocas")
plt.ylabel("Pérdida (MSE)")
plt.title("Evolución de la Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.tight_layout()

# Gráfico de Precisión
plt.subplot(1, 2, 2)
plt.plot(acc_dend_train, label="Neurona Dendrítica (Train Accuracy)")
plt.plot(acc_perc_train, label="Perceptrón Simple (Train Accuracy)")
plt.axhline(y=acc_dend_test, color='red', linestyle='--', label=f'Dendrítica (Test Acc): {acc_dend_test:.4f}')
plt.axhline(y=acc_perc_test, color='green', linestyle='--', label=f'Perceptrón (Test Acc): {acc_perc_test:.4f}')
plt.xlabel("Épocas")
plt.ylabel("Precisión")
plt.title("Evolución de la Precisión")
plt.legend()
plt.grid(True)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# --- Visualización del Límite de Decisión (solo si num_entradas_datos es 2) ---
if num_entradas_datos == 2:
    plt.figure(figsize=(12, 6))

    # Graficar datos de entrenamiento
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=50, edgecolors='k', alpha=0.7, label='Datos de Entrenamiento')

    # Línea de decisión para Perceptrón Simple
    # w1*x1 + w2*x2 + b = 0  => x2 = (-w1/w2)*x1 - (b/w2)
    x1_vals = np.array([X_train[:, 0].min() - 1, X_train[:, 0].max() + 1])
    if perceptron_simple.pesos[1] != 0:
        x2_vals_perc = (-perceptron_simple.pesos[0] / perceptron_simple.pesos[1]) * x1_vals - (perceptron_simple.sesgo / perceptron_simple.pesos[1])
        plt.plot(x1_vals, x2_vals_perc, 'g--', lw=2, label='Límite Decisión Perceptrón')
    else:
        plt.axvline(x=-perceptron_simple.sesgo / perceptron_simple.pesos[0], color='g', linestyle='--', lw=2, label='Límite Decisión Perceptrón')

    # Para la Neurona Dendrítica es más complejo dibujar una sola "línea"
    # debido a sus múltiples activaciones y pesos internos, lo que resulta
    # en una frontera de decisión potencialmente no lineal aunque el problema sea linealmente separable.
    # Aquí, vamos a aproximar visualizando las predicciones en una malla
    x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    grid_points = np.c_[xx.ravel(), yy.ravel()]

    # Predicciones de la Neurona Dendrítica en la malla
    Z_dend = np.array([neurona_dendritica.forward(p) for p in grid_points])
    Z_dend = Z_dend.reshape(xx.shape)
    plt.contourf(xx, yy, Z_dend, levels=[0, 0.5, 1], colors=['lightblue', 'lightcoral'], alpha=0.4, zorder=0)
    plt.contour(xx, yy, Z_dend, levels=[0.5], colors='red', linestyles='--', linewidths=2, label='Límite Decisión Dendrítica')

    plt.title('Límites de Decisión (Perceptrón vs. Neurona Dendrítica)')
    plt.xlabel('Característica 1')
    plt.ylabel('Característica 2')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
import gradio as gr
import plotly.graph_objects as go
import numpy as np

def generate_3d_spiral(num_points):
    t = np.linspace(0, 20 * np.pi, num_points)
    x = np.sin(t)
    y = np.cos(t)
    z = t

    fig = go.Figure(data=[go.Scatter3d(x=x, y=y, z=z, mode='lines+markers')])
    fig.update_layout(title=f'Espiral 3D con {num_points} puntos', margin=dict(l=0, r=0, b=0, t=40))
    return fig # Gradio sabe cómo renderizar objetos de Plotly

iface = gr.Interface(
    fn=generate_3d_spiral,
    inputs=gr.Slider(100, 10000, value=500, label="Número de Puntos"),
    outputs=gr.Plot(), # Usamos gr.Plot para mostrar objetos de Plotly
    title="Generador de Espiral 3D en Gradio"
)

iface.launch(inbrowser=True, share=True)

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Datos de ejemplo: superficie
x = np.outer(np.linspace(-2, 2, 30), np.ones(30))
y = x.copy().T
z = np.cos(x**2 + y**2)

fig = go.Figure(data=[go.Surface(x=x, y=y, z=z)])

fig.update_layout(title='Superficie 3D Interactiva con Plotly', autosize=False,
                  width=500, height=500,
                  margin=dict(l=65, r=50, b=65, t=90))

fig.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Datos de ejemplo
data = {'A': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'B': [10, 8, 6, 4, 2, 1, 3, 5, 7, 9],
        'C': ['X', 'Y', 'X', 'Y', 'X', 'Y', 'X', 'Y', 'X', 'Y']}
df = pd.DataFrame(data)

plt.figure(figsize=(8, 5))
sns.scatterplot(x='A', y='B', hue='C', size='B', sizes=(50, 500), palette='viridis', data=df)
plt.title('Gráfico de Dispersión con Seaborn (¡Bello!)', fontsize=15)
plt.xlabel('Eje A')
plt.ylabel('Eje B')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import altair as alt
import pandas as pd

# Datos de ejemplo
data = {'A': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        'B': [10, 8, 6, 4, 2, 1, 3, 5, 7, 9],
        'C': ['X', 'Y', 'X', 'Y', 'X', 'Y', 'X', 'Y', 'X', 'Y']}
df = pd.DataFrame(data)

chart = alt.Chart(df).mark_circle().encode(
    x='A',
    y='B',
    color='C',
    size=alt.Size('B', legend=alt.Legend(title="Tamaño de B")),
    tooltip=['A', 'B', 'C'] # Información al pasar el mouse
).properties(
    title='Gráfico de Dispersión Interactivo con Altair (¡Chingón y Bello!)'
).interactive() # Permite zoom y paneo

chart.display() # Para mostrar en Colab/Jupyter

In [ ]:
import gradio as gr
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import Progress, BarColumn, TextColumn, TimeRemainingColumn
from rich.text import Text
import math

console = Console(record=True) # Capturamos la salida de rich

def entrenar_red_rmb(epochs=2000):
    console.print(Panel("[bold yellow]--- Inicializando y Entrenando Red con Sesgo de Memoria Residual (RMB) ---[/]", expand=False))
    console.print(f"Número de parámetros en Red RMB: [bold green]3265[/]")
    # ... (código de entrenamiento simulado)
    console.print(f"Pérdida final (Red RMB): [bold blue]0.123456[/]")
    return console.export_html() # Exportamos la salida de rich como HTML

def gradio_app():
    iface = gr.Interface(
        fn=entrenar_red_rmb,
        inputs="number",
        outputs="html", # Salida como HTML
        title="Entrenamiento de Red RMB con Rich",
        description="Este ejemplo muestra cómo integrar la salida formateada con Rich en una interfaz de Gradio."
    )
    iface.launch()

if __name__ == "__main__":
    gradio_app()

In [ ]:
from tqdm import tqdm
import time

for i in tqdm(range(2000), desc="Entrenando..."):
    # Tu código de entrenamiento
    time.sleep(0.001)

#biolagicamwnte inspirados

In [ ]:
import numpy as np

class NeuronaDendritica:
    def __init__(self, num_entradas):
        # Cada "dendrita" tendrá sus propios pesos
        # y una función de activación específica.
        # Aquí, asumimos 2 dendritas para simplificar.
        if num_entradas < 2:
            raise ValueError("Se necesitan al menos 2 entradas para este modelo dendrítico.")

        self.num_entradas = num_entradas
        self.pesos_dendrita1 = np.random.rand(num_entradas // 2) * 0.1 # Pesos para la primera mitad de entradas
        self.pesos_dendrita2 = np.random.rand(num_entradas - (num_entradas // 2)) * 0.1 # Pesos para el resto
        self.sesgo = np.random.rand() * 0.1

    def activacion_dendrita1(self, x_dendrita):
        # Función de activación no lineal para la dendrita 1 (ej. ReLU)
        return np.maximum(0, np.dot(x_dendrita, self.pesos_dendrita1))

    def activacion_dendrita2(self, x_dendrita):
        # Función de activación más fuerte/exponencial para la dendrita 2
        return np.exp(np.dot(x_dendrita, self.pesos_dendrita2) / 10.0) # Escalado para evitar números muy grandes

    def forward(self, entradas):
        if len(entradas) != self.num_entradas:
            raise ValueError(f"Se esperaban {self.num_entradas} entradas, pero se recibieron {len(entradas)}")

        # Dividir las entradas entre las "dendritas"
        entradas_dendrita1 = entradas[:self.num_entradas // 2]
        entradas_dendrita2 = entradas[self.num_entradas // 2:]

        # Aplicar la activación específica a cada dendrita
        señal_dendrita1 = self.activacion_dendrita1(entradas_dendrita1)
        señal_dendrita2 = self.activacion_dendrita2(entradas_dendrita2)

        # Sumar las señales dendríticas y el sesgo para obtener la salida del "soma"
        # y aplicar una activación final (ej. sigmoide para salida entre 0 y 1)
        salida_soma = señal_dendrita1 + señal_dendrita2 + self.sesgo
        return 1 / (1 + np.exp(-salida_soma)) # Sigmoide

# --- Ejemplo de uso ---
# Crear una neurona con 4 entradas
neurona = NeuronaDendritica(num_entradas=4)

# Probar con algunas entradas
entradas_ejemplo1 = np.array([0.5, 0.2, 0.8, 0.1])
salida1 = neurona.forward(entradas_ejemplo1)
print(f"Salida para entradas {entradas_ejemplo1}: {salida1:.4f}")

entradas_ejemplo2 = np.array([0.1, 0.1, 0.1, 0.9]) # Donde la segunda "dendrita" podría tener más impacto
salida2 = neurona.forward(entradas_ejemplo2)
print(f"Salida para entradas {entradas_ejemplo2}: {salida2:.4f}")

entradas_ejemplo3 = np.array([0.9, 0.9, 0.1, 0.1]) # Donde la primera "dendrita" podría tener más impacto
salida3 = neurona.forward(entradas_ejemplo3)
print(f"Salida para entradas {entradas_ejemplo3}: {salida3:.4f}")

In [ ]:
import numpy as np

class NeuronaDendritica:
    def __init__(self, num_entradas):
        if num_entradas < 2:
            raise ValueError("Se necesitan al menos 2 entradas para este modelo dendrítico.")

        self.num_entradas = num_entradas
        # Asignamos pesos a dos "dendritas" simuladas
        self.pesos_dendrita1 = np.random.rand(num_entradas // 2) * 0.1
        self.pesos_dendrita2 = np.random.rand(num_entradas - (num_entradas // 2)) * 0.1
        self.sesgo = np.random.rand() * 0.1

    def activacion_dendrita1(self, x_dendrita):
        # Una función de activación para la primera "dendrita"
        return np.maximum(0, np.dot(x_dendrita, self.pesos_dendrita1)) # ReLU

    def activacion_dendrita2(self, x_dendrita):
        # Una función de activación diferente para la segunda "dendrita"
        return np.exp(np.dot(x_dendrita, self.pesos_dendrita2) / 10.0)

    def forward(self, entradas):
        if len(entradas) != self.num_entradas:
            raise ValueError(f"Se esperaban {self.num_entradas} entradas, pero se recibieron {len(entradas)}")

        # Dividir las entradas entre las "dendritas"
        entradas_dendrita1 = entradas[:self.num_entradas // 2]
        entradas_dendrita2 = entradas[self.num_entradas // 2:]

        # Aplicar la activación específica a cada dendrita
        señal_dendrita1 = self.activacion_dendrita1(entradas_dendrita1)
        señal_dendrita2 = self.activacion_dendrita2(entradas_dendrita2)

        # Sumar las señales dendríticas y el sesgo para obtener la salida del "soma"
        salida_soma = señal_dendrita1 + señal_dendrita2 + self.sesgo
        return 1 / (1 + np.exp(-salida_soma)) # Sigmoide final

class PerceptronSimple:
    def __init__(self, num_entradas):
        # Un único conjunto de pesos para todas las entradas
        self.pesos = np.random.rand(num_entradas) * 0.1
        self.sesgo = np.random.rand() * 0.1

    def forward(self, entradas):
        if len(entradas) != len(self.pesos):
            raise ValueError(f"Se esperaban {len(self.pesos)} entradas, pero se recibieron {len(entradas)}")

        # Suma ponderada de las entradas
        suma_ponderada = np.dot(entradas, self.pesos) + self.sesgo
        # Aplicación de una función de activación (ej. sigmoide)
        return 1 / (1 + np.exp(-suma_ponderada)) # Sigmoide

# --- Configuración y Prueba ---
num_entradas_global = 4

print("--- Probando Neurona Dendrítica Simplificada ---")
neurona_dendritica = NeuronaDendritica(num_entradas=num_entradas_global)
entradas_dendritica = np.array([0.5, 0.2, 0.8, 0.1])
salida_dendritica = neurona_dendritica.forward(entradas_dendritica)
print(f"Entradas: {entradas_dendritica}")
print(f"Salida (Neurona Dendrítica): {salida_dendritica:.4f}\n")

print("--- Probando Perceptrón Simple ---")
perceptron_simple = PerceptronSimple(num_entradas=num_entradas_global)
entradas_perceptron = np.array([0.5, 0.2, 0.8, 0.1])
salida_perceptron = perceptron_simple.forward(entradas_perceptron)
print(f"Entradas: {entradas_perceptron}")
print(f"Salida (Perceptrón Simple): {salida_perceptron:.4f}\n")

# Para una comparación más detallada, podrías intentar:
# 1. Fijar las semillas aleatorias para una inicialización idéntica.
# 2. Entrenar ambas neuronas en un pequeño conjunto de datos para ver cuál aprende mejor.
#    (Esto requeriría implementar un bucle de entrenamiento y una función de pérdida,
#     lo cual excede la simple comparación de "forward pass").

# --- Análisis de Complejidad (Parámetros) ---
print("--- Análisis de Parámetros ---")
# Parámetros de Neurona Dendrítica:
# pesos_dendrita1 + pesos_dendrita2 + sesgo
params_dendritica = (num_entradas_global // 2) + (num_entradas_global - (num_entradas_global // 2)) + 1
print(f"Número de parámetros en Neurona Dendrítica: {params_dendritica} (pesos y sesgo)")

# Parámetros de Perceptrón Simple:
# pesos + sesgo
params_perceptron = num_entradas_global + 1
print(f"Número de parámetros en Perceptrón Simple:  {params_perceptron} (pesos y sesgo)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fijamos la semilla para reproducibilidad
np.random.seed(42)

# Clase de la neurona dendrítica
class NeuronaDendritica:
    def __init__(self, num_entradas):
        self.num_entradas = num_entradas
        self.pesos_dendrita1 = np.random.rand(num_entradas // 2) * 0.1
        self.pesos_dendrita2 = np.random.rand(num_entradas - (num_entradas // 2)) * 0.1
        self.sesgo = np.random.rand() * 0.1

    def activacion_dendrita1(self, x):
        return np.maximum(0, np.dot(x, self.pesos_dendrita1))

    def activacion_dendrita2(self, x):
        return np.exp(np.dot(x, self.pesos_dendrita2) / 10.0)

    def forward(self, x):
        x1 = x[:self.num_entradas // 2]
        x2 = x[self.num_entradas // 2:]
        s1 = self.activacion_dendrita1(x1)
        s2 = self.activacion_dendrita2(x2)
        return 1 / (1 + np.exp(-(s1 + s2 + self.sesgo)))

    def train(self, X, y, lr=0.01, epochs=100):
        losses = []
        for _ in range(epochs):
            loss_epoch = 0
            for xi, yi in zip(X, y):
                out = self.forward(xi)
                error = yi - out
                grad = out * (1 - out) * error

                # backprop
                d1 = xi[:self.num_entradas // 2]
                d2 = xi[self.num_entradas // 2:]
                self.pesos_dendrita1 += lr * grad * d1
                self.pesos_dendrita2 += lr * grad * d2
                self.sesgo += lr * grad

                loss_epoch += error ** 2
            losses.append(loss_epoch / len(X))
        return losses

# Clase del perceptrón simple
class PerceptronSimple:
    def __init__(self, num_entradas):
        self.pesos = np.random.rand(num_entradas) * 0.1
        self.sesgo = np.random.rand() * 0.1

    def forward(self, x):
        z = np.dot(x, self.pesos) + self.sesgo
        return 1 / (1 + np.exp(-z))

    def train(self, X, y, lr=0.01, epochs=100):
        losses = []
        for _ in range(epochs):
            loss_epoch = 0
            for xi, yi in zip(X, y):
                out = self.forward(xi)
                error = yi - out
                grad = out * (1 - out) * error

                self.pesos += lr * grad * xi
                self.sesgo += lr * grad

                loss_epoch += error ** 2
            losses.append(loss_epoch / len(X))
        return losses

# Datos sintéticos
def generar_datos(n=200):
    X = np.random.rand(n, 4)
    y = (X[:, 0] + X[:, 1] * 2 - X[:, 2] + 0.5 * X[:, 3] > 1.5).astype(float)
    return X, y

# Entrenamiento y visualización
X_train, y_train = generar_datos(200)
X_test, y_test = generar_datos(100)

neurona = NeuronaDendritica(4)
perceptron = PerceptronSimple(4)

loss_dend_train = neurona.train(X_train, y_train, epochs=100)
loss_perc_train = perceptron.train(X_train, y_train, epochs=100)

# Evaluar en test
loss_dend_test = [np.mean([(yi - neurona.forward(xi)) ** 2 for xi, yi in zip(X_test, y_test)]) for _ in range(100)]
loss_perc_test = [np.mean([(yi - perceptron.forward(xi)) ** 2 for xi, yi in zip(X_test, y_test)]) for _ in range(100)]

# Graficar pérdidas
plt.figure(figsize=(10, 6))
plt.plot(loss_dend_train, label="Neurona Dendrítica (Train)")
plt.plot(loss_perc_train, label="Perceptrón Simple (Train)")
plt.plot(loss_dend_test, '--', label="Neurona Dendrítica (Test)")
plt.plot(loss_perc_test, '--', label="Perceptrón Simple (Test)")
plt.xlabel("Épocas")
plt.ylabel("Pérdida (MSE)")
plt.title("Comparación de Entrenamiento y Generalización")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Red desde cero con NumPy (variante)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fijamos la semilla para reproducibilidad
np.random.seed(42)

# --- Funciones de Activación Comunes ---
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def exponential_activation(x, scale=10.0):
    # Evita overflow para números grandes, usando np.clip
    x_clipped = np.clip(x / scale, -500, 500) # Limita el rango para np.exp
    return np.exp(x_clipped)

def exponential_activation_derivative(x, scale=10.0):
    x_clipped = np.clip(x / scale, -500, 500)
    return np.exp(x_clipped) / scale


# --- Clase de la Neurona Dendrítica Mejorada ---
class NeuronaDendriticaMejorada:
    def __init__(self, num_entradas):
        if num_entradas < 2:
            raise ValueError("La neurona dendrítica requiere al menos 2 entradas para dividir.")

        self.num_entradas = num_entradas
        # Dividimos las entradas de manera más flexible
        self.split_idx = num_entradas // 2
        self.pesos_dendrita1 = np.random.rand(self.split_idx) * 0.1
        self.pesos_dendrita2 = np.random.rand(num_entradas - self.split_idx) * 0.1
        self.sesgo = np.random.rand() * 0.1

        # Almacenamos valores intermedios para backpropagation
        self.last_s1_input = None
        self.last_s2_input = None
        self.last_soma_input = None

    def forward(self, x):
        # Aseguramos que x sea un array de numpy
        x = np.asarray(x)
        if x.shape[0] != self.num_entradas:
            raise ValueError(f"Las entradas deben tener {self.num_entradas} características. Recibido: {x.shape[0]}")

        x1 = x[:self.split_idx]
        x2 = x[self.split_idx:]

        # Cálculo de las entradas ponderadas para cada dendrita
        self.last_s1_input = np.dot(x1, self.pesos_dendrita1)
        self.last_s2_input = np.dot(x2, self.pesos_dendrita2)

        # Aplicación de activaciones dendríticas
        s1_activated = relu(self.last_s1_input)
        s2_activated = exponential_activation(self.last_s2_input)

        # Suma en el "soma"
        self.last_soma_input = s1_activated + s2_activated + self.sesgo

        # Activación final del soma
        return sigmoid(self.last_soma_input)

    def train(self, X, y, lr=0.01, epochs=100):
        losses = []
        accuracies = []
        for epoch in range(epochs):
            loss_epoch = 0
            correct_predictions = 0
            for i in range(len(X)):
                xi, yi = X[i], y[i]

                # Paso hacia adelante (forward pass)
                out = self.forward(xi)

                # Cálculo del error y la pérdida (MSE)
                error = yi - out
                loss_epoch += error ** 2

                # Clasificación para precisión
                predicted_class = 1 if out >= 0.5 else 0
                if predicted_class == yi:
                    correct_predictions += 1

                # Backpropagation:
                # 1. Gradiente de la activación final (sigmoide)
                grad_soma_output = sigmoid_derivative(self.last_soma_input) * error

                # 2. Gradiente del sesgo
                grad_sesgo = grad_soma_output

                # 3. Gradientes de las señales de las dendritas
                grad_s1_activated = grad_soma_output
                grad_s2_activated = grad_soma_output

                # 4. Gradientes de las entradas ponderadas de las dendritas
                grad_s1_input = relu_derivative(self.last_s1_input) * grad_s1_activated
                grad_s2_input = exponential_activation_derivative(self.last_s2_input) * grad_s2_activated

                # 5. Gradientes de los pesos de las dendritas
                x1_part = xi[:self.split_idx]
                x2_part = xi[self.split_idx:]

                grad_pesos_dendrita1 = grad_s1_input * x1_part
                grad_pesos_dendrita2 = grad_s2_input * x2_part

                # Actualización de pesos y sesgos
                self.pesos_dendrita1 += lr * grad_pesos_dendrita1
                self.pesos_dendrita2 += lr * grad_pesos_dendrita2
                self.sesgo += lr * grad_sesgo

            losses.append(loss_epoch / len(X))
            accuracies.append(correct_predictions / len(X))
        return losses, accuracies

# --- Clase del Perceptrón Simple Mejorada ---
class PerceptronSimpleMejorado:
    def __init__(self, num_entradas):
        self.pesos = np.random.rand(num_entradas) * 0.1
        self.sesgo = np.random.rand() * 0.1
        self.last_z = None # Almacenar la suma ponderada para el backprop

    def forward(self, x):
        x = np.asarray(x)
        if x.shape[0] != len(self.pesos):
            raise ValueError(f"Las entradas deben tener {len(self.pesos)} características. Recibido: {x.shape[0]}")

        self.last_z = np.dot(x, self.pesos) + self.sesgo
        return sigmoid(self.last_z)

    def train(self, X, y, lr=0.01, epochs=100):
        losses = []
        accuracies = []
        for epoch in range(epochs):
            loss_epoch = 0
            correct_predictions = 0
            for i in range(len(X)):
                xi, yi = X[i], y[i]

                # Paso hacia adelante
                out = self.forward(xi)

                # Cálculo del error y la pérdida
                error = yi - out
                loss_epoch += error ** 2

                # Clasificación para precisión
                predicted_class = 1 if out >= 0.5 else 0
                if predicted_class == yi:
                    correct_predictions += 1

                # Backpropagation
                grad_output = sigmoid_derivative(self.last_z) * error

                self.pesos += lr * grad_output * xi
                self.sesgo += lr * grad_output

            losses.append(loss_epoch / len(X))
            accuracies.append(correct_predictions / len(X))
        return losses, accuracies

# --- Datos sintéticos (mismos que los originales) ---
def generar_datos(n=200):
    X = np.random.rand(n, 4)
    # Ejemplo de un problema linealmente separable para empezar
    y = (X[:, 0] + X[:, 1] * 2 - X[:, 2] + 0.5 * X[:, 3] > 1.5).astype(float)
    return X, y

# --- Configuración y Entrenamiento ---
num_entradas_datos = 4
learning_rate = 0.1 # Aumentado para una convergencia más rápida
num_epochs = 500 # Aumentado para ver mejor la convergencia

X_train, y_train = generar_datos(200)
X_test, y_test = generar_datos(100)

print(f"Número de entradas para los modelos: {num_entradas_datos}")
print(f"Tamaño del conjunto de entrenamiento: {len(X_train)} muestras")
print(f"Tamaño del conjunto de prueba: {len(X_test)} muestras\n")


# Inicializar modelos
neurona_dendritica = NeuronaDendriticaMejorada(num_entradas_datos)
perceptron_simple = PerceptronSimpleMejorado(num_entradas_datos)

# Mostrar número de parámetros
params_dendritica = (neurona_dendritica.split_idx + (num_entradas_datos - neurona_dendritica.split_idx) + 1)
params_perceptron = num_entradas_datos + 1
print(f"Parámetros Neurona Dendrítica: {params_dendritica}")
print(f"Parámetros Perceptrón Simple:  {params_perceptron}\n")


print("--- Entrenando Neurona Dendrítica Mejorada ---")
loss_dend_train, acc_dend_train = neurona_dendritica.train(X_train, y_train, lr=learning_rate, epochs=num_epochs)
print(f"Pérdida final en entrenamiento (Dendrítica): {loss_dend_train[-1]:.4f}")
print(f"Precisión final en entrenamiento (Dendrítica): {acc_dend_train[-1]:.4f}\n")

print("--- Entrenando Perceptrón Simple Mejorado ---")
loss_perc_train, acc_perc_train = perceptron_simple.train(X_train, y_train, lr=learning_rate, epochs=num_epochs)
print(f"Pérdida final en entrenamiento (Perceptrón): {loss_perc_train[-1]:.4f}")
print(f"Precisión final en entrenamiento (Perceptrón): {acc_perc_train[-1]:.4f}\n")


# --- Evaluación en el conjunto de prueba ---
def evaluate_model(model, X, y):
    predictions = []
    loss_sum = 0
    correct_count = 0
    for xi, yi in zip(X, y):
        out = model.forward(xi)
        predictions.append(out)
        loss_sum += (yi - out) ** 2
        predicted_class = 1 if out >= 0.5 else 0
        if predicted_class == yi:
            correct_count += 1

    avg_loss = loss_sum / len(X)
    accuracy = correct_count / len(X)
    return avg_loss, accuracy

loss_dend_test, acc_dend_test = evaluate_model(neurona_dendritica, X_test, y_test)
loss_perc_test, acc_perc_test = evaluate_model(perceptron_simple, X_test, y_test)

print("--- Resultados en Conjunto de Prueba ---")
print(f"Neurona Dendrítica - Pérdida Test: {loss_dend_test:.4f}, Precisión Test: {acc_dend_test:.4f}")
print(f"Perceptrón Simple - Pérdida Test:  {loss_perc_test:.4f}, Precisión Test:  {acc_perc_test:.4f}")


# --- Visualización de Resultados ---
plt.figure(figsize=(15, 6))

# Gráfico de Pérdida
plt.subplot(1, 2, 1)
plt.plot(loss_dend_train, label="Neurona Dendrítica (Train Loss)")
plt.plot(loss_perc_train, label="Perceptrón Simple (Train Loss)")
plt.axhline(y=loss_dend_test, color='red', linestyle='--', label=f'Dendrítica (Test Loss): {loss_dend_test:.4f}')
plt.axhline(y=loss_perc_test, color='green', linestyle='--', label=f'Perceptrón (Test Loss): {loss_perc_test:.4f}')
plt.xlabel("Épocas")
plt.ylabel("Pérdida (MSE)")
plt.title("Evolución de la Pérdida")
plt.legend()
plt.grid(True)
plt.yscale('log') # Útil para ver la convergencia

# Gráfico de Precisión
plt.subplot(1, 2, 2)
plt.plot(acc_dend_train, label="Neurona Dendrítica (Train Accuracy)")
plt.plot(acc_perc_train, label="Perceptrón Simple (Train Accuracy)")
plt.axhline(y=acc_dend_test, color='red', linestyle='--', label=f'Dendrítica (Test Acc): {acc_dend_test:.4f}')
plt.axhline(y=acc_perc_test, color='green', linestyle='--', label=f'Perceptrón (Test Acc): {acc_perc_test:.4f}')
plt.xlabel("Épocas")
plt.ylabel("Precisión")
plt.title("Evolución de la Precisión")
plt.legend()
plt.grid(True)
plt.ylim(0, 1) # La precisión va de 0 a 1
plt.tight_layout()
plt.show()

#NeuronaEstadoInterno

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# --- 1. Dataset de Onda Sinusoidal ---
def generar_serie_sinusoidal(n_puntos, ventana_entrada):
    x = np.linspace(0, 20 * np.pi, n_puntos)
    y = np.sin(x) + 0.1 * np.random.randn(n_puntos)  # ruido leve

    X, Y = [], []
    for i in range(len(y) - ventana_entrada):
        X.append(y[i:(i + ventana_entrada)])
        Y.append(y[i + ventana_entrada])
    return np.array(X), np.array(Y)

N_PUNTOS = 1000
VENTANA_ENTRADA = 10
X_data, Y_data = generar_serie_sinusoidal(N_PUNTOS, VENTANA_ENTRADA)

# Entrenamiento / Prueba
split_ratio = 0.8
split_index = int(len(X_data) * split_ratio)
X_train, Y_train = X_data[:split_index], Y_data[:split_index]
X_test, Y_test = X_data[split_index:], Y_data[split_index:]

# --- 2. Capa con Estado Interno Adaptativo ---
class NeuronaEstadoInterno(tf.keras.layers.Layer):
    def __init__(self, unidades):
        super().__init__()
        self.unidades = unidades

    def build(self, input_shape):
        self.w_input = self.add_weight(shape=(input_shape[-1], self.unidades),
                                       initializer='glorot_uniform', trainable=True)
        self.w_state = self.add_weight(shape=(self.unidades, self.unidades),
                                       initializer='orthogonal', trainable=True)
        self.b = self.add_weight(shape=(self.unidades,), initializer='zeros', trainable=True)
        self.initial_state = self.add_weight(shape=(self.unidades,), initializer='zeros', trainable=True)

    def call(self, inputs):
        # Inicializar el estado interno como una variable que persiste durante la llamada
        state = tf.tile(tf.expand_dims(self.initial_state, 0), [tf.shape(inputs)[0], 1])
        state = tf.nn.tanh(tf.matmul(inputs, self.w_input) + tf.matmul(state, self.w_state) + self.b)
        return state

# --- 3. Modelos ---

# Modelo convencional
modelo_denso = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(VENTANA_ENTRADA,)),
    tf.keras.layers.Dense(1)
])
modelo_denso.compile(optimizer='adam', loss='mse')

# Modelo con estado adaptativo
modelo_estado = tf.keras.Sequential([
    NeuronaEstadoInterno(32),
    tf.keras.layers.Dense(1) # Capa de salida
])
modelo_estado.compile(optimizer='adam', loss='mse')

# --- 4. Entrenamiento ---
EPOCHS = 100 # Aumentamos las épocas para mejor comparación

print("Entrenando Modelo Denso (Convencional)...")
history_denso = modelo_denso.fit(X_train, Y_train, epochs=EPOCHS, verbose=0, validation_data=(X_test, Y_test))

print("Entrenando Modelo con Estado Interno Adaptativo...")
history_estado = modelo_estado.fit(X_train, Y_train, epochs=EPOCHS, verbose=0, validation_data=(X_test, Y_test))

# --- 5. Evaluación ---
loss_denso = modelo_denso.evaluate(X_test, Y_test, verbose=0)
loss_estado = modelo_estado.evaluate(X_test, Y_test, verbose=0)

print(f"\n--- Resultados ---")
print(f"MSE del Modelo Denso en datos de prueba: {loss_denso:.4f}")
print(f"MSE del Modelo con Estado Adaptativo en datos de prueba: {loss_estado:.4f}")

# --- 6. Visualización ---
plt.figure(figsize=(12, 6))
plt.plot(history_denso.history['loss'], label='Denso Train Loss')
plt.plot(history_denso.history['val_loss'], label='Denso Validation Loss')
plt.plot(history_estado.history['loss'], label='Estado Adaptativo Train Loss')
plt.plot(history_estado.history['val_loss'], label='Estado Adaptativo Validation Loss')
plt.title('Pérdida (Loss) durante el Entrenamiento')
plt.xlabel('Época')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.legend()
plt.grid(True)
plt.show()

predicciones_denso = modelo_denso.predict(X_test)
predicciones_estado = modelo_estado.predict(X_test)

plt.figure(figsize=(12, 6))
plt.plot(Y_test, label='Valores Reales')
plt.plot(predicciones_denso, label='Predicciones Denso', alpha=0.7)
plt.plot(predicciones_estado, label='Predicciones Estado Adaptativo', alpha=0.7)
plt.title('Predicciones de Serie Temporal en Datos de Prueba')
plt.xlabel('Punto en la Serie')
plt.ylabel('Valor')
plt.legend()
plt.grid(True)
plt.show()

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

HIPERPARAMETROS = {
    "random_seed": 0,
    "num_samples": 300,
    "input_features": 1,
    "hidden_features": 20,
    "output_features": 1,
    "learning_rate": 0.01,
    "epochs": 500,
    "tag_threshold": 0.01,
    "tag_decay_rate": 0.1,
    "consolidation_strength": 0.005,
    "consolidation_period": 10,
    "early_stop_patience": 50,
    "model_save_path_tagging": "best_model_tagging.pth", # Key para el modelo Tagging
    "model_save_path_standard": "best_model_standard.pth" # Key para el modelo Standard
}

torch.manual_seed(HIPERPARAMETROS["random_seed"])
np.random.seed(HIPERPARAMETROS["random_seed"])

X = torch.linspace(-3, 3, HIPERPARAMETROS["num_samples"]).unsqueeze(1)
y = X.pow(3) - X + 0.5 * torch.randn(X.size())

class CapaSynapticTagging(nn.Module):
    def __init__(self, in_features, out_features, params):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.pesos = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.tag_threshold = params["tag_threshold"]
        self.tag_decay_rate = params["tag_decay_rate"]
        self.consolidation_strength = params["consolidation_strength"]
        self.consolidation_period = params["consolidation_period"]

        self.register_buffer('tags', torch.zeros_like(self.pesos))
        self.register_buffer('tag_directions', torch.zeros_like(self.pesos))
        self.current_epoch = 0

    def forward(self, x):
        return F.linear(x, self.pesos, self.bias)

    def _update_tags_and_decay(self, current_grad_w):
        with torch.no_grad():
            self.tags.mul_(1 - self.tag_decay_rate)
            significant_grad_mask = torch.abs(current_grad_w) > self.tag_threshold
            self.tags[significant_grad_mask] = 1.0
            self.tag_directions[significant_grad_mask] = torch.sign(current_grad_w[significant_grad_mask])

    def apply_consolidation(self):
        self.current_epoch += 1
        if self.current_epoch % self.consolidation_period != 0:
            return

        if self.pesos.grad is None:
            return

        with torch.no_grad():
            grad_w_main = self.pesos.grad
            consolidation_mask = (self.tags > 0.1) & (torch.sign(grad_w_main) == self.tag_directions)
            self.pesos[consolidation_mask] -= self.consolidation_strength * grad_w_main[consolidation_mask]


class RedConSynapticTagging(nn.Module):
    def __init__(self, params):
        super().__init__()
        self.capa1 = CapaSynapticTagging(params["input_features"], params["hidden_features"], params)
        self.act = nn.Tanh()
        self.capa2 = nn.Linear(params["hidden_features"], params["output_features"])

    def forward(self, x):
        h = self.capa1(x)
        return self.capa2(self.act(h))

class RedEstandar(nn.Module):
    def __init__(self, params):
        super().__init__()
        self.capa1 = nn.Linear(params["input_features"], params["hidden_features"])
        self.act = nn.Tanh()
        self.capa2 = nn.Linear(params["hidden_features"], params["output_features"])

    def forward(self, x):
        h = self.capa1(x)
        return self.capa2(self.act(h))

def train_model(model, optimizer, X_data, y_data, params, model_key): # Use a simple key
    LOSS_FN = nn.MSELoss()
    history_loss = []
    best_loss = float('inf')
    epochs_no_improve = 0

    # Print based on the key, or add another display name parameter if needed
    print(f"--- Entrenando Modelo asociado a la clave '{model_key}' ---")

    for epoch in range(params["epochs"]):
        y_pred = model(X_data)
        loss = LOSS_FN(y_pred, y_data)
        history_loss.append(loss.item())

        optimizer.zero_grad()
        loss.backward()

        if isinstance(model, RedConSynapticTagging):
            if model.capa1.pesos.grad is not None:
                model.capa1._update_tags_and_decay(model.capa1.pesos.grad)
            model.capa1.apply_consolidation()

        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            epochs_no_improve = 0
            # Use the provided model_key directly
            torch.save(model.state_dict(), params[model_key])
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= params["early_stop_patience"]:
                print(f"Early stopping en Época {epoch}. Pérdida no mejoró por {params['early_stop_patience']} épocas.")
                break

        if epoch % 50 == 0:
            print(f"Modelo asociado a '{model_key}' - Época {epoch} - Pérdida: {loss.item():.6f}")

    return history_loss

if __name__ == "__main__":
    modelo_tagging = RedConSynapticTagging(HIPERPARAMETROS)
    opt_tagging = torch.optim.Adam(modelo_tagging.parameters(), lr=HIPERPARAMETROS["learning_rate"])
    # Pass the exact key from HIPERPARAMETROS
    historial_loss_tagging = train_model(modelo_tagging, opt_tagging, X, y, HIPERPARAMETROS, "model_save_path_tagging")

    modelo_estandar = RedEstandar(HIPERPARAMETROS)
    opt_estandar = torch.optim.Adam(modelo_estandar.parameters(), lr=HIPERPARAMETROS["learning_rate"])
    # Pass the exact key from HIPERPARAMETROS
    historial_loss_estandar = train_model(modelo_estandar, opt_estandar, X, y, HIPERPARAMETROS, "model_save_path_standard")

    with torch.no_grad():
        # Load state dict using the exact keys
        modelo_tagging.load_state_dict(torch.load(HIPERPARAMETROS["model_save_path_tagging"], weights_only=True))
        modelo_tagging.eval()
        pred_tagging = modelo_tagging(X).numpy()

        modelo_estandar.load_state_dict(torch.load(HIPERPARAMETROS["model_save_path_standard"], weights_only=True))
        modelo_estandar.eval()
        pred_estandar = modelo_estandar(X).numpy()

    X_np = X.numpy()
    y_np = y.numpy()

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 2, 1)
    plt.title("Comparación de Predicciones")
    plt.plot(X_np, y_np, label="Valores Reales", color='blue', alpha=0.7)
    plt.plot(X_np, pred_tagging, label="Predicción con Tagging", color='red', linestyle='--')
    plt.plot(X_np, pred_estandar, label="Predicción Estándar", color='green', linestyle=':')
    plt.legend()
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.title("Evolución de la Pérdida (MSE)")
    plt.plot(historial_loss_tagging, label="Pérdida con Tagging", color='red')
    plt.plot(historial_loss_estandar, label="Pérdida Estándar", color='green')
    plt.xlabel("Época")
    plt.ylabel("Pérdida")
    plt.legend()
    plt.grid(True)
    plt.yscale('log')
    plt.tight_layout()
    plt.show()

    print(f"\n--- Resumen de Pérdidas Finales ---")
    print(f"Pérdida final (Red con Synaptic Tagging): {historial_loss_tagging[-1]:.6f}")
    print(f"Pérdida final (Red Estándar):             {historial_loss_estandar[-1]:.6f}")

#SimpleOptiNet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

# --- Hiperparámetros Globales ---
HIPERPARAMETROS = {
    "num_neuronas_ocultas": 20,
    "initial_lr": 0.8,
    "epochs": 1000,
    "factor_reduccion_lr": 0.1,
    "paciencia_lr": 10,
    "min_lr": 1e-7,
    "n_samples_total": 1000,
    "factor_circles": 0.5,
    "noise_circles": 0.05,
    "random_state_data": 1,
    "test_split_ratio": 0.2,
    "val_split_ratio": 0.25,
    "model_save_path": "red_neuronal_optima.npz"
}

# --- Funciones de Activación ---
sigmoide = lambda x: 1 / (1 + np.exp(-x))
derivada_sigmoide = lambda y: y * (1 - y)

# --- Definición de Clases de Neurona y Red Neuronal ---
class Neurona:
    def __init__(self, num_entradas):
        self.pesos = np.random.randn(num_entradas) * 0.1
        self.sesgo = np.random.randn() * 0.1

class RedNeuronal:
    def __init__(self, num_entradas, num_neuronas_ocultas, num_salidas):
        self.capa_oculta = [Neurona(num_entradas) for _ in range(num_neuronas_ocultas)]
        self.capa_salida = [Neurona(num_neuronas_ocultas) for _ in range(num_salidas)]

# --- Predicción ---
def predecir_red(red, entradas):
    z_oculta = np.array([np.dot(entradas, n.pesos) + n.sesgo for n in red.capa_oculta])
    a_oculta = sigmoide(z_oculta)
    z_salida = np.array([np.dot(a_oculta, n.pesos) + n.sesgo for n in red.capa_salida])
    a_salida = sigmoide(z_salida)
    return a_salida, a_oculta

# --- Evaluación ---
def evaluar_rendimiento_red(red, X_set, y_set):
    error_set = 0
    aciertos_set = 0
    for entrada, objetivo in zip(X_set, y_set):
        pred, _ = predecir_red(red, entrada)
        pred_val = pred[0]
        error_set += 0.5 * (objetivo - pred_val) ** 2
        aciertos_set += int((pred_val >= 0.5) == objetivo)
    return error_set / len(X_set), aciertos_set / len(X_set)

# --- Entrenamiento ---
def entrenar_red_lr_adaptativo(red, X_train, y_train, X_val, y_val, params):
    print("\n--- Iniciando Entrenamiento con Learning Rate Adaptativo por Validación ---")
    hist_err_train, hist_acc_train = [], []
    hist_err_val, hist_acc_val = [], []
    hist_lr = []

    current_lr = params["initial_lr"]
    mejor_precision_val = -1
    epocas_sin_mejora_val = 0

    mejor_red_estado = {
        'oculta_pesos': [n.pesos.copy() for n in red.capa_oculta],
        'oculta_sesgos': [n.sesgo for n in red.capa_oculta],
        'salida_pesos': red.capa_salida[0].pesos.copy(),
        'salida_sesgo': red.capa_salida[0].sesgo
    }
    mejor_epoca = 0

    for epoch in range(1, params["epochs"] + 1):
        hist_lr.append(current_lr)
        error_total_train = 0
        aciertos_train = 0
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_shuffled, y_shuffled = X_train[indices], y_train[indices]

        for entrada, objetivo in zip(X_shuffled, y_shuffled):
            pred, a_oculta = predecir_red(red, entrada)
            pred_val = pred[0]
            aciertos_train += int((pred_val >= 0.5) == objetivo)
            error_total_train += 0.5 * (objetivo - pred_val) ** 2

            delta_salida = (objetivo - pred_val) * derivada_sigmoide(pred_val)
            error_oculta = delta_salida * red.capa_salida[0].pesos
            delta_oculta = error_oculta * derivada_sigmoide(a_oculta)

            red.capa_salida[0].pesos += current_lr * delta_salida * a_oculta
            red.capa_salida[0].sesgo += current_lr * delta_salida
            for j, n in enumerate(red.capa_oculta):
                n.pesos += current_lr * delta_oculta[j] * entrada
                n.sesgo += current_lr * delta_oculta[j]

        avg_error_train = error_total_train / len(X_train)
        precision_train = aciertos_train / len(X_train)
        hist_err_train.append(avg_error_train)
        hist_acc_train.append(precision_train)

        avg_error_val, precision_val = evaluar_rendimiento_red(red, X_val, y_val)
        hist_err_val.append(avg_error_val)
        hist_acc_val.append(precision_val)

        if precision_val > mejor_precision_val:
            mejor_precision_val = precision_val
            epocas_sin_mejora_val = 0
            mejor_epoca = epoch
            mejor_red_estado = {
                'oculta_pesos': [n.pesos.copy() for n in red.capa_oculta],
                'oculta_sesgos': [n.sesgo for n in red.capa_oculta],
                'salida_pesos': red.capa_salida[0].pesos.copy(),
                'salida_sesgo': red.capa_salida[0].sesgo
            }
        else:
            epocas_sin_mejora_val += 1

        if epocas_sin_mejora_val >= params["paciencia_lr"]:
            new_lr = current_lr * params["factor_reduccion_lr"]
            if new_lr >= params["min_lr"]:
                current_lr = new_lr
                print(f"LR Reducido. Época {epoch}/{params['epochs']}. Nuevo LR: {current_lr:.6f}")
                epocas_sin_mejora_val = 0
            else:
                current_lr = params["min_lr"]
                print(f"LR alcanzó el mínimo ({params['min_lr']:.6f}) en Época {epoch}. No más reducciones.")
                break

        if epoch % (params["epochs"] // 10 if params["epochs"] // 10 > 0 else 1) == 0 or epoch == 1:
            print(f"Época {epoch}/{params['epochs']} | Train (Err: {avg_error_train:.4f}, Acc: {precision_train:.4f}) | Val (Err: {avg_error_val:.4f}, Acc: {precision_val:.4f}) | LR: {current_lr:.6f}")

    print("--- Entrenamiento Finalizado ---")
    print(f"Mejor modelo guardado en la Época {mejor_epoca} con Precisión de Validación: {mejor_precision_val:.4f}")

    red_optima = RedNeuronal(len(red.capa_oculta[0].pesos), len(red.capa_oculta), len(red.capa_salida))
    for i, n in enumerate(red_optima.capa_oculta):
        n.pesos = mejor_red_estado['oculta_pesos'][i]
        n.sesgo = mejor_red_estado['oculta_sesgos'][i]
    red_optima.capa_salida[0].pesos = mejor_red_estado['salida_pesos']
    red_optima.capa_salida[0].sesgo = mejor_red_estado['salida_sesgo']

    return red_optima, hist_err_train, hist_acc_train, hist_err_val, hist_acc_val, hist_lr

# --- Funciones para Guardar y Cargar Modelo ---
def guardar_red(red: RedNeuronal, nombre_archivo: str):
    np.savez(nombre_archivo,
             oculta_pesos=[n.pesos for n in red.capa_oculta],
             oculta_sesgos=[n.sesgo for n in red.capa_oculta],
             salida_pesos=red.capa_salida[0].pesos,
             salida_sesgo=red.capa_salida[0].sesgo)
    print(f"Modelo guardado en {nombre_archivo}")

def cargar_red(nombre_archivo: str, num_entradas: int, num_neuronas_ocultas: int, num_salidas: int) -> RedNeuronal:
    data = np.load(nombre_archivo, allow_pickle=True)
    red = RedNeuronal(num_entradas, num_neuronas_ocultas, num_salidas)
    for i, n in enumerate(red.capa_oculta):
        n.pesos = data['oculta_pesos'][i]
        n.sesgo = data['oculta_sesgos'][i]
    red.capa_salida[0].pesos = data['salida_pesos']
    red.capa_salida[0].sesgo = data['salida_sesgo']
    print(f"Modelo cargado desde {nombre_archivo}")
    return red

# --- Visualización ---
def graficar_metricas_completas(hist_err_train, hist_acc_train, hist_err_val, hist_acc_val, hist_lr):
    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.plot(hist_err_train, label="Error Entrenamiento")
    plt.plot(hist_err_val, label="Error Validación", linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Error")
    plt.title("Error por Época")
    plt.grid()
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(hist_acc_train, label="Precisión Entrenamiento")
    plt.plot(hist_acc_val, label="Precisión Validación", linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Precisión")
    plt.title("Precisión por Época")
    plt.grid()
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(hist_lr, color='red', linestyle='--')
    plt.xlabel("Época")
    plt.ylabel("Learning Rate", color='red')
    plt.tick_params(axis='y', labelcolor='red')
    plt.title('Evolución del Learning Rate')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

def graficar_frontera_decision(red, X, y, title="Frontera de Decisión"):
    h = .02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    Z = np.array([predecir_red(red, np.array([x, y]))[0][0] for x, y in np.c_[xx.ravel(), yy.ravel()]])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdBu, edgecolors='k')
    plt.title(title)
    plt.xlabel("Característica 1")
    plt.ylabel("Característica 2")
    plt.show()

# --- Main Execution ---
if __name__ == "__main__":
    X, y = make_circles(
        n_samples=HIPERPARAMETROS["n_samples_total"],
        factor=HIPERPARAMETROS["factor_circles"],
        noise=HIPERPARAMETROS["noise_circles"],
        random_state=HIPERPARAMETROS["random_state_data"]
    )

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=HIPERPARAMETROS["test_split_ratio"], random_state=42
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=HIPERPARAMETROS["val_split_ratio"], random_state=42
    )

    print(f"Dataset de entrenamiento: {len(X_train)} muestras")
    print(f"Dataset de validación: {len(X_val)} muestras")
    print(f"Dataset de prueba: {len(X_test)} muestras")

    num_entradas = X.shape[1]
    num_salidas = 1
    red = RedNeuronal(
        num_entradas=num_entradas,
        num_neuronas_ocultas=HIPERPARAMETROS["num_neuronas_ocultas"],
        num_salidas=num_salidas
    )

    red_entrenada, err_train, acc_train_hist, err_val, acc_val_hist, lr_hist = \
        entrenar_red_lr_adaptativo(
            red, X_train, y_train, X_val, y_val,
            params=HIPERPARAMETROS
        )

    guardar_red(red_entrenada, HIPERPARAMETROS["model_save_path"])

    print("\n--- Evaluación Final del Modelo Entrenado en el conjunto de Prueba ---")
    final_error_test, final_precision_test = evaluar_rendimiento_red(red_entrenada, X_test, y_test)

    print(f"Error final (entrenamiento): {err_train[-1]:.4f}")
    print(f"Precisión final (entrenamiento): {acc_train_hist[-1]:.4f}")
    print(f"Error final (validación): {err_val[-1]:.4f}")
    print(f"Precisión final (validación): {acc_val_hist[-1]:.4f}")
    print(f"Error final (prueba): {final_error_test:.4f}")
    print(f"Precisión final (prueba): {final_precision_test:.4f}")

    graficar_metricas_completas(err_train, acc_train_hist, err_val, acc_val_hist, lr_hist)
    graficar_frontera_decision(red_entrenada, X_train, y_train, title="Frontera de Decisión (Entrenamiento)")
    graficar_frontera_decision(red_entrenada, X_test, y_test, title="Frontera de Decisión (Prueba - Generalización)")

    print("\n--- Cargando el modelo para probar con nuevos datasets ---")
    red_cargada = cargar_red(HIPERPARAMETROS["model_save_path"], num_entradas, HIPERPARAMETROS["num_neuronas_ocultas"], num_salidas)

    X_nuevo_1, y_nuevo_1 = make_circles(n_samples=200, factor=0.5, noise=0.08, random_state=123)
    print("\n--- Evaluación en Nuevo Dataset 1 ---")
    error_nuevo_1, precision_nuevo_1 = evaluar_rendimiento_red(red_cargada, X_nuevo_1, y_nuevo_1)
    print(f"Error (Nuevo Dataset 1): {error_nuevo_1:.4f}")
    print(f"Precisión (Nuevo Dataset 1): {precision_nuevo_1:.4f}")
    graficar_frontera_decision(red_cargada, X_nuevo_1, y_nuevo_1, title="Frontera de Decisión (Nuevo Dataset 1)")

    X_nuevo_2, y_nuevo_2 = make_circles(n_samples=200, factor=0.15, noise=0.20, random_state=789)
    print("\n--- Evaluación en Nuevo Dataset 2 ---")
    error_nuevo_2, precision_nuevo_2 = evaluar_rendimiento_red(red_cargada, X_nuevo_2, y_nuevo_2)
    print(f"Error (Nuevo Dataset 2): {error_nuevo_2:.4f}")
    print(f"Precisión (Nuevo Dataset 2): {precision_nuevo_2:.4f}")
    graficar_frontera_decision(red_cargada, X_nuevo_2, y_nuevo_2, title="Frontera de Decisión (Nuevo Dataset 2)")

#estado interno adaptativo

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# --- 1. Generación del Dataset de Onda Sinusoidal ---
def generar_serie_sinusoidal(n_puntos, ventana_entrada):
    x = np.linspace(0, 20 * np.pi, n_puntos)
    y = np.sin(x) + 0.1 * np.random.randn(n_puntos) # Onda sinusoidal con algo de ruido

    X, Y = [], []
    for i in range(len(y) - ventana_entrada):
        X.append(y[i:(i + ventana_entrada)])
        Y.append(y[i + ventana_entrada])
    return np.array(X), np.array(Y)

N_PUNTOS = 1000
VENTANA_ENTRADA = 10 # Usar 10 puntos anteriores para predecir el siguiente

X_data, Y_data = generar_serie_sinusoidal(N_PUNTOS, VENTANA_ENTRADA)

# Dividir en conjuntos de entrenamiento y prueba
split_ratio = 0.8
split_index = int(N_PUNTOS * split_ratio)

X_train, Y_train = X_data[:split_index], Y_data[:split_index]
X_test, Y_test = X_data[split_index:], Y_data[split_index:]

# --- 2. Los Modelos ---

# Modelo con "Neurona Común" (SGD Básico)
model_sgd = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(VENTANA_ENTRADA,)),
    tf.keras.layers.Dense(1)
])
model_sgd.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01), loss='mse')
print("--- Modelo SGD Básico ---")
model_sgd.summary()

# Modelo con "SGD Experimental con Momentum"
# Esto simula un "historial" al considerar los gradientes pasados
model_sgdm = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(VENTANA_ENTRADA,)),
    tf.keras.layers.Dense(1)
])
# Usamos SGD con momentum. El momentum es un "historial" de la dirección de los gradientes.
model_sgdm.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9), loss='mse')
print("\n--- Modelo SGD con Momentum (Experimental) ---")
model_sgdm.summary()

# --- 3. Entrenamiento y Evaluación ---

EPOCHS = 50 # Número de veces que el modelo verá todo el dataset

print("\nEntrenando Modelo SGD Básico...")
history_sgd = model_sgd.fit(X_train, Y_train, epochs=EPOCHS, verbose=0, validation_data=(X_test, Y_test))

print("Entrenando Modelo SGD con Momentum (Experimental)...")
history_sgdm = model_sgdm.fit(X_train, Y_train, epochs=EPOCHS, verbose=0, validation_data=(X_test, Y_test))

# Evaluación de rendimiento en el conjunto de prueba
loss_sgd = model_sgd.evaluate(X_test, Y_test, verbose=0)
loss_sgdm = model_sgdm.evaluate(X_test, Y_test, verbose=0)

print(f"\n--- Resultados ---")
print(f"MSE del Modelo SGD Básico en datos de prueba: {loss_sgd:.4f}")
print(f"MSE del Modelo SGD con Momentum en datos de prueba: {loss_sgdm:.4f}")

# Visualización del aprendizaje (pérdida en el tiempo)
plt.figure(figsize=(12, 6))
plt.plot(history_sgd.history['loss'], label='SGD Básico Train Loss')
plt.plot(history_sgd.history['val_loss'], label='SGD Básico Validation Loss')
plt.plot(history_sgdm.history['loss'], label='SGD con Momentum Train Loss')
plt.plot(history_sgdm.history['val_loss'], label='SGD con Momentum Validation Loss')
plt.title('Pérdida (Loss) durante el Entrenamiento')
plt.xlabel('Época')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# Visualización de predicciones
predicciones_sgd = model_sgd.predict(X_test)
predicciones_sgdm = model_sgdm.predict(X_test)

plt.figure(figsize=(12, 6))
plt.plot(Y_test, label='Valores Reales')
plt.plot(predicciones_sgd, label='Predicciones SGD Básico', alpha=0.7)
plt.plot(predicciones_sgdm, label='Predicciones SGD con Momentum', alpha=0.7)
plt.title('Predicciones de Serie Temporal en Datos de Prueba')
plt.xlabel('Punto en la Serie')
plt.ylabel('Valor')
plt.legend()
plt.grid(True)
plt.show()

#SGD con Momentum

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# --- 1. Dataset de Onda Sinusoidal ---
def generar_serie_sinusoidal(n_puntos, ventana_entrada):
    x = np.linspace(0, 20 * np.pi, n_puntos)
    y = np.sin(x) + 0.1 * np.random.randn(n_puntos)  # ruido leve

    X, Y = [], []
    for i in range(len(y) - ventana_entrada):
        X.append(y[i:(i + ventana_entrada)])
        Y.append(y[i + ventana_entrada])
    return np.array(X), np.array(Y)

N_PUNTOS = 1000
VENTANA_ENTRADA = 10
X_data, Y_data = generar_serie_sinusoidal(N_PUNTOS, VENTANA_ENTRADA)

# Entrenamiento / Prueba
split_ratio = 0.8
split_index = int(len(X_data) * split_ratio)
X_train, Y_train = X_data[:split_index], Y_data[:split_index]
X_test, Y_test = X_data[split_index:], Y_data[split_index:]

# --- 2. Capa con Estado Interno Adaptativo ---
class NeuronaEstadoInterno(tf.keras.layers.Layer):
    def __init__(self, unidades):
        super().__init__()
        self.unidades = unidades

    def build(self, input_shape):
        self.w_input = self.add_weight(shape=(input_shape[-1], self.unidades),
                                       initializer='glorot_uniform', trainable=True)
        self.w_state = self.add_weight(shape=(self.unidades, self.unidades),
                                       initializer='orthogonal', trainable=True)
        self.b = self.add_weight(shape=(self.unidades,), initializer='zeros', trainable=True)
        self.initial_state = self.add_weight(shape=(self.unidades,), initializer='zeros', trainable=True)

    def call(self, inputs):
        # Inicializar el estado interno como una variable que persiste durante la llamada
        state = tf.tile(tf.expand_dims(self.initial_state, 0), [tf.shape(inputs)[0], 1])
        state = tf.nn.tanh(tf.matmul(inputs, self.w_input) + tf.matmul(state, self.w_state) + self.b)
        return state

# --- 3. Modelos ---

# Modelo convencional
modelo_denso = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(VENTANA_ENTRADA,)),
    tf.keras.layers.Dense(1)
])
modelo_denso.compile(optimizer='adam', loss='mse')

# Modelo con estado adaptativo
modelo_estado = tf.keras.Sequential([
    NeuronaEstadoInterno(32),
    tf.keras.layers.Dense(1)
])
modelo_estado.compile(optimizer='adam', loss='mse')

# --- 4. Entrenamiento ---
EPOCHS = 50
print("Entrenando modelo común...")
hist_denso = modelo_denso.fit(X_train, Y_train, epochs=EPOCHS, validation_data=(X_test, Y_test), verbose=0)

print("Entrenando modelo con estado adaptativo...")
hist_estado = modelo_estado.fit(X_train, Y_train, epochs=EPOCHS, validation_data=(X_test, Y_test), verbose=0)

# --- 5. Evaluación ---
loss_denso = modelo_denso.evaluate(X_test, Y_test, verbose=0)
loss_estado = modelo_estado.evaluate(X_test, Y_test, verbose=0)

print("\n--- Resultados ---")
print(f"Modelo denso (común): MSE = {loss_denso:.4f}")
print(f"Modelo con estado interno adaptativo: MSE = {loss_estado:.4f}")

# --- 6. Visualización de la pérdida ---
plt.figure(figsize=(12, 6))
plt.plot(hist_denso.history['val_loss'], label='Modelo Denso (val_loss)')
plt.plot(hist_estado.history['val_loss'], label='Modelo Adaptativo (val_loss)')
plt.title("Comparación de Pérdida de Validación")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()

# --- 7. Predicciones ---
pred_denso = modelo_denso.predict(X_test)
pred_estado = modelo_estado.predict(X_test)

plt.figure(figsize=(12, 6))
plt.plot(Y_test, label='Real')
plt.plot(pred_denso, label='Denso', alpha=0.7)
plt.plot(pred_estado, label='Adaptativo', alpha=0.7)
plt.title("Predicción sobre Datos de Prueba")
plt.xlabel("Punto en la Serie")
plt.ylabel("Valor")
plt.legend()
plt.grid(True)
plt.show()

#CapaRuteoAdaptativo

In [ ]:
#Red Neuronal con Ruteo Adaptativo de Contribución (RARC-NN)"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# --- 1. Definición de las Rutas de Procesamiento (tus "Capas 1/2" y "Capa 3") ---

class RutaProcesamientoExito(layers.Layer):
    """
    Simula el camino de "contribución buena" (se mueve a Capa 1 y Capa 2).
    Asumimos que estas son capas densas secuenciales.
    """
    def __init__(self, units_h1, units_h2, **kwargs):
        super(RutaProcesamientoExito, self).__init__(**kwargs)
        self.dense1 = layers.Dense(units_h1, activation='relu', name="exito_h1")
        self.dense2 = layers.Dense(units_h2, activation='relu', name="exito_h2")

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        return x

class RutaProcesamientoReevaluacion(layers.Layer):
    """
    Simula el camino de "contribución mala" (se mueve a Capa 3).
    Podría tener una arquitectura diferente para "reparar" o "re-evaluar".
    """
    def __init__(self, units_h3, **kwargs):
        super(RutaProcesamientoReevaluacion, self).__init__(**kwargs)
        # Una capa más pequeña o diferente para la re-evaluación
        self.dense_reevaluacion = layers.Dense(units_h3, activation='tanh', name="reevaluacion_h3")
        # Podría tener más capas o una arquitectura recurrente aquí si fuera necesario

    def call(self, inputs):
        x = self.dense_reevaluacion(inputs)
        return x

# --- 2. Capa de Ruteo (el "cerebro" de los Neuros Nómadas) ---

class CapaRuteoAdaptativo(layers.Layer):
    """
    Decide la proporción en que la entrada se envía a cada ruta.
    Simula si una "neurona" (o su contribución) es "buena" o "mala".
    """
    def __init__(self, num_routes, **kwargs):
        super(CapaRuteoAdaptativo, self).__init__(**kwargs)
        self.num_routes = num_routes
        # La capa que predice los pesos para cada ruta
        self.gating_network = layers.Dense(num_routes, activation='softmax', name="gating_network")

    def call(self, inputs):
        # El gating network toma la entrada y produce pesos para cada ruta
        route_weights = self.gating_network(inputs)
        return route_weights

# --- Nuevo: Capa para Combinar Salidas Ponderadas ---
class CapaCombinacionPonderada(layers.Layer):
    """
    Combina las salidas de las rutas de procesamiento usando los pesos de ruteo.
    """
    def __init__(self, **kwargs):
        super(CapaCombinacionPonderada, self).__init__(**kwargs)

    def call(self, inputs):
        # inputs is a list: [pesos_rutas, salida_exito_final, salida_reevaluacion_final]
        pesos_rutas, salida_exito, salida_reevaluacion = inputs

        # Perform the weighted sum within the layer's call method
        # tf.expand_dims is now inside a Keras Layer call method
        salida_combinada = (tf.expand_dims(pesos_rutas[:, 0], axis=-1) * salida_exito +
                            tf.expand_dims(pesos_rutas[:, 1], axis=-1) * salida_reevaluacion)
        return salida_combinada


# --- 3. Construcción del Modelo "RARC-NN" ---

def construir_rarc_nn(input_shape,
                      exito_h1_units, exito_h2_units, # Para Ruta Éxito (Capas 1 y 2)
                      reevaluacion_h3_units,         # Para Ruta Re-evaluación (Capa 3)
                      output_units):
    """
    Construye la Red Neuronal con Ruteo Adaptativo de Contribución (RARC-NN).
    Simula "Neuros Nómadas" moviéndose entre rutas de procesamiento.
    """
    entrada = keras.Input(shape=input_shape, name="entrada_general")

    # 1. Capa de Ruteo
    # Decide los pesos para cada ruta basándose en la entrada
    ruteador = CapaRuteoAdaptativo(num_routes=2, name="ruteador_adaptativo")
    pesos_rutas = ruteador(entrada) # pesos_rutas[0] para éxito, pesos_rutas[1] para re-evaluación

    # 2. Rutas de Procesamiento
    ruta_exito = RutaProcesamientoExito(exito_h1_units, exito_h2_units, name="ruta_procesamiento_exito")
    ruta_reevaluacion = RutaProcesamientoReevaluacion(reevaluacion_h3_units, name="ruta_procesamiento_reevaluacion")

    # Obtenemos las salidas de cada ruta de procesamiento
    salida_exito = ruta_exito(entrada)
    salida_reevaluacion = ruta_reevaluacion(entrada)

    # Asegurarse de que las salidas de las rutas tengan la misma dimensión final
    salida_exito_final = layers.Dense(output_units, activation='linear', name="exito_a_final")(salida_exito)
    salida_reevaluacion_final = layers.Dense(output_units, activation='linear', name="reevaluacion_a_final")(salida_reevaluacion)

    # 3. Combinación de las salidas usando la nueva capa
    combinador = CapaCombinacionPonderada(name="combinador_ponderado")
    # La capa combinadora toma una lista de tensores como entrada:
    # los pesos de ruteo, la salida final de la ruta de éxito, y la salida final de la ruta de re-evaluación
    salida_combinada = combinador([pesos_rutas, salida_exito_final, salida_reevaluacion_final])


    # 4. Capa de Salida Final
    salida = layers.Dense(output_units, activation='softmax', name="salida_prediccion")(salida_combinada)

    # Construir el modelo
    model = keras.Model(inputs=entrada, outputs=salida, name="RARC_NN_Experimental")
    return model

# --- 4. Parámetros del Experimento ---
INPUT_DIM = 50           # Dimensiones de las características de entrada
EXITO_H1_UNITS = 128     # Neuronas en la Capa Oculta 1 de la Ruta Éxito
EXITO_H2_UNITS = 64      # Neuronas en la Capa Oculta 2 de la Ruta Éxito
REEVALUACION_H3_UNITS = 32 # Neuronas en la Capa Oculta de la Ruta Re-evaluación
OUTPUT_UNITS = 5         # Número de clases de salida

# --- 5. Construir y Resumir el Modelo Experimental ---
print("--- Construyendo la Red Neuronal con Ruteo Adaptativo de Contribución (RARC-NN) ---")
modelo_rarc = construir_rarc_nn(
    input_shape=(INPUT_DIM,),
    exito_h1_units=EXITO_H1_UNITS,
    exito_h2_units=EXITO_H2_UNITS,
    reevaluacion_h3_units=REEVALUACION_H3_UNITS,
    output_units=OUTPUT_UNITS
)

modelo_rarc.summary()

# --- 6. Preparar Datos Sintéticos para el Experimento ---
print("\n--- Generando datos sintéticos para el experimento ---")
num_samples = 1000 # Número de ejemplos
X_sintetico = np.random.rand(num_samples, INPUT_DIM).astype(np.float32)
y_sintetico = np.random.randint(0, OUTPUT_UNITS, num_samples) # Clases aleatorias

# --- 7. Compilar el Modelo para el Experimento ---
print("\n--- Compilando el modelo experimental ---")
modelo_rarc.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy', # Para clasificación
                    metrics=['accuracy'])

# --- 8. Ejecutar el Experimento (Entrenamiento) ---
print("\n--- Iniciando el experimento de entrenamiento ---")
history = modelo_rarc.fit(X_sintetico, y_sintetico,
                          epochs=5,          # Menos epochs para un experimento rápido
                          batch_size=32,
                          validation_split=0.2, # Usar una parte de los datos para validación
                          verbose=1)

print("\n--- Experimento de entrenamiento finalizado ---")

# --- 9. Evaluar el Rendimiento del Experimento ---
print("\n--- Evaluando el modelo experimental ---")
X_test = np.random.rand(100, INPUT_DIM).astype(np.float32)
y_test = np.random.randint(0, OUTPUT_UNITS, 100)

loss, accuracy = modelo_rarc.evaluate(X_test, y_test, verbose=0)
print(f"Pérdida en datos de prueba: {loss:.4f}")
print(f"Precisión en datos de prueba: {accuracy:.4f}")

print("\n--- Predicciones de ejemplo ---")
sample_prediction = modelo_rarc.predict(X_test[:2])
print(f"Predicciones para las primeras 2 muestras:\n{sample_prediction}")

# --- Observar el ruteo para algunas muestras ---
print("\n--- Observando los pesos de ruteo para algunas muestras ---")
# Para ver los pesos del ruteador, podemos crear un modelo que solo devuelva esos pesos
modelo_ruteo_solo = keras.Model(inputs=modelo_rarc.input, outputs=modelo_rarc.get_layer("ruteador_adaptativo").output)
pesos_ejemplo = modelo_ruteo_solo.predict(X_test[:5])
print(f"Pesos de ruteo para las primeras 5 muestras (Éxito | Re-evaluación):\n{pesos_ejemplo}")